In [29]:
import sys
from pathlib import Path
import json
from textwrap import wrap
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.gridspec import GridSpec
from matplotlib.backends.backend_pdf import PdfPages
import textwrap

ROOT = Path("/Users/andreali/Documents/Subgraph_Federated_Learning/")

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from andrea.multigraph_generation import TASKS

In [30]:
BENCHMARK = "specialized"   # "rep", "masked", or "specialized"

if BENCHMARK == "masked":
    SELECT_SUBSET_PATH = "clustering_masked"
    SELECT_SUBSET = "selected_subset"
    EXPERIMENT_LOG_FOLDER = "experiment_log_masked.csv"
    DATA_DIR = f"{SELECT_SUBSET_PATH}/cluster_generation_parameters.csv"
    OUT_DIR_NAME = "plot_outputs_masked"
elif BENCHMARK == "rep":
    SELECT_SUBSET_PATH = "clustering_rep"
    SELECT_SUBSET = "selected_subset"
    EXPERIMENT_LOG_FOLDER = "experiment_log_rep.csv"
    DATA_DIR = f"{SELECT_SUBSET_PATH}/cluster_generation_parameters.csv"
    OUT_DIR_NAME = "plot_outputs_rep"
elif BENCHMARK == "specialized":
    SELECT_SUBSET_PATH = "clustering_task_specialized"
    SELECT_SUBSET = "selected_subset"
    EXPERIMENT_LOG_FOLDER = "experiment_log_specialized_0.05_exp3.csv"
    DATA_DIR = f"{SELECT_SUBSET_PATH}/cluster_generation_parameters.csv"
    OUT_DIR_NAME = "plot_outputs_specialized_0.05_exp3"
else:
    raise ValueError(f"Unknown BENCHMARK={BENCHMARK}")

SELECTED_SUBSETS_CSV_PATH = Path(f"./{SELECT_SUBSET_PATH}/{SELECT_SUBSET}.csv")

selected_subset = pd.read_csv(SELECTED_SUBSETS_CSV_PATH)
exp_log = pd.read_csv(EXPERIMENT_LOG_FOLDER)
test_gen = pd.read_csv(DATA_DIR)

print("Loaded selected subset rows:", len(selected_subset))
print("Loaded experiment log rows:", len(exp_log))
print("Loaded registry rows:", len(test_gen))

if "mask_fraction" in exp_log.columns:
    print("\nRows by mask_fraction:")
    print(exp_log["mask_fraction"].value_counts(dropna=False).sort_index())

Loaded selected subset rows: 3
Loaded experiment log rows: 81
Loaded registry rows: 15

Rows by mask_fraction:
mask_fraction
0.2    27
0.5    27
0.8    27
Name: count, dtype: int64


In [31]:
# BASELINE_ROUND_BUDGET = 80
# APPLE_ROUND_BUDGET = 160

# APPLE_PLOT_ROUNDS = 160
# APPLE_PLOT_DR_LR = 1e-3
# APPLE_PLOT_MU = 1e-2
# APPLE_PLOT_SCHEDULER_FRACTION = 0.3

BASELINE_ROUND_BUDGET = 80
APPLE_ROUND_BUDGET = 80

APPLE_PLOT_ROUNDS = 80
APPLE_PLOT_DR_LR = 1e-3
APPLE_PLOT_MU = 1e-2
APPLE_PLOT_SCHEDULER_FRACTION = 0.3

In [32]:
print("\nRows by run_type:")
print(exp_log["run_type"].value_counts(dropna=False))

print("\nRounds by run_type:")
print(
    exp_log.groupby("run_type")["rounds"]
    .apply(lambda s: sorted(pd.to_numeric(s, errors="coerce").dropna().unique()))
)

apple_debug = exp_log[exp_log["run_type"].astype(str) == "apple"].copy()
print("\nAPPLE config:")
print(
    apple_debug[
        [
            "seed",
            "mask_fraction",
            "rounds",
            "apple_dr_lr",
            "apple_mu",
            "apple_scheduler_fraction",
            "out_csv",
        ]
    ].to_string(index=False)
)


Rows by run_type:
run_type
local        45
fedavg        9
fedprox       9
gcfl_plus     9
apple         9
Name: count, dtype: int64

Rounds by run_type:
run_type
apple        [80]
fedavg       [80]
fedprox      [80]
gcfl_plus    [80]
local        [80]
Name: rounds, dtype: object

APPLE config:
 seed  mask_fraction  rounds  apple_dr_lr  apple_mu  apple_scheduler_fraction                                                                                                                                                                                       out_csv
    0            0.2      80        0.001      0.01                       0.3 andrea/runs/apple2000000|2000001|2000002|2000003|2000004_rounds80_epoch1_drlr0.001_mu0.01_schedcosine_Lfrac0.3_mcwauto_layers6_lr0.001_wd0.0001_do0.1_hd64_egoTrue_bs64_selectlocal_seed0.csv
    0            0.5      80        0.001      0.01                       0.3 andrea/runs/apple2000005|2000006|2000007|2000008|2000009_rounds80_epoch1_drlr0.001_mu0.01

In [33]:
def normalize_manifest_and_experiment_log(selected_subset, exp_log):
    selected_subset = selected_subset.copy()
    exp_log = exp_log.copy()

    selected_subset["subset_clients"] = selected_subset["subset_clients"].astype(str)
    exp_log["subset_clients"] = exp_log["subset_clients"].astype(str)

    manifest_cols = [
        "subset_clients",
        "subset_id",
        "subset_size",
        "gamma",
        "mask_fraction",
        "specialization_fraction",
        "designed_heterogeneity",
        "global_visible_support_fraction_ideal",
        "controlled_benchmark",
        "mask_mode",
        "task_profile_jsd_mean",
        "task_profile_jsd_median",
        "task_profile_jsd_max",
        "between_family_centroid_jsd_mean",
        "family_order_json",
        "family_to_graph_ids_json",
        "graph_to_family_json",
        "family_counts_json",
        "family_to_task_means_json",
        "family_to_task_stds_json",
        "membership_json",
    ]

    manifest_cols = [c for c in manifest_cols if c in selected_subset.columns]

    exp_log = exp_log.merge(
        selected_subset[manifest_cols],
        on="subset_clients",
        how="left",
        suffixes=("", "_manifest"),
    )

    fill_cols = [
        "subset_size",
        "gamma",
        "mask_fraction",
        "specialization_fraction",
        "designed_heterogeneity",
        "global_visible_support_fraction_ideal",
        "controlled_benchmark",
        "mask_mode",
        "task_profile_jsd_mean",
        "task_profile_jsd_median",
        "task_profile_jsd_max",
        "between_family_centroid_jsd_mean",
        "family_order_json",
        "family_to_graph_ids_json",
        "graph_to_family_json",
        "family_counts_json",
        "family_to_task_means_json",
        "family_to_task_stds_json",
        "membership_json",
    ]

    for col in fill_cols:
        mcol = f"{col}_manifest"
        if mcol in exp_log.columns:
            if col not in exp_log.columns:
                exp_log[col] = exp_log[mcol]
            else:
                exp_log[col] = exp_log[col].combine_first(exp_log[mcol])

    # Preserve manifest subset id separately.
    if "subset_id_manifest" in exp_log.columns:
        if "manifest_subset_id" not in exp_log.columns:
            exp_log["manifest_subset_id"] = exp_log["subset_id_manifest"]

    for df in [selected_subset, exp_log]:
        if "gamma" not in df.columns:
            df["gamma"] = np.nan
        if "mask_fraction" not in df.columns:
            df["mask_fraction"] = np.nan
        if "specialization_fraction" not in df.columns:
            df["specialization_fraction"] = np.nan

        df["heterogeneity_value"] = df.apply(
            lambda r: (
                float(r["specialization_fraction"])
                if pd.notna(r.get("specialization_fraction", np.nan))
                else float(r["mask_fraction"])
                if pd.notna(r.get("mask_fraction", np.nan))
                else float(r["gamma"])
                if pd.notna(r.get("gamma", np.nan))
                else np.nan
            ),
            axis=1,
        )

    return selected_subset, exp_log

In [34]:
selected_subset, exp_log = normalize_manifest_and_experiment_log(
    selected_subset,
    exp_log,
)
print(selected_subset)

                                              family  \
0  controlled_task_specialized_five_client_benchmark   
1  controlled_task_specialized_five_client_benchmark   
2  controlled_task_specialized_five_client_benchmark   

                                           subset_id  mask_fraction  \
0  controlled_specialized_base0_x0.2_2000000_2000...            0.2   
1  controlled_specialized_base0_x0.5_2000005_2000...            0.5   
2  controlled_specialized_base0_x0.8_2000010_2000...            0.8   

   subset_size                           subset_clients  \
0            5  2000000|2000001|2000002|2000003|2000004   
1            5  2000005|2000006|2000007|2000008|2000009   
2            5  2000010|2000011|2000012|2000013|2000014   

                              graph_ids_json  \
0  [2000000,2000001,2000002,2000003,2000004]   
1  [2000005,2000006,2000007,2000008,2000009]   
2  [2000010,2000011,2000012,2000013,2000014]   

                                    dataset_ids_json  \
0  [

In [35]:
def check_local_coverage_for_plots(selected_subset, exp_log, model_tag=None, local_epochs=1):
    local = exp_log[exp_log["run_type"] == "local"].copy()

    if model_tag is not None:
        local = local[local["model_tag"].astype(str) == str(model_tag)].copy()

    local = local[
        pd.to_numeric(local["local_epochs"], errors="coerce") == int(local_epochs)
    ].copy()

    rows = []

    for _, sel in selected_subset.iterrows():
        subset_clients = str(sel["subset_clients"])
        graph_ids = [str(x) for x in parse_json_list_safe(sel["graph_ids_json"])]

        sub = local[local["subset_clients"].astype(str) == subset_clients].copy()

        if len(sub) == 0:
            rows.append({
                "subset_clients": subset_clients,
                "heterogeneity_value": sel.get("heterogeneity_value", np.nan),
                "heterogeneity_level": _heterogeneity_label_from_row(sel),
                "expected_clients": len(graph_ids),
                "found_clients": 0,
                "missing_clients": "|".join(graph_ids),
            })
            continue

        found = set(sub["graph_id"].astype(str).tolist())
        missing = [g for g in graph_ids if g not in found]

        rows.append({
            "subset_clients": subset_clients,
            "heterogeneity_value": sel.get("heterogeneity_value", np.nan),
            "heterogeneity_level": _heterogeneity_label_from_row(sel),
            "expected_clients": len(graph_ids),
            "found_clients": len(found),
            "missing_clients": "|".join(missing),
        })

    return pd.DataFrame(rows)

## 1. Parsing and manifest helpers

In [36]:
def parse_json_list_safe(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return []
    if isinstance(x, list):
        return x
    return json.loads(x)


def parse_json_dict_safe(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return {}
    if isinstance(x, dict):
        return x
    return json.loads(x)


def resolve_project_path(path_like):
    p = Path(path_like)
    candidates = []
    if p.is_absolute():
        candidates.append(p)
    else:
        candidates.extend(
            [
                p,
                Path.cwd() / p,
                ROOT / p,
            ]
        )
    for cand in candidates:
        if cand.exists():
            return cand.resolve()
    raise FileNotFoundError(f"Could not resolve path: {path_like}")


def _summary_to_mean(x):
    if not isinstance(x, str):
        return np.nan
    x = x.strip()
    if x in {"-", "", "NA"}:
        return np.nan

    import re

    m = re.match(r"^\s*([-+]?\d+(?:\.\d+)?)\s*%", x)
    if m:
        return float(m.group(1))

    m = re.match(r"^\s*([-+]?\d+(?:\.\d+)?)", x)
    if m:
        return float(m.group(1))

    return np.nan


def _effective_step(row):
    round_val = row["round"] if "round" in row.index else np.nan
    local_epoch_val = row["local_epoch"] if "local_epoch" in row.index else np.nan

    if pd.notna(round_val):
        return int(round_val)

    if pd.notna(local_epoch_val):
        return int(local_epoch_val)

    return np.nan


def _split_pipe_values(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return []

    out = []
    for part in str(x).split("|"):
        s = str(part).strip()
        if s == "" or s.lower() == "nan":
            continue
        try:
            f = float(s)
            if float(f).is_integer():
                s = str(int(f))
        except Exception:
            pass
        out.append(s)
    return out


def _members_to_multiline(member_str, per_line=6):
    gids = _split_pipe_values(member_str)
    if not gids:
        return "-"
    lines = []
    for i in range(0, len(gids), per_line):
        lines.append(", ".join(gids[i : i + per_line]))
    return "\n".join(lines)

In [37]:
METHOD_ORDER = ["fedavg", "fedprox", "gcfl_plus", "apple"]

METHOD_LABELS = {
    "local": "Local",
    "fedavg": "FedAvg",
    "fedprox": "FedProx",
    "gcfl_plus": "GCFL+",
    "apple": "APPLE",
}

OLD_FAMILY_ORDER = [
    "strong_decreasing",
    "mild_decreasing",
    "flat_balanced",
    "mild_increasing",
    "strong_increasing",
]

MASKED_FAMILY_ORDER = [
    "masked_cycle2",
    "masked_cycle3",
    "masked_cycle4",
    "masked_cycle5",
    "masked_cycle6",
]

SPECIALIZED_FAMILY_ORDER = [
    "specialized_cycle2",
    "specialized_cycle3",
    "specialized_cycle4",
    "specialized_cycle5",
    "specialized_cycle6",
]

if BENCHMARK == "specialized":
    FAMILY_ORDER = SPECIALIZED_FAMILY_ORDER
elif BENCHMARK == "masked":
    FAMILY_ORDER = MASKED_FAMILY_ORDER
else:
    FAMILY_ORDER = OLD_FAMILY_ORDER


def load_run_csv(csv_path_like) -> pd.DataFrame:
    csv_path = resolve_project_path(csv_path_like)
    return pd.read_csv(csv_path)


def filter_local_manifest(
    exp_log, subset_clients=None, model_tag=None, graph_ids=None, local_epochs=1
):
    part = exp_log[exp_log["run_type"] == "local"].copy()

    if model_tag is not None:
        part = part[part["model_tag"].astype(str) == str(model_tag)].copy()

    part = part[
        pd.to_numeric(part["local_epochs"], errors="coerce") == int(local_epochs)
    ].copy()

    if graph_ids is not None:
        graph_ids = {str(g) for g in graph_ids}
        part = part[part["graph_id"].astype(str).isin(graph_ids)].copy()

    part["seed"] = pd.to_numeric(part["seed"], errors="coerce").astype("Int64")

    # Safety: if the same local client/seed appears more than once, keep one.
    part = part.sort_values(["seed", "graph_id"]).drop_duplicates(
        subset=["seed", "graph_id", "model_tag", "local_epochs"],
        keep="last",
    )

    return part.sort_values(["seed", "graph_id"]).reset_index(drop=True)


def filter_method_manifest(
    exp_log,
    subset_clients=None,
    model_tag=None,
    method=None,
    local_epochs=1,
):
    """
    Filter experiment-level manifest rows for one federated method.

    Works for:
      - fedavg
      - fedprox
      - gcfl_plus

    It intentionally excludes local rows.
    """
    part = exp_log.copy()

    if method is not None:
        part = part[part["run_type"].astype(str) == str(method)].copy()

    if subset_clients is not None:
        part = part[part["subset_clients"].astype(str) == str(subset_clients)].copy()

    if model_tag is not None:
        part = part[part["model_tag"].astype(str) == str(model_tag)].copy()

    if "local_epochs" in part.columns:
        part = part[
            pd.to_numeric(part["local_epochs"], errors="coerce") == int(local_epochs)
        ].copy()

    # For APPLE long-run diagnostics, explicitly select the new 160-round config.
    # Without this, if the experiment log contains both old APPLE-80 and new
    # APPLE-160 rows, the notebook may accidentally aggregate both.
    if method is not None and str(method) == "apple":
        if "rounds" in part.columns:
            part = part[
                pd.to_numeric(part["rounds"], errors="coerce") == int(APPLE_PLOT_ROUNDS)
            ].copy()

        if "apple_dr_lr" in part.columns:
            part = part[
                np.isclose(
                    pd.to_numeric(part["apple_dr_lr"], errors="coerce"),
                    float(APPLE_PLOT_DR_LR),
                    rtol=0.0,
                    atol=1e-12,
                )
            ].copy()

        if "apple_mu" in part.columns:
            part = part[
                np.isclose(
                    pd.to_numeric(part["apple_mu"], errors="coerce"),
                    float(APPLE_PLOT_MU),
                    rtol=0.0,
                    atol=1e-12,
                )
            ].copy()

        if "apple_scheduler_fraction" in part.columns:
            part = part[
                np.isclose(
                    pd.to_numeric(part["apple_scheduler_fraction"], errors="coerce"),
                    float(APPLE_PLOT_SCHEDULER_FRACTION),
                    rtol=0.0,
                    atol=1e-12,
                )
            ].copy()

    if "seed" in part.columns:
        part["seed"] = pd.to_numeric(part["seed"], errors="coerce").astype("Int64")

    # Keep one row per method/subset/model/seed.
    # This protects against duplicated experiment_log entries.

    dedup_cols = [
        c
        for c in [
            "run_type",
            "subset_clients",
            "model_tag",
            "local_epochs",
            "seed",
            # Different round budgets / APPLE configs should not collapse together.
            "rounds",
            "apple_dr_lr",
            "apple_mu",
            "apple_scheduler_type",
            "apple_scheduler_fraction",
            "apple_dr_init",
            "apple_dr_constraint",
            "apple_download_strategy",
            # FedProx / GCFL+ config fields.
            "fedprox_mu",
            "warmup_rounds",
            "min_cluster_size",
            "min_child_size",
            "grad_seq_len",
            "eps1_quantile",
            "eps2_quantile",
        ]
        if c in part.columns
    ]

    if dedup_cols:
        part = part.sort_values(dedup_cols).drop_duplicates(
            subset=dedup_cols,
            keep="last",
        )

    sort_cols = [c for c in ["seed", "run_type"] if c in part.columns]
    if sort_cols:
        part = part.sort_values(sort_cols)

    return part.reset_index(drop=True)


def load_local_items(local_rows):
    items = []
    for _, row in local_rows.iterrows():
        items.append(
            {
                "seed": int(row["seed"]),
                "graph_id": str(row["graph_id"]),
                "family": row.get("family", None),
                "local_epochs": int(row["local_epochs"]),
                "df": load_run_csv(row["out_csv"]),
            }
        )
    return items


def load_gcfl_cluster_csv(path_like) -> pd.DataFrame:
    path = resolve_project_path(path_like)
    if not path.exists() or path.stat().st_size == 0:
        return pd.DataFrame()
    return pd.read_csv(path)


def load_apple_dr_csv(path_like) -> pd.DataFrame:
    path = resolve_project_path(path_like)
    if not path.exists() or path.stat().st_size == 0:
        return pd.DataFrame()
    return pd.read_csv(path)


def load_method_items(method_rows):
    items = []

    for _, row in method_rows.iterrows():
        item = {
            "seed": int(row["seed"]),
            "local_epochs": int(row["local_epochs"]),
            "method": str(row["run_type"]),
            "df": load_run_csv(row["out_csv"]),
        }

        # Keep method metadata on the item so later plotting code can use it.
        for col in [
            "subset_clients",
            "model_tag",
            "warmup_rounds",
            "min_cluster_size",
            "min_child_size",
            "grad_seq_len",
            "eps1_quantile",
            "eps2_quantile",
            "clusters_csv_path",
            "dr_csv_path",
            "apple_dr_lr",
            "apple_mu",
            "apple_scheduler_type",
            "apple_scheduler_fraction",
            "apple_dr_init",
            "apple_dr_constraint",
            "apple_download_strategy",
        ]:
            if col in row.index and pd.notna(row[col]):
                item[col] = row[col]

        if (
            str(row["run_type"]) == "gcfl_plus"
            and "clusters_csv_path" in row.index
            and pd.notna(row["clusters_csv_path"])
        ):
            item["clusters_df"] = load_gcfl_cluster_csv(row["clusters_csv_path"])

        if (
            str(row["run_type"]) == "apple"
            and "dr_csv_path" in row.index
            and pd.notna(row["dr_csv_path"])
        ):
            item["dr_df"] = load_apple_dr_csv(row["dr_csv_path"])

        items.append(item)

    return items

In [38]:
def _methods_available_for_subset(
    exp_log, subset_clients, model_tag, methods, local_epochs=1
):
    ok = []
    for method in methods:
        part = filter_method_manifest(
            exp_log,
            subset_clients=subset_clients,
            model_tag=model_tag,
            method=method,
            local_epochs=local_epochs,
        )
        if len(part) > 0:
            ok.append(method)
    return ok


OLD_HETEROGENEITY_LEVELS = [
    ("low", 0.07),
    ("mid", 0.14),
    ("high", 0.21),
]

MASKED_HETEROGENEITY_LEVELS = [
    ("low", 0.20),
    ("mid", 0.50),
    ("high", 0.80),
]

SPECIALIZED_HETEROGENEITY_LEVELS = [
    ("low", 0.20),
    ("mid", 0.50),
    ("high", 0.80),
]

HETEROGENEITY_LEVEL_ORDER = ["low", "mid", "high"]
HETEROGENEITY_LEVEL_TO_ORDER = {
    name: idx for idx, name in enumerate(HETEROGENEITY_LEVEL_ORDER)
}


def _benchmark_name_from_row(row):
    for col in ["controlled_benchmark", "controlled_benchmark_manifest"]:
        if col in row.index and pd.notna(row.get(col, np.nan)):
            return str(row[col])
    return ""


def _is_specialized_row(row):
    benchmark = _benchmark_name_from_row(row)

    if benchmark == "task_specialized_label_masking":
        return True

    if "specialization_fraction" in row.index and pd.notna(
        row.get("specialization_fraction", np.nan)
    ):
        return True

    family = str(row.get("family", ""))
    return family.startswith("specialized_")


def _is_masked_row(row):
    benchmark = _benchmark_name_from_row(row)

    if benchmark in {
        "task_positive_label_masking",
        "task_specialized_label_masking",
    }:
        return True

    if "mask_fraction" in row.index and pd.notna(row.get("mask_fraction", np.nan)):
        return True

    return False


def _heterogeneity_value_from_row(row):
    """
    Unified heterogeneity knob:
    - old experiment: gamma
    - masked experiment: mask_fraction
    - specialized experiment: specialization_fraction / mask_fraction
    """
    if "specialization_fraction" in row.index and pd.notna(
        row.get("specialization_fraction", np.nan)
    ):
        return float(row["specialization_fraction"])

    if "mask_fraction" in row.index and pd.notna(row.get("mask_fraction", np.nan)):
        return float(row["mask_fraction"])

    if "designed_heterogeneity" in row.index and pd.notna(
        row.get("designed_heterogeneity", np.nan)
    ):
        return float(row["designed_heterogeneity"])

    if "gamma" in row.index and pd.notna(row.get("gamma", np.nan)):
        return float(row["gamma"])

    sid = str(row.get("subset_id", ""))

    import re

    m = re.search(r"_x([0-9]+(?:\.[0-9]+)?)_", sid)
    if m:
        return float(m.group(1))

    m = re.search(r"gamma_([0-9]+p[0-9]+)", sid)
    if m:
        return float(m.group(1).replace("p", "."))

    return np.nan


def _gamma_from_row(row):
    return _heterogeneity_value_from_row(row)


def _heterogeneity_level_from_value(value, row=None):
    if value is None or pd.isna(value):
        return "unknown"

    value = float(value)

    if row is not None and _is_specialized_row(row):
        levels = SPECIALIZED_HETEROGENEITY_LEVELS
    elif row is not None and _is_masked_row(row):
        levels = MASKED_HETEROGENEITY_LEVELS
    else:
        levels = OLD_HETEROGENEITY_LEVELS

    best_name, best_value = min(
        levels,
        key=lambda pair: abs(value - float(pair[1])),
    )

    if abs(value - float(best_value)) < 1e-6:
        return best_name

    if row is not None and _is_specialized_row(row):
        return f"specialization={value:.3g}"

    if row is not None and _is_masked_row(row):
        return f"mask={value:.3g}"

    return f"gamma={value:.3g}"


def _heterogeneity_label_from_row(row, title_case=False):
    if "heterogeneity_level" in row.index and pd.notna(
        row.get("heterogeneity_level", np.nan)
    ):
        level = str(row["heterogeneity_level"])
    else:
        value = _heterogeneity_value_from_row(row)
        level = _heterogeneity_level_from_value(value, row=row)

    return level.title() if title_case else level


def _heterogeneity_order_idx(level):
    return HETEROGENEITY_LEVEL_TO_ORDER.get(str(level).lower(), 999)


HETEROGENEITY_COLS = [
    "task_profile_jsd_mean",
    "task_profile_jsd_median",
    "task_profile_jsd_max",
    "between_family_centroid_jsd_mean",
] + [f"{task}_pos_rate_std_across_clients" for task in TASKS]

METHOD_ORDER = ["fedavg", "fedprox", "gcfl_plus", "apple"]

METHOD_LABELS = {
    "local": "Local",
    "fedavg": "FedAvg",
    "fedprox": "FedProx",
    "gcfl_plus": "GCFL+",
    "apple": "APPLE",
}

POST_SCALAR_PHASE_BY_METHOD = {
    "fedavg": "global_val_client",
    "fedprox": "global_val_client",
    "gcfl_plus": "cluster_val_client",
    "apple": "val_epoch",
}

POST_TASK_PHASE_BY_METHOD = {
    "fedavg": "global_val_client_task",
    "fedprox": "global_val_client_task",
    "gcfl_plus": "cluster_val_client_task",
    "apple": "val_epoch_task",
}

BEST_PHASE_PREFIX_BY_METHOD = {
    "local": "best_local",
    "fedavg": "best_global",
    "fedprox": "best_global",
    "gcfl_plus": "best_cluster",
    "apple": "best_apple",
}


def _subset_heterogeneity_payload(sel_row):
    out = {}
    for col in HETEROGENEITY_COLS:
        if col in sel_row.index and pd.notna(sel_row[col]):
            out[col] = float(sel_row[col])
    return out


def build_global_run_table(
    selected_subset_df, exp_log, methods=METHOD_ORDER, local_epochs=1
):
    rows = []

    for _, sel_row in selected_subset_df.iterrows():
        subset_clients = str(sel_row["subset_clients"])
        graph_ids = [str(x) for x in parse_json_list_safe(sel_row["graph_ids_json"])]
        family_order = parse_json_list_safe(sel_row["family_order_json"])

        family_to_graph_ids = {
            str(k): [str(v) for v in vals]
            for k, vals in parse_json_dict_safe(
                sel_row["family_to_graph_ids_json"]
            ).items()
        }

        family_counts = parse_json_dict_safe(sel_row["family_counts_json"])

        heterogeneity_value = _heterogeneity_value_from_row(sel_row)
        heterogeneity_level = _heterogeneity_label_from_row(sel_row)
        heterogeneity_order_idx = _heterogeneity_order_idx(heterogeneity_level)

        candidate_models = sorted(exp_log["model_tag"].dropna().astype(str).unique())

        for model_tag in candidate_models:
            local_rows = filter_local_manifest(
                exp_log,
                subset_clients=subset_clients,
                model_tag=model_tag,
                graph_ids=graph_ids,
                local_epochs=local_epochs,
            )
            if len(local_rows) == 0:
                continue

            available_methods = _methods_available_for_subset(
                exp_log,
                subset_clients=subset_clients,
                model_tag=model_tag,
                methods=methods,
                local_epochs=local_epochs,
            )
            if len(available_methods) == 0:
                continue

            rows.append(
                {
                    "subset_id": sel_row.get("subset_id", None),
                    "subset_clients": subset_clients,
                    "subset_size": int(sel_row.get("subset_size", len(graph_ids))),
                    "graph_ids": graph_ids,
                    "family_order": (
                        family_order if len(family_order) > 0 else FAMILY_ORDER
                    ),
                    "family_to_graph_ids": family_to_graph_ids,
                    "family_counts": family_counts,
                    # Backward-compatible + new heterogeneity fields
                    "gamma": heterogeneity_value,
                    "heterogeneity_value": heterogeneity_value,
                    "mask_fraction": sel_row.get("mask_fraction", np.nan),
                    "specialization_fraction": sel_row.get(
                        "specialization_fraction", np.nan
                    ),
                    "designed_heterogeneity": sel_row.get(
                        "designed_heterogeneity", np.nan
                    ),
                    "global_visible_support_fraction_ideal": sel_row.get(
                        "global_visible_support_fraction_ideal", np.nan
                    ),
                    "controlled_benchmark": sel_row.get("controlled_benchmark", None),
                    "mask_mode": sel_row.get("mask_mode", None),
                    "heterogeneity_level": heterogeneity_level,
                    "heterogeneity_order_idx": heterogeneity_order_idx,
                    "model_tag": model_tag,
                    "local_epochs": int(local_epochs),
                    "methods": available_methods,
                    **_subset_heterogeneity_payload(sel_row),
                }
            )

    out = pd.DataFrame(rows)
    if not out.empty:
        out = out.sort_values(["heterogeneity_order_idx", "model_tag"]).reset_index(
            drop=True
        )

    return out


def build_family_run_table(
    selected_subset_df, exp_log, methods=METHOD_ORDER, local_epochs=1
):
    rows = []

    for _, sel_row in selected_subset_df.iterrows():
        subset_clients = str(sel_row["subset_clients"])
        family_order = parse_json_list_safe(sel_row["family_order_json"])

        family_to_graph_ids = {
            str(k): [str(v) for v in vals]
            for k, vals in parse_json_dict_safe(
                sel_row["family_to_graph_ids_json"]
            ).items()
        }

        heterogeneity_value = _heterogeneity_value_from_row(sel_row)
        heterogeneity_level = _heterogeneity_label_from_row(sel_row)
        heterogeneity_order_idx = _heterogeneity_order_idx(heterogeneity_level)

        candidate_models = sorted(exp_log["model_tag"].dropna().astype(str).unique())

        for model_tag in candidate_models:
            available_methods = _methods_available_for_subset(
                exp_log,
                subset_clients=subset_clients,
                model_tag=model_tag,
                methods=methods,
                local_epochs=local_epochs,
            )
            if len(available_methods) == 0:
                continue

            ordered_families = family_order if len(family_order) > 0 else FAMILY_ORDER

            for fam_idx, family in enumerate(ordered_families):
                graph_ids = family_to_graph_ids.get(str(family), [])
                if not graph_ids:
                    continue

                rows.append(
                    {
                        "subset_id": sel_row.get("subset_id", None),
                        "subset_clients": subset_clients,
                        "family": str(family),
                        "family_order_idx": fam_idx,
                        "subset_size": len(graph_ids),
                        "graph_ids": [str(g) for g in graph_ids],
                        # Backward-compatible + new heterogeneity fields
                        "gamma": heterogeneity_value,
                        "heterogeneity_value": heterogeneity_value,
                        "mask_fraction": sel_row.get("mask_fraction", np.nan),
                        "specialization_fraction": sel_row.get(
                            "specialization_fraction", np.nan
                        ),
                        "designed_heterogeneity": sel_row.get(
                            "designed_heterogeneity", np.nan
                        ),
                        "global_visible_support_fraction_ideal": sel_row.get(
                            "global_visible_support_fraction_ideal", np.nan
                        ),
                        "controlled_benchmark": sel_row.get(
                            "controlled_benchmark", None
                        ),
                        "mask_mode": sel_row.get("mask_mode", None),
                        "heterogeneity_level": heterogeneity_level,
                        "heterogeneity_order_idx": heterogeneity_order_idx,
                        "model_tag": model_tag,
                        "local_epochs": int(local_epochs),
                        "methods": available_methods,
                        **_subset_heterogeneity_payload(sel_row),
                    }
                )

    out = pd.DataFrame(rows)
    if not out.empty:
        out = out.sort_values(
            ["heterogeneity_order_idx", "family_order_idx", "model_tag"]
        ).reset_index(drop=True)

    return out


def build_client_run_table(
    selected_subset_df, exp_log, methods=METHOD_ORDER, local_epochs=1
):
    rows = []

    for _, sel_row in selected_subset_df.iterrows():
        subset_clients = str(sel_row["subset_clients"])
        family_order = parse_json_list_safe(sel_row["family_order_json"])
        family_to_graph_ids = {
            str(k): [str(v) for v in vals]
            for k, vals in parse_json_dict_safe(
                sel_row["family_to_graph_ids_json"]
            ).items()
        }
        graph_to_family = {
            str(k): str(v)
            for k, v in parse_json_dict_safe(sel_row["graph_to_family_json"]).items()
        }

        heterogeneity_value = _heterogeneity_value_from_row(sel_row)
        heterogeneity_level = _heterogeneity_label_from_row(sel_row)
        heterogeneity_order_idx = _heterogeneity_order_idx(heterogeneity_level)

        candidate_models = sorted(exp_log["model_tag"].dropna().astype(str).unique())

        for model_tag in candidate_models:
            available_methods = _methods_available_for_subset(
                exp_log,
                subset_clients=subset_clients,
                model_tag=model_tag,
                methods=methods,
                local_epochs=local_epochs,
            )
            if len(available_methods) == 0:
                continue

            ordered_families = family_order if len(family_order) > 0 else FAMILY_ORDER

            for fam_idx, family in enumerate(ordered_families):
                ordered_graph_ids = [
                    str(g) for g in family_to_graph_ids.get(str(family), [])
                ]

                for graph_idx, graph_id in enumerate(ordered_graph_ids):

                    rows.append(
                        {
                            "subset_id": sel_row.get("subset_id", None),
                            "subset_clients": subset_clients,
                            "family": graph_to_family.get(str(graph_id), str(family)),
                            "family_order_idx": fam_idx,
                            "graph_order_idx": graph_idx,
                            "graph_id": str(graph_id),
                            # Backward-compatible + new heterogeneity fields
                            "gamma": heterogeneity_value,
                            "heterogeneity_value": heterogeneity_value,
                            "mask_fraction": sel_row.get("mask_fraction", np.nan),
                            "specialization_fraction": sel_row.get(
                                "specialization_fraction", np.nan
                            ),
                            "designed_heterogeneity": sel_row.get(
                                "designed_heterogeneity", np.nan
                            ),
                            "global_visible_support_fraction_ideal": sel_row.get(
                                "global_visible_support_fraction_ideal", np.nan
                            ),
                            "controlled_benchmark": sel_row.get(
                                "controlled_benchmark", None
                            ),
                            "mask_mode": sel_row.get("mask_mode", None),
                            "heterogeneity_level": heterogeneity_level,
                            "heterogeneity_order_idx": heterogeneity_order_idx,
                            "model_tag": model_tag,
                            "local_epochs": int(local_epochs),
                            "methods": available_methods,
                            **_subset_heterogeneity_payload(sel_row),
                        }
                    )

    out = pd.DataFrame(rows)
    if not out.empty:
        out = out.sort_values(
            [
                "heterogeneity_order_idx",
                "family_order_idx",
                "graph_order_idx",
                "model_tag",
            ]
        ).reset_index(drop=True)
    return out

In [39]:
def get_group_pooled_task_f1_postagg(
    *,
    method_items,
    task,
    graph_ids,
    phase="global_val_client_task",
    split="val",
):
    target_graph_ids = [str(g) for g in graph_ids]
    curves = []

    for item in method_items:
        count_curves = []
        df = item["df"]

        for gid in target_graph_ids:
            count_curve = get_task_count_curve(
                df,
                phase=phase,
                split=split,
                task=task,
                graph_id=gid,
            )
            count_curves.append(count_curve)

        pooled = _pool_count_curves(count_curves)
        if pooled.empty:
            continue

        tp = pooled["tp"].to_numpy(dtype=float)
        fp = pooled["fp"].to_numpy(dtype=float)
        fn = pooled["fn"].to_numpy(dtype=float)
        pooled["pooled_f1"] = binary_f1_from_counts(tp, fp, fn)

        curves.append(pooled[["step", "pooled_f1"]])

    return aggregate_seed_curves(curves, "pooled_f1")


def _pooled_micro_macro_from_task_counts_df(task_counts_df):
    if task_counts_df is None or task_counts_df.empty:
        return pd.DataFrame(columns=["step", "micro_f1", "macro_f1"])

    df = task_counts_df.copy()
    for c in ["tp", "fp", "tn", "fn"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df["step"] = pd.to_numeric(df["step"], errors="coerce")
    df = df.dropna(subset=["step", "task", "tp", "fp", "tn", "fn"]).copy()
    if df.empty:
        return pd.DataFrame(columns=["step", "micro_f1", "macro_f1"])

    df["step"] = df["step"].astype(int)

    by_task = (
        df.groupby(["step", "task"], as_index=False)[["tp", "fp", "tn", "fn"]]
        .sum()
        .sort_values(["step", "task"])
        .reset_index(drop=True)
    )

    by_task["task_f1"] = binary_f1_from_counts(
        by_task["tp"].to_numpy(dtype=float),
        by_task["fp"].to_numpy(dtype=float),
        by_task["fn"].to_numpy(dtype=float),
    )

    macro_df = (
        by_task.groupby("step", as_index=False)["task_f1"]
        .mean()
        .rename(columns={"task_f1": "macro_f1"})
    )

    micro_counts = (
        by_task.groupby("step", as_index=False)[["tp", "fp", "fn"]]
        .sum()
        .sort_values("step")
        .reset_index(drop=True)
    )
    micro_counts["micro_f1"] = binary_f1_from_counts(
        micro_counts["tp"].to_numpy(dtype=float),
        micro_counts["fp"].to_numpy(dtype=float),
        micro_counts["fn"].to_numpy(dtype=float),
    )

    out = micro_counts[["step", "micro_f1"]].merge(
        macro_df[["step", "macro_f1"]],
        on="step",
        how="outer",
    )
    out = out.sort_values("step").reset_index(drop=True)
    return out


def _seed_local_task_counts_df(
    seed_local_items, graph_ids, split="val", phase="val_epoch_task"
):
    target_graph_ids = {str(g) for g in graph_ids}
    frames = []

    for task in TASKS:
        count_curves = []
        for item in seed_local_items:
            if str(item["graph_id"]) not in target_graph_ids:
                continue

            count_curve = get_task_count_curve(
                item["df"],
                phase=phase,
                split=split,
                task=task,
                graph_id=None,
            )
            if not count_curve.empty:
                count_curves.append(count_curve)

        pooled = _pool_count_curves(count_curves)
        if pooled.empty:
            continue

        pooled = pooled.copy()
        pooled["task"] = task
        frames.append(pooled)

    if len(frames) == 0:
        return pd.DataFrame(columns=["step", "task", "tp", "fp", "tn", "fn"])

    return pd.concat(frames, axis=0, ignore_index=True)


def _seed_method_task_counts_df(
    method_item, graph_ids, *, split="val", phase="global_val_client_task"
):
    target_graph_ids = [str(g) for g in graph_ids]
    frames = []
    df = method_item["df"]

    for task in TASKS:
        count_curves = []
        for gid in target_graph_ids:
            count_curve = get_task_count_curve(
                df,
                phase=phase,
                split=split,
                task=task,
                graph_id=gid,
            )
            if not count_curve.empty:
                count_curves.append(count_curve)

        pooled = _pool_count_curves(count_curves)
        if pooled.empty:
            continue

        pooled = pooled.copy()
        pooled["task"] = task
        frames.append(pooled)

    if len(frames) == 0:
        return pd.DataFrame(columns=["step", "task", "tp", "fp", "tn", "fn"])

    return pd.concat(frames, axis=0, ignore_index=True)


def _seed_local_pooled_micro_macro_curve(
    seed_local_items, graph_ids, split="val", phase="val_epoch_task"
):
    task_counts_df = _seed_local_task_counts_df(
        seed_local_items,
        graph_ids,
        split=split,
        phase=phase,
    )
    return _pooled_micro_macro_from_task_counts_df(task_counts_df)


def _seed_method_pooled_micro_macro_curve(
    method_item, graph_ids, *, split="val", phase="global_val_client_task"
):
    task_counts_df = _seed_method_task_counts_df(
        method_item,
        graph_ids,
        split=split,
        phase=phase,
    )
    return _pooled_micro_macro_from_task_counts_df(task_counts_df)


def get_group_pooled_micro_macro_local(local_items, graph_ids, split="val"):
    micro_curves, macro_curves = [], []

    seeds = sorted({int(item["seed"]) for item in local_items})
    for seed in seeds:
        seed_local_items = [item for item in local_items if int(item["seed"]) == seed]
        pooled = _seed_local_pooled_micro_macro_curve(
            seed_local_items,
            graph_ids,
            split=split,
            phase="val_epoch_task",
        )
        if pooled.empty:
            continue

        micro_curves.append(pooled[["step", "micro_f1"]].copy())
        macro_curves.append(pooled[["step", "macro_f1"]].copy())

    return {
        "micro_f1": aggregate_seed_curves(micro_curves, "micro_f1"),
        "macro_f1": aggregate_seed_curves(macro_curves, "macro_f1"),
    }


def get_group_pooled_micro_macro_method(
    *,
    method_items,
    graph_ids,
    phase,
    split="val",
):
    micro_curves, macro_curves = [], []

    for item in method_items:
        pooled = _seed_method_pooled_micro_macro_curve(
            item,
            graph_ids,
            split=split,
            phase=phase,
        )
        if pooled.empty:
            continue

        micro_curves.append(pooled[["step", "micro_f1"]].copy())
        macro_curves.append(pooled[["step", "macro_f1"]].copy())

    return {
        "micro_f1": aggregate_seed_curves(micro_curves, "micro_f1"),
        "macro_f1": aggregate_seed_curves(macro_curves, "macro_f1"),
    }


def get_group_delta_local_minus_method_micro_macro(
    *,
    local_items,
    method_items,
    graph_ids,
    method_phase,
    split="val",
):
    method_by_seed = {int(item["seed"]): item for item in method_items}

    delta_micro_curves = []
    delta_macro_curves = []

    seeds = sorted({int(item["seed"]) for item in local_items})
    for seed in seeds:
        if seed not in method_by_seed:
            continue

        seed_local_items = [item for item in local_items if int(item["seed"]) == seed]

        local_curve = _seed_local_pooled_micro_macro_curve(
            seed_local_items,
            graph_ids,
            split=split,
            phase="val_epoch_task",
        )
        method_curve = _seed_method_pooled_micro_macro_curve(
            method_by_seed[seed],
            graph_ids,
            split=split,
            phase=method_phase,
        )

        if local_curve.empty or method_curve.empty:
            continue

        merged = local_curve.merge(
            method_curve,
            on="step",
            how="inner",
            suffixes=("_local", "_method"),
        )
        if merged.empty:
            continue

        delta_micro = merged[["step"]].copy()
        delta_micro["delta_micro_f1"] = (
            merged["micro_f1_local"] - merged["micro_f1_method"]
        )

        delta_macro = merged[["step"]].copy()
        delta_macro["delta_macro_f1"] = (
            merged["macro_f1_local"] - merged["macro_f1_method"]
        )

        delta_micro_curves.append(delta_micro)
        delta_macro_curves.append(delta_macro)

    return {
        "micro_f1": aggregate_seed_curves(delta_micro_curves, "delta_micro_f1"),
        "macro_f1": aggregate_seed_curves(delta_macro_curves, "delta_macro_f1"),
    }

def get_group_delta_method_minus_method_micro_macro(
    *,
    method_items,
    graph_ids,
    phase_a,
    phase_b,
    split="val",
):
    delta_micro_curves = []
    delta_macro_curves = []

    for item in method_items:
        curve_a = _seed_method_pooled_micro_macro_curve(
            item,
            graph_ids,
            split=split,
            phase=phase_a,
        )
        curve_b = _seed_method_pooled_micro_macro_curve(
            item,
            graph_ids,
            split=split,
            phase=phase_b,
        )

        if curve_a.empty or curve_b.empty:
            continue

        merged = curve_a.merge(
            curve_b,
            on="step",
            how="inner",
            suffixes=("_a", "_b"),
        )

        if merged.empty:
            continue

        micro = merged[["step"]].copy()
        micro["delta_micro_f1"] = merged["micro_f1_a"] - merged["micro_f1_b"]

        macro = merged[["step"]].copy()
        macro["delta_macro_f1"] = merged["macro_f1_a"] - merged["macro_f1_b"]

        delta_micro_curves.append(micro)
        delta_macro_curves.append(macro)

    return {
        "micro_f1": aggregate_seed_curves(delta_micro_curves, "delta_micro_f1"),
        "macro_f1": aggregate_seed_curves(delta_macro_curves, "delta_macro_f1"),
    }
    
    
def get_weighted_group_loss_for_method(
    *,
    method_items,
    phase,
    graph_ids,
    split="val",
):
    return get_weighted_group_scalar_stats(
        items=method_items,
        phase=phase,
        split=split,
        metric_col="eval_loss",
        graph_ids=graph_ids,
        use_graph_filter=True,
    )


def get_pooled_delta_local_minus_method(
    *,
    local_items,
    method_items,
    task,
    graph_ids,
    split="val",
):
    target_graph_ids = {str(g) for g in graph_ids}
    method_by_seed = {int(item["seed"]): item for item in method_items}
    delta_curves = []

    seeds = sorted({int(item["seed"]) for item in local_items})
    for seed in seeds:
        if seed not in method_by_seed:
            continue

        # pooled local
        local_count_curves = []
        seed_local_items = [
            item
            for item in local_items
            if int(item["seed"]) == seed and str(item["graph_id"]) in target_graph_ids
        ]
        for item in seed_local_items:
            count_curve = get_task_count_curve(
                item["df"],
                phase="val_epoch_task",
                split=split,
                task=task,
                graph_id=None,
            )
            local_count_curves.append(count_curve)

        pooled_local = _pool_count_curves(local_count_curves)
        if pooled_local.empty:
            continue
        pooled_local["pooled_f1"] = binary_f1_from_counts(
            pooled_local["tp"].to_numpy(dtype=float),
            pooled_local["fp"].to_numpy(dtype=float),
            pooled_local["fn"].to_numpy(dtype=float),
        )

        # pooled method postagg
        method_df = method_by_seed[seed]["df"]
        method_count_curves = []
        for gid in target_graph_ids:
            count_curve = get_task_count_curve(
                method_df,
                phase="global_val_client_task",
                split=split,
                task=task,
                graph_id=gid,
            )
            method_count_curves.append(count_curve)

        pooled_method = _pool_count_curves(method_count_curves)
        if pooled_method.empty:
            continue
        pooled_method["pooled_f1"] = binary_f1_from_counts(
            pooled_method["tp"].to_numpy(dtype=float),
            pooled_method["fp"].to_numpy(dtype=float),
            pooled_method["fn"].to_numpy(dtype=float),
        )

        merged = pd.merge(
            pooled_local[["step", "pooled_f1"]],
            pooled_method[["step", "pooled_f1"]],
            on="step",
            how="inner",
            suffixes=("_local", "_method"),
        )
        if merged.empty:
            continue

        merged["delta"] = merged["pooled_f1_local"] - merged["pooled_f1_method"]
        delta_curves.append(merged[["step", "delta"]])

    return aggregate_seed_curves(delta_curves, "delta")

In [40]:
def _level_title(row):
    return _heterogeneity_label_from_row(row, title_case=True)


def _level_text(row):
    return _heterogeneity_label_from_row(row, title_case=False)


def _benchmark_prefix(row):
    if _is_specialized_row(row):
        return "Controlled task-specialization level"
    if _is_masked_row(row):
        return "Controlled masking level"
    return "Heterogeneity level"


def build_overview_global_title(row):
    return (
        f"{_benchmark_prefix(row)}: {_level_title(row)}"
        f" | aggregated / pooled performance over {int(row['subset_size'])} clients"
        f" | model={row['model_tag']}"
    )


def build_overview_family_title(row):
    return (
        f"{_benchmark_prefix(row)}: {_level_title(row)}"
        f" | family aggregated / pooled performance"
        f" | family={row['family']}"
        f" | n_clients={int(row['subset_size'])}"
        f" | model={row['model_tag']}"
    )


def build_method_global_title(method, row):
    return (
        f"{_benchmark_prefix(row)}: {_level_title(row)}"
        f" | {METHOD_LABELS[method]} diagnostics"
        f" | aggregated / pooled over {int(row['subset_size'])} clients"
        f" | model={row['model_tag']}"
    )


def build_method_family_title(method, row):
    return (
        f"{_benchmark_prefix(row)}: {_level_title(row)}"
        f" | {METHOD_LABELS[method]} diagnostics"
        f" | family={row['family']}"
        f" | n_clients={int(row['subset_size'])}"
        f" | model={row['model_tag']}"
    )


def build_gcfl_seed_title(row, seed_pack, section_title):
    return (
        f"{_benchmark_prefix(row)}: {_level_title(row)}"
        f" | GCFL+ diagnostics"
        f" | aggregated / pooled over {int(row['subset_size'])} clients"
        f" | seed={int(seed_pack['seed'])} | {section_title}"
    )


def _compose_global_overview_header_text(row, test_gen):
    return _compose_header_text(_global_family_rate_text(row, test_gen), row)


def _compose_family_overview_header_text(row, test_gen):
    return _compose_header_text(
        _family_rate_text(row["graph_ids"], test_gen, family_name=row["family"]),
        row,
    )


def _compose_method_global_header_text(method, row, test_gen):
    return _compose_header_text(_global_family_rate_text(row, test_gen), row)


def _compose_method_family_header_text(method, row, test_gen):
    return _compose_header_text(
        _family_rate_text(row["graph_ids"], test_gen, family_name=row["family"]),
        row,
    )


def _compose_gcfl_seed_header_text(row, test_gen):
    base = _compose_header_text(_global_family_rate_text(row, test_gen), row)
    return f"model={row['model_tag']}\n{base}"

## 2. Curve utilities

In [41]:
def aggregate_seed_curves(curves, value_col):
    parts = []
    for seed_idx, curve in enumerate(curves):
        if curve is None or curve.empty:
            continue
        part = curve[["step", value_col]].dropna().copy()
        if part.empty:
            continue
        part["seed_idx"] = seed_idx
        parts.append(part)

    if not parts:
        return pd.DataFrame(columns=["step", "mean", "std", "count"])

    full = pd.concat(parts, axis=0, ignore_index=True)
    agg = (
        full.groupby("step")[value_col]
        .agg(["mean", "std", "count"])
        .reset_index()
        .sort_values("step")
    )
    agg["std"] = agg["std"].fillna(0.0)
    return agg


def get_task_count_curve(
    df,
    *,
    phase,
    split,
    task,
    graph_id=None,
):
    part = df[
        (df["phase"] == phase) & (df["split"] == split) & (df["task"] == task)
    ].copy()

    if graph_id is not None and "graph_id" in part.columns:
        part = part[part["graph_id"].astype(str) == str(graph_id)].copy()

    needed = ["tp", "fp", "tn", "fn"]
    if part.empty or any(c not in part.columns for c in needed):
        return pd.DataFrame(columns=["step", "tp", "fp", "tn", "fn"])

    part["step"] = part.apply(_effective_step, axis=1)
    part = part[pd.notna(part["step"])].copy()
    if part.empty:
        return pd.DataFrame(columns=["step", "tp", "fp", "tn", "fn"])

    part["step"] = part["step"].astype(int)

    for c in needed:
        part[c] = pd.to_numeric(part[c], errors="coerce")

    part = part.dropna(subset=needed).copy()
    if part.empty:
        return pd.DataFrame(columns=["step", "tp", "fp", "tn", "fn"])

    out = (
        part.groupby("step", as_index=False)[needed]
        .sum()
        .sort_values("step")
        .reset_index(drop=True)
    )
    return out


def binary_f1_from_counts(tp, fp, fn):
    denom = 2.0 * tp + fp + fn
    out = np.full_like(tp, np.nan, dtype=float)
    valid = denom > 0
    out[valid] = (2.0 * tp[valid]) / denom[valid]
    return out


def plot_mean_std(
    ax,
    agg_df,
    *,
    label,
    linestyle="-",
    color=None,
    alpha_fill=0.16,
    linewidth=2.0,
):
    if agg_df is None or agg_df.empty:
        return False

    x = agg_df["step"].to_numpy()
    y = agg_df["mean"].to_numpy()
    s = agg_df["std"].to_numpy()

    (line,) = ax.plot(
        x,
        y,
        label=label,
        linestyle=linestyle,
        color=color,
        linewidth=linewidth,
    )
    fill_color = line.get_color()
    ax.fill_between(x, y - s, y + s, alpha=alpha_fill, color=fill_color)
    return True


def set_round_budget_axis(
    ax,
    *,
    max_round,
    show_baseline_budget_marker=False,
):
    ax.set_xlim(0, max_round)

    # Only show the marker when APPLE and baselines have different round budgets.
    if show_baseline_budget_marker and int(BASELINE_ROUND_BUDGET) != int(max_round):
        ax.axvline(
            BASELINE_ROUND_BUDGET,
            linestyle=":",
            linewidth=1.4,
            color="black",
            alpha=0.75,
            label=f"baseline budget = {BASELINE_ROUND_BUDGET} rounds",
        )


def annotate_no_data(ax, text="No data"):
    ax.text(
        0.5,
        0.5,
        text,
        ha="center",
        va="center",
        transform=ax.transAxes,
        fontsize=10,
        color="gray",
    )

## 3. Group aggregation

In [42]:
def _detect_weight_col(df):
    candidates = ["num_nodes", "n_nodes", "node_count", "num_samples"]
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(
        "Could not find a node-count column in the run CSV. "
        "Expected one of: num_nodes, n_nodes, node_count, num_samples"
    )


def get_weighted_group_scalar_stats(
    *,
    items,
    phase,
    split,
    metric_col,
    graph_ids=None,
    use_graph_filter=True,
):
    seed_curves = []

    for item in items:
        df = item["df"].copy()

        part = df[(df["phase"] == phase) & (df["split"] == split)].copy()
        if part.empty or metric_col not in part.columns:
            continue

        if use_graph_filter and graph_ids is not None and "graph_id" in part.columns:
            part = part[
                part["graph_id"].astype(str).isin([str(g) for g in graph_ids])
            ].copy()

        if part.empty:
            continue

        weight_col = _detect_weight_col(part)

        part["step"] = part.apply(_effective_step, axis=1)
        part = part[pd.notna(part["step"])].copy()
        if part.empty:
            continue

        part["step"] = part["step"].astype(int)

        part[metric_col] = pd.to_numeric(part[metric_col], errors="coerce")
        part[weight_col] = pd.to_numeric(part[weight_col], errors="coerce")

        part = part[
            pd.notna(part[metric_col])
            & pd.notna(part[weight_col])
            & (part[weight_col] > 0)
        ].copy()
        if part.empty:
            continue

        grouped = (
            part.groupby("step", as_index=False)
            .apply(
                lambda g: pd.Series(
                    {
                        metric_col: np.average(
                            g[metric_col].to_numpy(dtype=float),
                            weights=g[weight_col].to_numpy(dtype=float),
                        )
                    }
                )
            )
            .reset_index(drop=True)
            .sort_values("step")
        )

        if not grouped.empty:
            seed_curves.append(grouped[["step", metric_col]])

    return aggregate_seed_curves(seed_curves, metric_col)


def _pool_count_curves(curves):
    good = []
    for curve in curves:
        if curve is None or curve.empty:
            continue
        good.append(curve[["step", "tp", "fp", "tn", "fn"]].copy())

    if not good:
        return pd.DataFrame(columns=["step", "tp", "fp", "tn", "fn"])

    full = pd.concat(good, axis=0, ignore_index=True)
    pooled = (
        full.groupby("step")[["tp", "fp", "tn", "fn"]]
        .sum()
        .reset_index()
        .sort_values("step")
    )
    return pooled


def get_group_pooled_task_f1_local(
    *,
    local_items,
    task,
    graph_ids,
    split="val",
):
    target_graph_ids = {str(g) for g in graph_ids}
    curves = []

    seeds = sorted({int(item["seed"]) for item in local_items})

    for seed in seeds:
        count_curves = []
        seed_items = [
            item
            for item in local_items
            if int(item["seed"]) == seed and str(item["graph_id"]) in target_graph_ids
        ]

        for item in seed_items:
            count_curve = get_task_count_curve(
                item["df"],
                phase="val_epoch_task",
                split=split,
                task=task,
                graph_id=None,
            )
            count_curves.append(count_curve)

        pooled = _pool_count_curves(count_curves)
        if pooled.empty:
            continue

        tp = pooled["tp"].to_numpy(dtype=float)
        fp = pooled["fp"].to_numpy(dtype=float)
        fn = pooled["fn"].to_numpy(dtype=float)

        pooled["pooled_f1"] = binary_f1_from_counts(tp, fp, fn)
        curves.append(pooled[["step", "pooled_f1"]])

    return aggregate_seed_curves(curves, "pooled_f1")


def _rate_col_from_test_gen(task, test_gen):
    candidates = [
        f"train_{task}_pos_rate",
        f"val_{task}_pos_rate",
        f"test_{task}_pos_rate",
        f"p_{task}",
        task,
    ]
    for c in candidates:
        if c in test_gen.columns:
            return c
    raise KeyError(
        f"Could not find a rate column for task={task}. " f"Tried: {candidates}"
    )


def _graph_rate_text(graph_id, test_gen):
    part = test_gen[test_gen["graph_id"].astype(str) == str(graph_id)].copy()
    if part.empty:
        return f"graph_id={graph_id} | pos rates: unavailable"

    vals = []
    for task in TASKS:
        col = _rate_col_from_test_gen(task, test_gen)
        v = float(part.iloc[0][col])
        vals.append(f"{task}={100*v:.1f}%")

    return f"graph_id={graph_id} | " + " | ".join(vals)


def _family_rate_text(graph_ids, test_gen, family_name=None):
    graph_ids = {str(g) for g in graph_ids}
    sub = test_gen[test_gen["graph_id"].astype(str).isin(graph_ids)].copy()

    if len(sub) == 0:
        return ""

    parts = []

    if family_name is not None:
        parts.append(f"family={family_name}")
    parts.append(f"n_clients={len(sub)}")

    task_parts = []
    for task in TASKS:
        visible_col = f"train_{task}_pos_rate"
        true_col = f"true_train_{task}_pos_rate"

        if visible_col not in sub.columns:
            continue

        visible_mean = float(sub[visible_col].mean())
        visible_std = float(sub[visible_col].std(ddof=0))

        if true_col in sub.columns:
            true_mean = float(sub[true_col].mean())
            task_parts.append(
                f"{task}=visible {visible_mean*100:.1f}%±{visible_std*100:.1f}%"
                f" / true {true_mean*100:.1f}%"
            )
        else:
            task_parts.append(f"{task}={visible_mean*100:.1f}%±{visible_std*100:.1f}%")

    return " | ".join(parts + task_parts)


def _global_family_rate_text(row, test_gen):
    lines = []

    fam_map = row.get("family_to_graph_ids", None)
    fam_order = row.get("family_order", [])

    if fam_map is None:
        return ""

    if isinstance(fam_map, float) and pd.isna(fam_map):
        return ""

    if not isinstance(fam_map, dict):
        return ""

    if isinstance(fam_order, float) and pd.isna(fam_order):
        fam_order = []

    if not isinstance(fam_order, list):
        fam_order = []

    if len(fam_order) == 0:
        fam_order = list(fam_map.keys())

    for fam in fam_order:
        gids = fam_map.get(fam, [])
        if gids is None:
            gids = []

        text = _family_rate_text(gids, test_gen, family_name=fam)
        if text:
            lines.append(text)

    return "\n".join(lines)


def _fmt_float(x, digits=3):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return "NA"
    return f"{float(x):.{digits}f}"


def _heterogeneity_text(row):
    level = _heterogeneity_label_from_row(row, title_case=False)
    x = _heterogeneity_value_from_row(row)

    if _is_specialized_row(row):
        prefix = f"specialization={level}"
        if pd.notna(x):
            prefix += f" | specialization_fraction={float(x):.2f}"

        if "global_visible_support_fraction_ideal" in row.index and pd.notna(
            row.get("global_visible_support_fraction_ideal", np.nan)
        ):
            prefix += (
                " | global_visible_support="
                f"{float(row['global_visible_support_fraction_ideal']):.2f}"
            )

    elif _is_masked_row(row):
        prefix = f"masking={level}"
        if pd.notna(x):
            prefix += f" | mask_fraction={float(x):.2f}"

    else:
        prefix = f"heterogeneity={level}"
        if pd.notna(x):
            prefix += f" | gamma={float(x):.2f}"

    pieces = [prefix]

    for key, label in [
        ("task_profile_jsd_mean", "task_jsd_mean"),
        ("task_profile_jsd_median", "task_jsd_median"),
        ("task_profile_jsd_max", "task_jsd_max"),
        ("between_family_centroid_jsd_mean", "family_centroid_jsd_mean"),
    ]:
        if key in row.index and pd.notna(row[key]):
            pieces.append(f"{label}={float(row[key]):.3f}")

    return " | ".join(pieces)


def _compose_header_text(base_text, row):
    het = _heterogeneity_text(row)
    if het.strip() == "":
        return base_text
    return f"{base_text}\n{het}"


def _header_top_from_text(text):
    n_lines = max(1, str(text).count("\n") + 1)
    return max(0.74, 0.92 - 0.022 * (n_lines - 1))


def _wrap_header_block(text: str, width: int = 125) -> str:
    if text is None:
        return ""

    lines = []
    for raw in str(text).split("\n"):
        raw = str(raw).strip()
        if raw == "":
            continue
        wrapped = wrap(
            raw,
            width=width,
            break_long_words=False,
            break_on_hyphens=False,
        )
        if len(wrapped) == 0:
            lines.append(raw)
        else:
            lines.extend(wrapped)
    return "\n".join(lines)


def _wrap_header_block(text: str, width: int = 120) -> str:
    if text is None:
        return ""

    lines = []
    for raw in str(text).split("\n"):
        raw = str(raw).strip()
        if raw == "":
            continue
        wrapped = wrap(
            raw,
            width=width,
            break_long_words=False,
            break_on_hyphens=False,
        )
        if len(wrapped) == 0:
            lines.append(raw)
        else:
            lines.extend(wrapped)
    return "\n".join(lines)


def _auto_header_fontsize(n_header_lines):
    """
    Larger font for short headers, smaller font only when the header is dense.
    """
    if n_header_lines <= 3:
        return 8.2
    if n_header_lines <= 6:
        return 7.4
    if n_header_lines <= 9:
        return 6.7
    if n_header_lines <= 12:
        return 6.1
    return 5.6


def _apply_page_header(
    fig,
    title,
    header_text,
    *,
    title_width=92,
    header_width=165,
    title_fontsize="auto",
    header_fontsize="auto",
    min_subplot_top=0.58,
    max_subplot_top=0.82,
    left=0.055,
    right=0.985,
    bottom=0.055,
    hspace=0.42,
    wspace=0.25,
):
    """
    Adaptive title + metadata layout.

    Goal:
      - no overlap between header and plots
      - avoid tiny unreadable headers
      - avoid wasting too much vertical space

    Short metadata blocks get larger font and more room for plots.
    Long metadata blocks get slightly smaller font and lower plot area.
    """
    title_wrapped = "\n".join(
        wrap(
            str(title),
            width=title_width,
            break_long_words=False,
            break_on_hyphens=False,
        )
    )

    header_wrapped = _wrap_header_block(header_text, width=header_width)

    n_title_lines = max(1, title_wrapped.count("\n") + 1)
    n_header_lines = max(0, header_wrapped.count("\n") + 1) if header_wrapped else 0

    if title_fontsize == "auto":
        title_fs = 12.0 if n_title_lines <= 2 else 10.8
    else:
        title_fs = float(title_fontsize)

    auto_header_fs = _auto_header_fontsize(n_header_lines)

    if header_fontsize == "auto":
        header_fs = auto_header_fs
    else:
        # Do not let manually supplied tiny values make the header unreadable.
        header_fs = max(float(header_fontsize), auto_header_fs)

    title_y = 0.988
    header_y = title_y - 0.032 * n_title_lines - 0.010

    # Estimate vertical space used by metadata.
    # This is intentionally conservative but not as aggressive as before.
    if n_header_lines > 0:
        line_height = 0.0125 * (header_fs / 6.5)
        header_height = n_header_lines * line_height
    else:
        header_height = 0.0

    subplot_top = header_y - header_height - 0.040
    subplot_top = min(max_subplot_top, max(min_subplot_top, subplot_top))

    fig.subplots_adjust(
        top=subplot_top,
        left=left,
        right=right,
        bottom=bottom,
        hspace=hspace,
        wspace=wspace,
    )

    fig.text(
        0.5,
        title_y,
        title_wrapped,
        ha="center",
        va="top",
        fontsize=title_fs,
        fontweight="bold",
        linespacing=1.05,
    )

    if header_wrapped:
        fig.text(
            0.5,
            header_y,
            header_wrapped,
            ha="center",
            va="top",
            fontsize=header_fs,
            linespacing=1.06,
        )


def _wrap_cell_text(val, width=28):
    s = "" if pd.isna(val) else str(val)
    s = s.replace("|", ", ")
    parts = wrap(
        s,
        width=width,
        break_long_words=False,
        break_on_hyphens=False,
    )
    return "\n".join(parts) if len(parts) > 0 else s


def _compact_gcfl_summary_df(summary_df: pd.DataFrame) -> pd.DataFrame:
    if summary_df is None or summary_df.empty:
        return pd.DataFrame()

    rename_map = {
        "Seed": "Seed",
        "eps1 q": "eps1 q",
        "eps2 q": "eps2 q",
        "Warmup": "Warmup",
        "Hist len": "Hist",
        "Min cluster": "Min cl",
        "Min child": "Min ch",
        "Triggered rounds": "Triggered",
        "Applied rounds": "Applied",
        "Final active IDs": "Final IDs",
        "Final active sizes": "Final sizes",
    }
    out = summary_df.rename(columns=rename_map).copy()

    for c in ["Triggered", "Applied", "Final IDs", "Final sizes"]:
        if c in out.columns:
            out[c] = out[c].apply(lambda x: _wrap_cell_text(x, width=18))

    return out


def _compact_gcfl_eligible_df(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or df.empty:
        return pd.DataFrame()

    out = df.copy()

    rename_map = {
        "round": "round",
        "cluster_id": "cid",
        "cluster_size": "size",
        "mean_update_norm": "mean",
        "eps1": "eps1",
        "max_update_norm": "max",
        "eps2": "eps2",
        "split_status": "status",
        "split_triggered": "trig",
        "split_applied": "appl",
        "cut_status": "cut",
        "split_rejected_reason": "reject",
        "left_size": "left",
        "right_size": "right",
        "child_balance_ratio": "balance",
        "singleton_graph_id": "singleton",
        "cut_value": "cut_val",
    }
    out = out.rename(columns=rename_map)

    keep_cols = [
        "round",
        "cid",
        "size",
        "mean",
        "eps1",
        "max",
        "eps2",
        "status",
        "trig",
        "appl",
        "cut",
        "reject",
        "left",
        "right",
        "balance",
        "singleton",
        "cut_val",
    ]
    keep_cols = [c for c in keep_cols if c in out.columns]
    out = out[keep_cols].copy()

    for c in ["round", "cid", "size", "trig", "appl", "left", "right", "singleton"]:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce")

    for c in ["round", "cid", "size", "trig", "appl", "left", "right"]:
        if c in out.columns:
            out[c] = out[c].apply(lambda x: "" if pd.isna(x) else int(x))

    for c in ["mean", "eps1", "max", "eps2", "balance", "cut_val"]:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce").apply(
                lambda x: "" if pd.isna(x) else f"{x:.3f}"
            )

    if "reject" in out.columns:
        out["reject"] = out["reject"].fillna("-").astype(str)
        out["reject"] = out["reject"].apply(lambda x: _wrap_cell_text(x, width=18))

    if "status" in out.columns:
        out["status"] = out["status"].fillna("").astype(str)

    if "cut" in out.columns:
        out["cut"] = out["cut"].fillna("").astype(str)

    if "singleton" in out.columns:
        out["singleton"] = out["singleton"].apply(
            lambda x: "" if pd.isna(x) else str(int(x))
        )

    return out


def _compact_gcfl_finals_df(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or df.empty:
        return pd.DataFrame()

    out = df.copy()

    if "members" in out.columns:
        out["members"] = out["members"].apply(lambda x: _wrap_cell_text(x, width=30))
    if "member_indices" in out.columns:
        out["member_indices"] = out["member_indices"].apply(
            lambda x: _wrap_cell_text(x, width=30)
        )
    if "family_mix" in out.columns:
        out["family_mix"] = out["family_mix"].apply(
            lambda x: _wrap_cell_text(x, width=26)
        )

    return out


def _compact_gcfl_lineage_df(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or df.empty:
        return pd.DataFrame()

    out = df.copy()
    for c in ["left_members", "right_members"]:
        if c in out.columns:
            out[c] = out[c].apply(lambda x: _wrap_cell_text(x, width=28))
    return out


def _parse_pipe_members(text):
    if text is None:
        return []

    if isinstance(text, float) and pd.isna(text):
        return []

    s = str(text).strip()
    if s == "" or s.lower() == "nan":
        return []

    return [x.strip() for x in s.split("|") if x.strip() != ""]


def _safe_int(x):
    if pd.isna(x):
        return None
    return int(float(x))


def _gcfl_story_df(seed_pack):
    hist = _gcfl_history_df(seed_pack["history_df"])
    if hist.empty or "split_applied" not in hist.columns:
        return pd.DataFrame()

    part = hist[
        pd.to_numeric(hist["split_applied"], errors="coerce").fillna(0).eq(1)
    ].copy()

    if part.empty:
        return pd.DataFrame()

    graph_to_family = _graph_to_family_map_from_tables()
    rows = []

    for _, rr in part.sort_values(["round", "cluster_id"]).iterrows():
        left_size = _safe_int(rr.get("left_size", np.nan))
        right_size = _safe_int(rr.get("right_size", np.nan))

        left_child = _safe_int(rr.get("left_child_id", np.nan))
        right_child = _safe_int(rr.get("right_child_id", np.nan))

        left_members = rr.get("left_members", "")
        right_members = rr.get("right_members", "")
        singleton_graph_id = _normalize_gid_token(rr.get("singleton_graph_id", np.nan))

        if left_size is None or right_size is None:
            continue

        # smaller side = split-off child
        if left_size <= right_size:
            splitoff_child = left_child
            splitoff_size = left_size
            splitoff_raw_members = left_members

            main_child = right_child
            main_size = right_size
            main_raw_members = right_members
        else:
            splitoff_child = right_child
            splitoff_size = right_size
            splitoff_raw_members = right_members

            main_child = left_child
            main_size = left_size
            main_raw_members = left_members

        splitoff_gids = _parse_members_any(splitoff_raw_members)
        main_gids = _parse_members_any(main_raw_members)

        # critical fallback: if splitoff member string is missing, use singleton_graph_id
        if len(splitoff_gids) == 0 and singleton_graph_id is not None:
            splitoff_gids = [singleton_graph_id]

        splitoff_members_text = "|".join(splitoff_gids)
        main_members_text = "|".join(main_gids)

        splitoff_family_mix = _family_mix_from_members_text(splitoff_members_text)
        main_family_mix = _family_mix_from_members_text(main_members_text)

        # second fallback: if still empty/unknown but singleton is known, use direct lookup
        if (
            splitoff_family_mix is None
            or str(splitoff_family_mix).strip() == ""
            or str(splitoff_family_mix).startswith("?:")
        ) and singleton_graph_id is not None:
            fam = graph_to_family.get(singleton_graph_id, "?")
            splitoff_family_mix = f"{fam}:1"

        trigger_rule = (
            f"{pd.to_numeric(rr.get('mean_update_norm', np.nan), errors='coerce'):.3f}"
            f" < "
            f"{pd.to_numeric(rr.get('eps1', np.nan), errors='coerce'):.3f}"
            f" | "
            f"{pd.to_numeric(rr.get('max_update_norm', np.nan), errors='coerce'):.3f}"
            f" > "
            f"{pd.to_numeric(rr.get('eps2', np.nan), errors='coerce'):.3f}"
        )

        rows.append(
            {
                "round": _safe_int(rr.get("round", np.nan)),
                "parent_cluster_id": _safe_int(rr.get("cluster_id", np.nan)),
                "parent_cluster_size": _safe_int(rr.get("cluster_size", np.nan)),
                "trigger_rule": trigger_rule,
                "splitoff_child_id": splitoff_child,
                "splitoff_size": splitoff_size,
                "splitoff_family_mix": splitoff_family_mix,
                "main_child_id": main_child,
                "main_size": main_size,
                "main_family_mix": main_family_mix,
                "splitoff_members": splitoff_members_text,
                "main_members": main_members_text,
            }
        )

    return pd.DataFrame(rows)


def _compact_gcfl_story_df(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or df.empty:
        return pd.DataFrame()

    out = df.copy()

    out["trigger rule"] = out["trigger_rule"].apply(
        lambda x: _wrap_cell_text(x, width=24)
    )

    out["split-off cluster"] = out.apply(
        lambda r: (
            f"C{int(r['splitoff_child_id'])} ({int(r['splitoff_size'])} client)"
            if int(r["splitoff_size"]) == 1
            else f"C{int(r['splitoff_child_id'])} ({int(r['splitoff_size'])} clients)"
        ),
        axis=1,
    )

    out["continuing cluster"] = out.apply(
        lambda r: f"C{int(r['main_child_id'])} ({int(r['main_size'])} clients)",
        axis=1,
    )

    out["split-off family"] = out["splitoff_family_mix"].apply(
        lambda x: _wrap_cell_text(x, width=18)
    )
    out["continuing family mix"] = out["main_family_mix"].apply(
        lambda x: _wrap_cell_text(x, width=28)
    )

    keep_cols = [
        "round",
        "parent_cluster_id",
        "parent_cluster_size",
        "trigger rule",
        "split-off cluster",
        "split-off family",
        "continuing cluster",
        "continuing family mix",
    ]
    out = out[keep_cols].copy()

    out = out.rename(
        columns={
            "round": "round",
            "parent_cluster_id": "parent cluster",
            "parent_cluster_size": "parent size",
        }
    )

    return out


def _draw_gcfl_story_lineage(ax, story_df: pd.DataFrame, seed: int):
    ax.axis("off")

    if story_df is None or story_df.empty:
        ax.text(
            0.5, 0.5, "No applied split events", ha="center", va="center", fontsize=12
        )
        return

    chain_ids = [int(story_df.iloc[0]["parent_cluster_id"])]
    chain_sizes = [int(story_df.iloc[0]["parent_cluster_size"])]

    for _, rr in story_df.iterrows():
        chain_ids.append(int(rr["main_child_id"]))
        chain_sizes.append(int(rr["main_size"]))

    xs = np.linspace(0.08, 0.92, len(chain_ids))
    y_chain = 0.76

    # main continuing chain
    for i, (cid, csize) in enumerate(zip(chain_ids, chain_sizes)):
        ax.text(
            xs[i],
            y_chain,
            f"Cluster {cid}\n{csize} clients",
            ha="center",
            va="center",
            fontsize=11,
            bbox=dict(boxstyle="round,pad=0.35", fc="#eef3fb", ec="#4c78a8", lw=1.2),
        )
        if i < len(chain_ids) - 1:
            ax.annotate(
                "",
                xy=(xs[i + 1] - 0.05, y_chain),
                xytext=(xs[i] + 0.05, y_chain),
                arrowprops=dict(arrowstyle="->", lw=1.4),
            )

    # split-off story
    for i, (_, rr) in enumerate(story_df.iterrows()):
        xm = (xs[i] + xs[i + 1]) / 2.0

        ax.text(
            xm,
            0.89,
            f"round {int(rr['round'])}",
            ha="center",
            va="center",
            fontsize=10,
        )

        splitoff_family = rr["splitoff_family_mix"]
        if splitoff_family is None or str(splitoff_family).strip() == "":
            splitoff_family = "family unknown"

        ax.text(
            xm,
            0.52,
            (
                f"split off -> Cluster {int(rr['splitoff_child_id'])}\n"
                f"{int(rr['splitoff_size'])} client\n"
                f"{splitoff_family}"
            ),
            ha="center",
            va="center",
            fontsize=9.5,
            bbox=dict(boxstyle="round,pad=0.35", fc="#fff3e6", ec="#f58518", lw=1.2),
        )

        ax.annotate(
            "",
            xy=(xm, 0.60),
            xytext=(xm, y_chain - 0.07),
            arrowprops=dict(arrowstyle="->", lw=1.1, linestyle="--"),
        )

    ax.text(
        0.5,
        0.97,
        f"Split lineage story | seed={seed}",
        ha="center",
        va="top",
        fontsize=13,
    )

## 4. Plot panels

In [43]:
def plot_overview_loss_panel(ax, *, local_items, method_item_map, graph_ids):
    ok = False

    local_agg = get_weighted_group_scalar_stats(
        items=local_items,
        phase="val_epoch",
        split="val",
        metric_col="eval_loss",
        graph_ids=graph_ids,
        use_graph_filter=False,
    )
    ok |= plot_mean_std(ax, local_agg, label="Local centralized")

    for method in METHOD_ORDER:
        method_items = method_item_map.get(method, [])
        if len(method_items) == 0:
            continue

        post_phase = POST_SCALAR_PHASE_BY_METHOD[method]
        method_agg = get_weighted_group_loss_for_method(
            method_items=method_items,
            phase=post_phase,
            graph_ids=graph_ids,
        )
        ok |= plot_mean_std(ax, method_agg, label=METHOD_LABELS[method])

    ax.set_title(
        "Validation loss comparison "
        f"(all methods {BASELINE_ROUND_BUDGET} rounds; "
        f"APPLE mu={APPLE_PLOT_MU:g}, Lfrac={APPLE_PLOT_SCHEDULER_FRACTION:g})"
    )

    ax.set_xlabel("round")
    ax.set_ylabel("eval_loss")
    set_round_budget_axis(
        ax,
        max_round=APPLE_ROUND_BUDGET,
        show_baseline_budget_marker=True,
    )
    ax.grid(alpha=0.3)

    if ok:
        ax.legend(fontsize=8, ncol=3)
    else:
        annotate_no_data(ax)


def plot_overview_local_f1_panel(
    ax, *, local_items, graph_ids, title="Pooled local centralized positive F1"
):
    ok = False
    for task in TASKS:
        agg = get_group_pooled_task_f1_local(
            local_items=local_items,
            task=task,
            graph_ids=graph_ids,
        )
        ok |= plot_mean_std(ax, agg, label=task)

    ax.set_title(title)
    ax.set_xlabel("round")
    ax.set_ylabel("positive_f1")
    ax.set_ylim(0.0, 1.0)

    set_round_budget_axis(
        ax,
        max_round=APPLE_ROUND_BUDGET,
        show_baseline_budget_marker=True,
    )

    ax.grid(alpha=0.3)
    if ok:
        ax.legend(fontsize=8, ncol=2)
    else:
        annotate_no_data(ax)


def plot_overview_method_global_f1_panel(ax, *, method_items, graph_ids, method_name):
    ok = False

    post_task_phase = POST_TASK_PHASE_BY_METHOD[method_name]

    for task in TASKS:
        agg = get_group_pooled_task_f1_postagg(
            method_items=method_items,
            task=task,
            graph_ids=graph_ids,
            phase=post_task_phase,
            split="val",
        )
        ok |= plot_mean_std(ax, agg, label=task)

    if method_name == "apple":
        ax.set_title(
            f"Pooled APPLE positive F1 "
            f"(after-local selected model, equal {APPLE_ROUND_BUDGET}-round budget)"
        )
    else:
        ax.set_title(f"Pooled {METHOD_LABELS[method_name]} positive F1")
    ax.set_xlabel("round")
    ax.set_ylabel("positive_f1")
    ax.set_ylim(0.0, 1.0)

    if method_name == "apple":
        set_round_budget_axis(
            ax,
            max_round=APPLE_ROUND_BUDGET,
            show_baseline_budget_marker=True,
        )
    else:
        set_round_budget_axis(
            ax,
            max_round=APPLE_ROUND_BUDGET,
            show_baseline_budget_marker=True,
        )

    ax.grid(alpha=0.3)

    if ok:
        ax.legend(fontsize=8, ncol=2)
    else:
        annotate_no_data(ax)


def _plot_micro_macro_pack(
    ax,
    pack,
    title,
    ylabel="F1",
    max_round=None,
    show_baseline_budget_marker=False,
):
    ok = False

    if pack is not None and "micro_f1" in pack:
        ok |= plot_mean_std(ax, pack["micro_f1"], label="Micro-F1")
    if pack is not None and "macro_f1" in pack:
        ok |= plot_mean_std(ax, pack["macro_f1"], label="Macro Pos-F1")

    ax.set_title(title)
    ax.set_xlabel("round")
    ax.set_ylabel(ylabel)
    ax.set_ylim(0.0, 1.0)
    if max_round is not None:
        set_round_budget_axis(
            ax,
            max_round=max_round,
            show_baseline_budget_marker=show_baseline_budget_marker,
        )
    ax.grid(alpha=0.3)

    if ok:
        ax.legend(fontsize=8)
    else:
        annotate_no_data(ax)


def plot_method_loss_panel(ax, *, local_items, method_items, graph_ids, method_name):
    ok = False

    local_agg = get_weighted_group_scalar_stats(
        items=local_items,
        phase="val_epoch",
        split="val",
        metric_col="eval_loss",
        graph_ids=graph_ids,
        use_graph_filter=False,
    )

    post_phase = (
        "cluster_val_client" if method_name == "gcfl_plus" else "global_val_client"
    )
    post_agg = get_weighted_group_loss_for_method(
        method_items=method_items,
        phase=post_phase,
        graph_ids=graph_ids,
    )

    ok |= plot_mean_std(ax, local_agg, label="Local centralized")
    ok |= plot_mean_std(ax, post_agg, label=METHOD_LABELS[method_name])

    ax.set_title("Validation loss comparison")
    ax.set_xlabel("round")
    ax.set_ylabel("eval_loss")
    ax.grid(alpha=0.3)
    if ok:
        ax.legend(fontsize=8)
    else:
        annotate_no_data(ax)


def plot_method_micro_macro_panel(
    ax, *, local_items, method_items, graph_ids, source, method_name
):
    if source == "local":
        pack = get_group_pooled_micro_macro_local(
            local_items=local_items,
            graph_ids=graph_ids,
            split="val",
        )
        title = "Pooled local centralized micro / macro F1"
    elif source == "preagg":
        pack = get_group_pooled_micro_macro_method(
            method_items=method_items,
            graph_ids=graph_ids,
            phase="val_epoch_task",
            split="val",
        )
        title = (
            f"Pooled {METHOD_LABELS[method_name]} micro / macro F1 (pre-aggregation)"
        )
    else:
        post_phase = (
            "cluster_val_client_task"
            if method_name == "gcfl_plus"
            else "global_val_client_task"
        )
        pack = get_group_pooled_micro_macro_method(
            method_items=method_items,
            graph_ids=graph_ids,
            phase=post_phase,
            split="val",
        )
        title = (
            f"Pooled {METHOD_LABELS[method_name]} micro / macro F1 (post-aggregation)"
        )

    _plot_micro_macro_pack(ax, pack, title, ylabel="F1")


def plot_method_delta_micro_macro_panel(
    ax, *, local_items, method_items, graph_ids, method_name
):
    pre_pack = get_group_delta_local_minus_method_micro_macro(
        local_items=local_items,
        method_items=method_items,
        graph_ids=graph_ids,
        method_phase="val_epoch_task",
        split="val",
    )

    post_phase = (
        "cluster_val_client_task"
        if method_name == "gcfl_plus"
        else "global_val_client_task"
    )
    post_pack = get_group_delta_local_minus_method_micro_macro(
        local_items=local_items,
        method_items=method_items,
        graph_ids=graph_ids,
        method_phase=post_phase,
        split="val",
    )

    ok = False
    ok |= plot_mean_std(ax, pre_pack["micro_f1"], label="Local - pre Micro-F1")
    ok |= plot_mean_std(ax, pre_pack["macro_f1"], label="Local - pre Macro Pos-F1")
    ok |= plot_mean_std(ax, post_pack["micro_f1"], label="Local - post Micro-F1")
    ok |= plot_mean_std(ax, post_pack["macro_f1"], label="Local - post Macro Pos-F1")

    ax.axhline(0.0, linestyle="--", linewidth=1.0, color="black")
    ax.set_title(f"Delta vs {METHOD_LABELS[method_name]} (local minus pre / post)")
    ax.set_xlabel("round")
    ax.set_ylabel("F1 delta")
    ax.grid(alpha=0.3)

    if ok:
        ax.legend(fontsize=8, ncol=2)
    else:
        annotate_no_data(ax)


def _gcfl_cluster_count_curve(item):
    hist = _gcfl_history_df(item.get("clusters_df", pd.DataFrame()))
    if hist.empty:
        return pd.DataFrame(columns=["step", "cluster_count"])

    out = (
        hist.groupby("round")["cluster_id"]
        .nunique()
        .reset_index(name="cluster_count")
        .sort_values("round")
    )
    out["step"] = pd.to_numeric(out["round"], errors="coerce")
    out = out[pd.notna(out["step"])].copy()
    out["step"] = out["step"].astype(int)

    return out[["step", "cluster_count"]]


def plot_gcfl_cluster_evolution_panel(ax, *, gcfl_items):
    seed_curves = []
    applied_rounds = []

    for item in gcfl_items:
        curve = _gcfl_cluster_count_curve(item)
        if not curve.empty:
            seed_curves.append(curve)

        hist = _gcfl_history_df(item.get("clusters_df", pd.DataFrame()))
        if not hist.empty and "split_applied" in hist.columns:
            rr = (
                hist.loc[
                    pd.to_numeric(hist["split_applied"], errors="coerce") == 1, "round"
                ]
                .dropna()
                .astype(int)
                .tolist()
            )
            applied_rounds.extend(rr)

    agg = aggregate_seed_curves(seed_curves, "cluster_count")
    ok = plot_mean_std(ax, agg, label="active clusters")

    for rr in sorted(pd.unique(pd.Series(applied_rounds).dropna()).tolist()):
        ax.axvline(rr, linestyle=":", linewidth=1.0, alpha=0.18)

    ax.set_title("GCFL+ clustering evolution")
    ax.set_xlabel("round")
    ax.set_ylabel("# active clusters")
    ax.grid(alpha=0.3)

    if ok:
        ax.legend(fontsize=8)
    else:
        annotate_no_data(ax)


def get_group_pooled_task_f1_gcfl_postagg(
    *,
    method_items,
    task,
    graph_ids,
    split="val",
):
    target_graph_ids = [str(g) for g in graph_ids]
    curves = []

    for item in method_items:
        count_curves = []
        df = item["df"]

        for gid in target_graph_ids:
            count_curve = get_task_count_curve(
                df,
                phase="cluster_val_client_task",
                split=split,
                task=task,
                graph_id=gid,
            )
            count_curves.append(count_curve)

        pooled = _pool_count_curves(count_curves)
        if pooled.empty:
            continue

        tp = pooled["tp"].to_numpy(dtype=float)
        fp = pooled["fp"].to_numpy(dtype=float)
        fn = pooled["fn"].to_numpy(dtype=float)
        pooled["pooled_f1"] = binary_f1_from_counts(tp, fp, fn)
        curves.append(pooled[["step", "pooled_f1"]])

    return aggregate_seed_curves(curves, "pooled_f1")

In [44]:
def plot_method_loss_panel(ax, *, local_items, method_items, graph_ids, method_name):
    ok = False

    local_agg = get_weighted_group_scalar_stats(
        items=local_items,
        phase="val_epoch",
        split="val",
        metric_col="eval_loss",
        graph_ids=graph_ids,
        use_graph_filter=False,
    )
    post_agg = get_weighted_group_loss_for_method(
        method_items=method_items,
        phase="global_val_client",
        graph_ids=graph_ids,
    )

    ok |= plot_mean_std(ax, local_agg, label="pooled standalone val")
    ok |= plot_mean_std(
        ax, post_agg, label=f"{METHOD_LABELS[method_name]} post-aggregation"
    )

    ax.set_title("validation loss comparison")
    ax.set_xlabel("round")
    ax.set_ylabel("eval_loss")
    ax.grid(alpha=0.3)
    if ok:
        ax.legend(fontsize=8)
    else:
        annotate_no_data(ax)


def plot_method_f1_panel(
    ax, *, local_items, method_items, graph_ids, source, method_name
):
    ok = False

    for task in TASKS:
        if source == "local":
            agg = get_group_pooled_task_f1_local(
                local_items=local_items,
                task=task,
                graph_ids=graph_ids,
            )
            title = "pooled standalone positive F1"
        else:
            agg = get_group_pooled_task_f1_postagg(
                method_items=method_items,
                task=task,
                graph_ids=graph_ids,
            )
            title = (
                f"{METHOD_LABELS[method_name]} pooled positive F1 (post-aggregation)"
            )

        ok |= plot_mean_std(ax, agg, label=task)

    ax.set_title(title)
    ax.set_xlabel("round")
    ax.set_ylabel("positive_f1")
    ax.set_ylim(0.0, 1.0)
    ax.grid(alpha=0.3)
    if ok:
        ax.legend(fontsize=8, ncol=2)
    else:
        annotate_no_data(ax)


def plot_method_delta_panel(ax, *, local_items, method_items, graph_ids, method_name):
    ok = False
    for task in TASKS:
        agg = get_pooled_delta_local_minus_method(
            local_items=local_items,
            method_items=method_items,
            task=task,
            graph_ids=graph_ids,
        )
        ok |= plot_mean_std(ax, agg, label=task)

    ax.axhline(0.0, linestyle="--", linewidth=1, color="black")
    ax.set_title(f"delta = pooled standalone - {METHOD_LABELS[method_name]} global")
    ax.set_xlabel("round")
    ax.set_ylabel("positive_f1 delta")
    ax.grid(alpha=0.3)
    if ok:
        ax.legend(fontsize=8, ncol=2)
    else:
        annotate_no_data(ax)


def get_pooled_delta_local_minus_gcfl(
    *,
    local_items,
    method_items,
    task,
    graph_ids,
    split="val",
):
    target_graph_ids = {str(g) for g in graph_ids}
    method_by_seed = {int(item["seed"]): item for item in method_items}
    delta_curves = []

    seeds = sorted({int(item["seed"]) for item in local_items})
    for seed in seeds:
        if seed not in method_by_seed:
            continue

        local_count_curves = []
        seed_local_items = [
            item
            for item in local_items
            if int(item["seed"]) == seed and str(item["graph_id"]) in target_graph_ids
        ]
        for item in seed_local_items:
            count_curve = get_task_count_curve(
                item["df"],
                phase="val_epoch_task",
                split=split,
                task=task,
                graph_id=None,
            )
            local_count_curves.append(count_curve)

        pooled_local = _pool_count_curves(local_count_curves)
        if pooled_local.empty:
            continue
        pooled_local["pooled_f1"] = binary_f1_from_counts(
            pooled_local["tp"].to_numpy(dtype=float),
            pooled_local["fp"].to_numpy(dtype=float),
            pooled_local["fn"].to_numpy(dtype=float),
        )

        method_df = method_by_seed[seed]["df"]
        method_count_curves = []
        for gid in target_graph_ids:
            count_curve = get_task_count_curve(
                method_df,
                phase="cluster_val_client_task",
                split=split,
                task=task,
                graph_id=gid,
            )
            method_count_curves.append(count_curve)

        pooled_method = _pool_count_curves(method_count_curves)
        if pooled_method.empty:
            continue
        pooled_method["pooled_f1"] = binary_f1_from_counts(
            pooled_method["tp"].to_numpy(dtype=float),
            pooled_method["fp"].to_numpy(dtype=float),
            pooled_method["fn"].to_numpy(dtype=float),
        )

        merged = pd.merge(
            pooled_local[["step", "pooled_f1"]],
            pooled_method[["step", "pooled_f1"]],
            on="step",
            how="inner",
            suffixes=("_local", "_method"),
        )
        if merged.empty:
            continue

        merged["delta"] = merged["pooled_f1_local"] - merged["pooled_f1_method"]
        delta_curves.append(merged[["step", "delta"]])

    return aggregate_seed_curves(delta_curves, "delta")


def plot_gcfl_loss_panel(ax, *, local_items, method_items, graph_ids):
    ok = False

    local_agg = get_weighted_group_scalar_stats(
        items=local_items,
        phase="val_epoch",
        split="val",
        metric_col="eval_loss",
        graph_ids=graph_ids,
        use_graph_filter=False,
    )
    gcfl_agg = get_weighted_group_loss_for_method(
        method_items=method_items,
        phase="cluster_val_client",
        graph_ids=graph_ids,
    )

    ok |= plot_mean_std(ax, local_agg, label="pooled standalone val")
    ok |= plot_mean_std(ax, gcfl_agg, label="GCFL+ post-aggregation")

    ax.set_title("validation loss comparison")
    ax.set_xlabel("round")
    ax.set_ylabel("eval_loss")
    ax.grid(alpha=0.3)
    if ok:
        ax.legend(fontsize=8)
    else:
        annotate_no_data(ax)


def plot_gcfl_f1_panel(ax, *, local_items, method_items, graph_ids, source):
    ok = False

    for task in TASKS:
        if source == "local":
            agg = get_group_pooled_task_f1_local(
                local_items=local_items,
                task=task,
                graph_ids=graph_ids,
            )
            title = "pooled standalone positive F1"
        else:
            agg = get_group_pooled_task_f1_gcfl_postagg(
                method_items=method_items,
                task=task,
                graph_ids=graph_ids,
            )
            title = "GCFL+ pooled positive F1 (post-aggregation)"

        ok |= plot_mean_std(ax, agg, label=task)

    ax.set_title(title)
    ax.set_xlabel("round")
    ax.set_ylabel("positive_f1")
    ax.set_ylim(0.0, 1.0)
    ax.grid(alpha=0.3)
    if ok:
        ax.legend(fontsize=8, ncol=2)
    else:
        annotate_no_data(ax)


def plot_gcfl_delta_panel(ax, *, local_items, method_items, graph_ids):
    ok = False
    for task in TASKS:
        agg = get_pooled_delta_local_minus_gcfl(
            local_items=local_items,
            method_items=method_items,
            task=task,
            graph_ids=graph_ids,
        )
        ok |= plot_mean_std(ax, agg, label=task)

    ax.axhline(0.0, linestyle="--", linewidth=1, color="black")
    ax.set_title("delta = pooled standalone - GCFL+ clustered")
    ax.set_xlabel("round")
    ax.set_ylabel("positive_f1 delta")
    ax.grid(alpha=0.3)
    if ok:
        ax.legend(fontsize=8, ncol=2)
    else:
        annotate_no_data(ax)

## 5. Table helpers


In [45]:
BEST_SCALAR_METRICS = [
    ("Micro-F1", "micro_f1", True),
    ("Macro Pos-F1", "macro_pos_f1", True),
]

SUMMARY_TABLE_COLUMNS = (
    ["Algorithm"]
    + [label for label, _, _ in BEST_SCALAR_METRICS]
    + [f"{task} pos-F1" for task in TASKS]
)


def _best_phase_name(source_kind: str, split: str, task: str | None = None) -> str:
    source_kind = str(source_kind)

    if source_kind not in BEST_PHASE_PREFIX_BY_METHOD:
        raise ValueError(f"Unknown source_kind={source_kind}")

    base = f"{BEST_PHASE_PREFIX_BY_METHOD[source_kind]}_{split}"

    if task is not None:
        base += "_task"

    return base


def _select_best_rows(df, *, source_kind, split, graph_ids=None, task=None):
    phase = _best_phase_name(source_kind, split, task=task)
    part = df[(df["phase"] == phase) & (df["split"] == split)].copy()

    if graph_ids is not None and "graph_id" in part.columns:
        target = {str(g) for g in graph_ids}
        part = part[part["graph_id"].astype(str).isin(target)].copy()

    if task is None:
        if "task" in part.columns:
            part = part[part["task"].isna()].copy()
    else:
        if "task" in part.columns:
            part = part[part["task"].astype(str) == str(task)].copy()
        else:
            part = part.iloc[0:0].copy()

    return part


def _pool_scalar_rows(part, metric_specs):
    if part is None or part.empty:
        return None

    weight_col = _detect_weight_col(part)
    part = part.copy()
    part[weight_col] = pd.to_numeric(part[weight_col], errors="coerce")
    part = part[pd.notna(part[weight_col]) & (part[weight_col] > 0)].copy()
    if part.empty:
        return None

    out = {
        "num_nodes": float(part[weight_col].sum()),
    }

    weights = part[weight_col].to_numpy(dtype=float)

    for _, col, _ in metric_specs:
        if col not in part.columns:
            out[col] = np.nan
            continue

        vals = pd.to_numeric(part[col], errors="coerce").to_numpy(dtype=float)
        mask = np.isfinite(vals) & np.isfinite(weights) & (weights > 0)

        if not np.any(mask):
            out[col] = np.nan
        else:
            out[col] = float(np.average(vals[mask], weights=weights[mask]))

    return out


def _pool_task_rows(part):
    if part is None or part.empty:
        return None

    needed = ["tp", "fp", "fn"]
    if not all(c in part.columns for c in needed):
        return None

    part = part.copy()
    for c in needed:
        part[c] = pd.to_numeric(part[c], errors="coerce")

    part = part.dropna(subset=needed).copy()
    if part.empty:
        return None

    tp = float(part["tp"].sum())
    fp = float(part["fp"].sum())
    fn = float(part["fn"].sum())

    denom = 2.0 * tp + fp + fn
    pos_f1 = np.nan if denom <= 0 else (2.0 * tp) / denom

    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "positive_f1": pos_f1,
    }


def collect_best_scalar_by_seed_local(local_items, graph_ids, split="test"):
    target = {str(g) for g in graph_ids}
    by_seed = {}

    for item in local_items:
        graph_id = str(item["graph_id"])
        if graph_id not in target:
            continue

        part = _select_best_rows(
            item["df"],
            source_kind="local",
            split=split,
            graph_ids=[graph_id],
            task=None,
        )
        if part.empty:
            continue

        by_seed.setdefault(int(item["seed"]), []).append(part)

    rows = []
    for seed, frames in sorted(by_seed.items()):
        pooled = _pool_scalar_rows(
            pd.concat(frames, axis=0, ignore_index=True),
            BEST_SCALAR_METRICS,
        )
        if pooled is None:
            continue
        pooled["seed"] = int(seed)
        rows.append(pooled)

    return pd.DataFrame(rows)


def collect_best_scalar_by_seed_method(method_items, graph_ids, split="test"):
    rows = []

    for item in method_items:
        method_name = str(item["method"])

        part = _select_best_rows(
            item["df"],
            source_kind=method_name,
            split=split,
            graph_ids=graph_ids,
            task=None,
        )

        pooled = _pool_scalar_rows(part, BEST_SCALAR_METRICS)
        if pooled is None:
            continue

        pooled["seed"] = int(item["seed"])
        rows.append(pooled)

    return pd.DataFrame(rows)


def collect_best_task_f1_by_seed_local(local_items, graph_ids, task, split="test"):
    target = {str(g) for g in graph_ids}
    by_seed = {}

    for item in local_items:
        graph_id = str(item["graph_id"])
        if graph_id not in target:
            continue

        part = _select_best_rows(
            item["df"],
            source_kind="local",
            split=split,
            graph_ids=[graph_id],
            task=task,
        )
        if part.empty:
            continue

        by_seed.setdefault(int(item["seed"]), []).append(part)

    rows = []
    for seed, frames in sorted(by_seed.items()):
        pooled = _pool_task_rows(pd.concat(frames, axis=0, ignore_index=True))
        if pooled is None:
            continue
        pooled["seed"] = int(seed)
        rows.append(pooled)

    return pd.DataFrame(rows)


def collect_best_task_f1_by_seed_method(method_items, graph_ids, task, split="test"):
    rows = []

    for item in method_items:
        method_name = str(item["method"])

        part = _select_best_rows(
            item["df"],
            source_kind=method_name,
            split=split,
            graph_ids=graph_ids,
            task=task,
        )

        pooled = _pool_task_rows(part)
        if pooled is None:
            continue

        pooled["seed"] = int(item["seed"])
        rows.append(pooled)

    return pd.DataFrame(rows)


def summarize_seed_metric(seed_df, value_col):
    if seed_df is None or seed_df.empty or value_col not in seed_df.columns:
        return {"mean": np.nan, "std": np.nan, "count": 0}

    vals = (
        pd.to_numeric(seed_df[value_col], errors="coerce")
        .dropna()
        .to_numpy(dtype=float)
    )
    if len(vals) == 0:
        return {"mean": np.nan, "std": np.nan, "count": 0}

    return {
        "mean": float(np.mean(vals)),
        "std": float(np.std(vals, ddof=0)),
        "count": int(len(vals)),
    }


def format_metric_summary(stats, *, as_percent=True, digits=1):
    if stats["count"] == 0 or pd.isna(stats["mean"]):
        return "-"

    scale = 100.0 if as_percent else 1.0
    suffix = "%" if as_percent else ""
    return f"{stats['mean'] * scale:.{digits}f}{suffix} ± {stats['std'] * scale:.{digits}f}{suffix}"


def build_overview_algo_table(local_items, method_item_map, graph_ids, split="test"):
    row_map = {"Local centralized": {}}

    for method in METHOD_ORDER:
        if method in method_item_map:
            row_map[METHOD_LABELS[method]] = {}

    scalar_local = collect_best_scalar_by_seed_local(
        local_items, graph_ids, split=split
    )
    scalar_methods = {
        method: collect_best_scalar_by_seed_method(items, graph_ids, split=split)
        for method, items in method_item_map.items()
    }

    for label, col, as_percent in BEST_SCALAR_METRICS:
        row_map["Local centralized"][label] = format_metric_summary(
            summarize_seed_metric(scalar_local, col),
            as_percent=as_percent,
        )

        for method in METHOD_ORDER:
            if method in scalar_methods:
                row_map[METHOD_LABELS[method]][label] = format_metric_summary(
                    summarize_seed_metric(scalar_methods[method], col),
                    as_percent=as_percent,
                )

    for task in TASKS:
        metric_name = f"{task} pos-F1"

        task_local = collect_best_task_f1_by_seed_local(
            local_items,
            graph_ids,
            task=task,
            split=split,
        )
        row_map["Local centralized"][metric_name] = format_metric_summary(
            summarize_seed_metric(task_local, "positive_f1"),
            as_percent=True,
        )

        for method in METHOD_ORDER:
            if method in method_item_map:
                task_method = collect_best_task_f1_by_seed_method(
                    method_item_map[method],
                    graph_ids,
                    task=task,
                    split=split,
                )
                row_map[METHOD_LABELS[method]][metric_name] = format_metric_summary(
                    summarize_seed_metric(task_method, "positive_f1"),
                    as_percent=True,
                )

    ordered_algos = ["Local centralized"] + [
        METHOD_LABELS[m] for m in METHOD_ORDER if m in method_item_map
    ]

    rows = []
    for algo_name in ordered_algos:
        vals = row_map[algo_name]
        row = {"Algorithm": algo_name}
        for c in SUMMARY_TABLE_COLUMNS[1:]:
            row[c] = vals.get(c, "-")
        rows.append(row)

    return pd.DataFrame(rows)[SUMMARY_TABLE_COLUMNS]


def build_method_algo_table(local_items, method_items, graph_ids, method, split="test"):
    method_label = METHOD_LABELS[method]

    row_map = {
        "Local centralized": {},
        method_label: {},
    }

    scalar_local = collect_best_scalar_by_seed_local(
        local_items, graph_ids, split=split
    )
    scalar_method = collect_best_scalar_by_seed_method(
        method_items, graph_ids, split=split
    )

    for label, col, as_percent in BEST_SCALAR_METRICS:
        row_map["Local centralized"][label] = format_metric_summary(
            summarize_seed_metric(scalar_local, col),
            as_percent=as_percent,
        )
        row_map[method_label][label] = format_metric_summary(
            summarize_seed_metric(scalar_method, col),
            as_percent=as_percent,
        )

    for task in TASKS:
        metric_name = f"{task} pos-F1"

        task_local = collect_best_task_f1_by_seed_local(
            local_items,
            graph_ids,
            task=task,
            split=split,
        )
        task_method = collect_best_task_f1_by_seed_method(
            method_items,
            graph_ids,
            task=task,
            split=split,
        )

        row_map["Local centralized"][metric_name] = format_metric_summary(
            summarize_seed_metric(task_local, "positive_f1"),
            as_percent=True,
        )
        row_map[method_label][metric_name] = format_metric_summary(
            summarize_seed_metric(task_method, "positive_f1"),
            as_percent=True,
        )

    rows = []
    for algo_name, vals in row_map.items():
        row = {"Algorithm": algo_name}
        for c in SUMMARY_TABLE_COLUMNS[1:]:
            row[c] = vals.get(c, "-")
        rows.append(row)

    return pd.DataFrame(rows)[SUMMARY_TABLE_COLUMNS]


def draw_algo_table(ax, table_df, title, fontsize=7.6):
    ax.axis("off")

    if table_df is None or table_df.empty:
        annotate_no_data(ax, text="No summary table available")
        ax.set_title(title, fontsize=11, pad=1)
        return

    ncols = len(table_df.columns)
    nrows = len(table_df)

    algo_width = 0.155
    metric_width = (1.0 - algo_width) / max(ncols - 1, 1)
    col_widths = [algo_width] + [metric_width] * (ncols - 1)

    tbl = ax.table(
        cellText=table_df.values,
        colLabels=table_df.columns,
        cellLoc="center",
        colLoc="center",
        colWidths=col_widths,
        bbox=[0.005, 0.04, 0.99, 0.90],
    )

    tbl.auto_set_font_size(False)
    tbl.set_fontsize(fontsize)

    for (r, c), cell in tbl.get_celld().items():
        cell.set_linewidth(0.40)
        if r == 0:
            cell.set_text_props(weight="bold")
            cell.set_facecolor("#EAEAF2")
        elif c == 0:
            cell.set_text_props(weight="bold")
            cell.get_text().set_color("black")
        else:
            cell.get_text().set_color("#444444")

    header_h = 0.105
    body_h = min(0.145, 0.78 / max(nrows, 1))

    for c in range(ncols):
        tbl[(0, c)].set_height(header_h)

    for r in range(1, nrows + 1):
        for c in range(ncols):
            tbl[(r, c)].set_height(body_h)

    for c_idx, col in enumerate(table_df.columns[1:], start=1):
        vals = table_df[col].apply(_summary_to_mean).to_numpy(dtype=float)
        if np.all(np.isnan(vals)):
            continue

        best = np.nanmax(vals)
        for r_idx, val in enumerate(vals, start=1):
            if np.isfinite(val) and np.isclose(val, best):
                txt = tbl[(r_idx, c_idx)].get_text()
                txt.set_weight("bold")
                txt.set_color("black")

    ax.set_title(title, fontsize=11, pad=1)


def _graph_to_family_map_from_row(row):
    out = {}
    fam_map = row.get("family_to_graph_ids", {})
    for fam, gids in fam_map.items():
        for gid in gids:
            out[str(gid)] = str(fam)
    return out


def _members_to_family_mix(member_str, graph_to_family):
    FAMILY_ABBREV = {
        "strong_decreasing": "SD",
        "mild_decreasing": "MD",
        "flat_balanced": "FB",
        "mild_increasing": "MI",
        "strong_increasing": "SI",
    }

    gids = _split_pipe_values(member_str)
    if not gids:
        return "-"

    fams = []
    for g in gids:
        fam = graph_to_family.get(str(g), f"unknown:{g}")
        fams.append(FAMILY_ABBREV.get(fam, fam))

    cnt = pd.Series(fams).value_counts().to_dict()
    order = {"SD": 0, "MD": 1, "FB": 2, "MI": 3, "SI": 4}

    parts = sorted(cnt.items(), key=lambda kv: order.get(kv[0], 99))
    return " | ".join(f"{fam}:{n}" for fam, n in parts)


def draw_generic_table(ax, df, title, fontsize=8.0, max_rows=12, wrap_widths=None):
    import textwrap

    ax.axis("off")

    if df is None or df.empty:
        annotate_no_data(ax, text="No data available")
        ax.set_title(title, fontsize=11, pad=2)
        return

    if wrap_widths is None:
        wrap_widths = {}

    show_df = df.head(max_rows).copy()

    def _wrap_text(val, width):
        if pd.isna(val):
            s = "-"
        else:
            s = str(val)
        if width is None:
            return s
        wrapped = textwrap.wrap(
            s,
            width=width,
            break_long_words=False,
            break_on_hyphens=False,
        )
        return "\n".join(wrapped) if wrapped else s

    for col, width in wrap_widths.items():
        if col in show_df.columns:
            show_df[col] = show_df[col].apply(lambda x: _wrap_text(x, width))

    tbl = ax.table(
        cellText=show_df.values,
        colLabels=show_df.columns,
        cellLoc="center",
        colLoc="center",
        bbox=[0.01, 0.02, 0.98, 0.93],
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(fontsize)

    ncols = len(show_df.columns)

    for (r, c), cell in tbl.get_celld().items():
        cell.set_linewidth(0.45)
        if r == 0:
            cell.set_text_props(weight="bold")
            cell.set_facecolor("#EAEAF2")

    for r in range(1, len(show_df) + 1):
        max_lines = 1
        for c in range(ncols):
            txt = str(show_df.iloc[r - 1, c])
            max_lines = max(max_lines, txt.count("\n") + 1)

        row_height = min(0.05 * max_lines, 0.24)
        for c in range(ncols):
            tbl[(r, c)].set_height(row_height)

    for c in range(ncols):
        tbl[(0, c)].set_height(0.075)

    for c_idx, col in enumerate(show_df.columns):
        if col in {
            "Members",
            "All mix",
            "Left mix",
            "Right mix",
            "Family mix",
            "Status counts",
        }:
            for r in range(1, len(show_df) + 1):
                tbl[(r, c_idx)].get_text().set_ha("left")

    ax.set_title(title, fontsize=11, pad=2)


def _gcfl_history_df(cdf):
    if cdf is None or cdf.empty:
        return pd.DataFrame()

    out = cdf.copy()

    numeric_cols = [
        "seed",
        "round",
        "cluster_id",
        "cluster_size",
        "full_participation",
        "born_round",
        "cluster_age",
        "mean_update_norm",
        "max_update_norm",
        "eps1",
        "eps2",
        "split_triggered",
        "split_applied",
        "dtw_dist_mean",
        "dtw_dist_max",
    ]
    for col in numeric_cols:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")

    if "split_status" in out.columns:
        out["split_status"] = out["split_status"].fillna("NA").astype(str)

    return out.sort_values(["cluster_id", "round"]).reset_index(drop=True)


def _gcfl_round_list_text(values):
    vals = (
        pd.to_numeric(pd.Series(values), errors="coerce").dropna().astype(int).tolist()
    )
    vals = sorted(pd.unique(vals).tolist())
    return ", ".join(map(str, vals)) if vals else "-"


def _format_status_counts_for_cell(status_counts):
    if not status_counts:
        return "-"

    short = {
        "eligible": "elig",
        "warmup": "warm",
        "insufficient_history": "hist",
        "not_full_participation": "full=0",
        "too_small": "small",
        "not_old_enough": "age",
    }

    parts = []
    for k, v in status_counts.items():
        kk = short.get(str(k), str(k))
        parts.append(f"{kk}:{v}")

    # one item per line is safest for narrow summary cells
    return "\n".join(parts)


def _gcfl_summary_df_from_history(history_df, item):
    if history_df is None or history_df.empty:
        return pd.DataFrame()

    triggered_text = _gcfl_round_list_text(
        history_df.loc[history_df["split_triggered"] == 1, "round"]
    )
    applied_text = _gcfl_round_list_text(
        history_df.loc[history_df["split_applied"] == 1, "round"]
    )

    status_counts = history_df["split_status"].fillna("NA").value_counts().to_dict()
    status_text = _format_status_counts_for_cell(status_counts)

    active_clusters = (
        sorted(history_df["cluster_id"].dropna().astype(int).unique().tolist())
        if "cluster_id" in history_df.columns
        else []
    )

    last_round = (
        int(history_df["round"].max()) if "round" in history_df.columns else None
    )
    final_cluster_count = (
        int(history_df.loc[history_df["round"] == last_round, "cluster_id"].nunique())
        if last_round is not None
        else 0
    )

    return pd.DataFrame(
        [
            {
                "Seed": int(item["seed"]),
                "eps1 q": item.get("eps1_quantile", np.nan),
                "eps2 q": item.get("eps2_quantile", np.nan),
                "Warmup": item.get("warmup_rounds", np.nan),
                "Hist len": item.get("grad_seq_len", np.nan),
                "Min cluster": item.get("min_cluster_size", np.nan),
                "Min child": item.get("min_child_size", np.nan),
                "Triggered rounds": triggered_text,
                "Applied rounds": applied_text,
                "Final #clusters": final_cluster_count,
                "Cluster IDs seen": (
                    ", ".join(map(str, active_clusters)) if active_clusters else "-"
                ),
                "Status counts": status_text,
            }
        ]
    )


def _pipe_members_to_list(x):
    if pd.isna(x):
        return []
    s = str(x).strip()
    if s == "" or s == "-" or s.lower() == "nan":
        return []

    out = []
    for tok in s.split("|"):
        tok = tok.strip()
        if tok == "":
            continue
        try:
            out.append(int(float(tok)))
        except Exception:
            continue
    return out


def _family_mix_from_member_list(member_list, graph_to_family):
    if not member_list or not graph_to_family:
        return "-"

    order = [
        "strong_decreasing",
        "mild_decreasing",
        "flat_balanced",
        "mild_increasing",
        "strong_increasing",
    ]
    short = {
        "strong_decreasing": "SD",
        "mild_decreasing": "MD",
        "flat_balanced": "FB",
        "mild_increasing": "MI",
        "strong_increasing": "SI",
    }

    counts = {k: 0 for k in order}
    for gid in member_list:
        fam = graph_to_family.get(gid)
        if fam in counts:
            counts[fam] += 1

    nonzero = [(fam, counts[fam]) for fam in order if counts[fam] > 0]
    if not nonzero:
        return "-"

    parts = [f"{short[fam]}:{cnt}" for fam, cnt in nonzero]

    # wrap after about 3 buckets so it stays inside the cell
    lines = []
    chunk = []
    for p in parts:
        chunk.append(p)
        if len(chunk) == 3:
            lines.append(" | ".join(chunk))
            chunk = []
    if chunk:
        lines.append(" | ".join(chunk))

    return "\n".join(lines)


def _pipe_members_to_family_mix(x, graph_to_family):
    members = _pipe_members_to_list(x)
    return _family_mix_from_member_list(members, graph_to_family)


def _pipe_members_to_size(x):
    return len(_pipe_members_to_list(x))


def _gcfl_split_events_df_from_history(history_df, graph_to_family):
    """
    Split-events table = ONLY rows where an actual split event happened:
    triggered or applied.

    Child columns are shown as compact family-mix summaries instead of raw
    long member-id strings, so the table stays readable.
    """
    if history_df is None or history_df.empty:
        return pd.DataFrame()

    part = history_df.loc[
        (pd.to_numeric(history_df["split_triggered"], errors="coerce") == 1)
        | (pd.to_numeric(history_df["split_applied"], errors="coerce") == 1)
    ].copy()

    if part.empty:
        return pd.DataFrame(
            [
                {
                    "Info": (
                        "No split events: no round had split_triggered=1 or "
                        "split_applied=1."
                    )
                }
            ]
        )

    # derive compact child summaries before dropping raw columns
    if "left_members" in part.columns:
        part["left_size"] = part["left_members"].apply(_pipe_members_to_size)
        part["left_mix"] = part["left_members"].apply(
            lambda x: _pipe_members_to_family_mix(x, graph_to_family)
        )
    else:
        part["left_size"] = np.nan
        part["left_mix"] = "-"

    if "right_members" in part.columns:
        part["right_size"] = part["right_members"].apply(_pipe_members_to_size)
        part["right_mix"] = part["right_members"].apply(
            lambda x: _pipe_members_to_family_mix(x, graph_to_family)
        )
    else:
        part["right_size"] = np.nan
        part["right_mix"] = "-"

    keep_cols = [
        "round",
        "cluster_id",
        "cluster_size",
        "mean_update_norm",
        "eps1",
        "max_update_norm",
        "eps2",
        "split_status",
        "split_triggered",
        "split_applied",
        "dtw_dist_mean",
        "dtw_dist_max",
        "left_size",
        "left_mix",
        "right_size",
        "right_mix",
    ]
    keep_cols = [c for c in keep_cols if c in part.columns]
    part = part[keep_cols].copy()

    part = part.rename(
        columns={
            "round": "Round",
            "cluster_id": "Cluster",
            "cluster_size": "Size",
            "mean_update_norm": "MeanNorm",
            "eps1": "eps1",
            "max_update_norm": "MaxNorm",
            "eps2": "eps2",
            "split_status": "Status",
            "split_triggered": "Trig",
            "split_applied": "Applied",
            "dtw_dist_mean": "DTW mean",
            "dtw_dist_max": "DTW max",
            "left_size": "Left size",
            "left_mix": "Left mix",
            "right_size": "Right size",
            "right_mix": "Right mix",
        }
    )

    for col in ["MeanNorm", "eps1", "MaxNorm", "eps2", "DTW mean", "DTW max"]:
        if col in part.columns:
            part[col] = pd.to_numeric(part[col], errors="coerce").map(
                lambda x: "-" if pd.isna(x) else f"{float(x):.5f}"
            )

    for col in [
        "Round",
        "Cluster",
        "Size",
        "Trig",
        "Applied",
        "Left size",
        "Right size",
    ]:
        if col in part.columns:
            part[col] = pd.to_numeric(part[col], errors="coerce").map(
                lambda x: "-" if pd.isna(x) else str(int(x))
            )

    for col in ["Left mix", "Right mix"]:
        if col in part.columns:
            part[col] = part[col].fillna("-").astype(str)

    return part.reset_index(drop=True)


def _gcfl_finals_df_from_history(history_df, graph_to_family):
    if history_df is None or history_df.empty:
        return pd.DataFrame()

    final_round = int(pd.to_numeric(history_df["round"], errors="coerce").max())
    finals_df = history_df[
        pd.to_numeric(history_df["round"], errors="coerce") == final_round
    ].copy()

    if finals_df.empty:
        return pd.DataFrame()

    finals_df["family_mix"] = finals_df["members"].apply(
        lambda x: _members_to_family_mix(x, graph_to_family)
    )
    finals_df["members"] = finals_df["members"].apply(_members_to_multiline)

    finals_df = finals_df[
        ["cluster_id", "cluster_size", "members", "family_mix"]
    ].copy()

    finals_df = finals_df.rename(
        columns={
            "cluster_id": "Cluster",
            "cluster_size": "Size",
            "members": "Members",
            "family_mix": "Family mix",
        }
    )
    return finals_df.reset_index(drop=True)


def _plot_gcfl_metric_threshold_panel(
    ax,
    history_df,
    *,
    metric_col,
    threshold_col,
    metric_label,
    threshold_label,
    ylabel,
    title,
):
    hist = _gcfl_history_df(history_df)
    if hist.empty:
        annotate_no_data(ax)
        return

    cluster_ids = sorted(hist["cluster_id"].dropna().astype(int).unique().tolist())
    if len(cluster_ids) == 0:
        annotate_no_data(ax)
        return

    cmap = plt.cm.get_cmap("tab10", max(len(cluster_ids), 1))

    for i, cluster_id in enumerate(cluster_ids):
        color = cmap(i)
        sub = hist[hist["cluster_id"] == cluster_id].sort_values("round").copy()

        x = pd.to_numeric(sub["round"], errors="coerce").to_numpy(dtype=float)
        y = pd.to_numeric(sub[metric_col], errors="coerce").to_numpy(dtype=float)
        mask = np.isfinite(x) & np.isfinite(y)

        if np.any(mask):
            ax.plot(
                x[mask],
                y[mask],
                color=color,
                linewidth=2.0,
                label=f"C{cluster_id} {metric_label}",
            )

        tsub = sub.dropna(subset=[threshold_col]).copy()
        if not tsub.empty:
            ax.plot(
                tsub["round"].to_numpy(dtype=float),
                pd.to_numeric(tsub[threshold_col], errors="coerce").to_numpy(
                    dtype=float
                ),
                color=color,
                linestyle="--",
                linewidth=1.8,
                alpha=0.95,
                label=f"C{cluster_id} {threshold_label}",
            )

    triggered_rounds = sorted(
        hist.loc[hist["split_triggered"] == 1, "round"]
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )
    applied_rounds = sorted(
        hist.loc[hist["split_applied"] == 1, "round"]
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )

    for rr in triggered_rounds:
        ax.axvline(rr, color="red", linestyle=":", linewidth=1.0, alpha=0.18)
    for rr in applied_rounds:
        ax.axvline(rr, color="green", linestyle="-.", linewidth=1.0, alpha=0.18)

    ax.set_title(title)
    ax.set_xlabel("round")
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8, ncol=2)


def _gcfl_history_df(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or len(df) == 0:
        return pd.DataFrame()

    out = df.copy()
    if "round" in out.columns:
        out["round"] = pd.to_numeric(out["round"], errors="coerce")
    if "cluster_id" in out.columns:
        out["cluster_id"] = pd.to_numeric(out["cluster_id"], errors="coerce")
    for c in [
        "split_triggered",
        "split_applied",
        "left_size",
        "right_size",
        "cluster_size",
        "cut_value",
        "child_balance_ratio",
        "eps1",
        "eps2",
        "mean_update_norm",
        "max_update_norm",
    ]:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce")

    out = out.sort_values(
        [c for c in ["round", "cluster_id"] if c in out.columns]
    ).reset_index(drop=True)
    return out


def _gcfl_item_meta(item, key, default=np.nan):
    if key in item:
        return item[key]
    for carrier in [item.get("manifest_row"), item.get("row")]:
        if isinstance(carrier, pd.Series) and key in carrier.index:
            return carrier[key]
        if isinstance(carrier, dict) and key in carrier:
            return carrier[key]
    return default


def _gcfl_round_diag_df(item: dict) -> pd.DataFrame:
    df = item.get("df", pd.DataFrame()).copy()
    if df.empty or "phase" not in df.columns:
        return pd.DataFrame()

    part = df[df["phase"].astype(str) == "gcfl_round_diag"].copy()
    if part.empty:
        return pd.DataFrame()

    if "round" in part.columns:
        part["round"] = pd.to_numeric(part["round"], errors="coerce")
    for c in [
        "num_active_clusters",
        "num_split_triggered",
        "num_split_applied",
        "eval_loss",
        "micro_f1",
        "macro_pos_f1",
    ]:
        if c in part.columns:
            part[c] = pd.to_numeric(part[c], errors="coerce")

    return part.sort_values("round").reset_index(drop=True)


def _gcfl_triggered_rounds(hist: pd.DataFrame) -> list[int]:
    if hist.empty or "split_triggered" not in hist.columns:
        return []
    return sorted(
        hist.loc[pd.to_numeric(hist["split_triggered"], errors="coerce") == 1, "round"]
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )


def _gcfl_applied_rounds(hist: pd.DataFrame) -> list[int]:
    if hist.empty or "split_applied" not in hist.columns:
        return []
    return sorted(
        hist.loc[pd.to_numeric(hist["split_applied"], errors="coerce") == 1, "round"]
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )


def _gcfl_status_count_df(hist: pd.DataFrame, col: str) -> pd.DataFrame:
    if hist.empty or col not in hist.columns:
        return pd.DataFrame(columns=["label", "count"])

    out = (
        hist[col]
        .fillna("nan")
        .astype(str)
        .value_counts(dropna=False)
        .rename_axis("label")
        .reset_index(name="count")
        .sort_values(["count", "label"], ascending=[False, True])
        .reset_index(drop=True)
    )
    return out


def _graph_to_family_map_from_tables():
    mapping = {}

    frames = []
    for obj_name in ["selected_subset", "family_run_table", "global_run_table"]:
        if obj_name in globals():
            obj = globals()[obj_name]
            if isinstance(obj, pd.DataFrame) and not obj.empty:
                frames.append(obj)

    for df in frames:
        if "family" not in df.columns:
            continue

        for _, rr in df.iterrows():
            fam = str(rr["family"])
            gids = rr.get("graph_ids", None)

            if isinstance(gids, list):
                for gid in gids:
                    mapping[str(gid)] = fam
            elif isinstance(gids, str) and gids.strip() != "":
                for gid in gids.split("|"):
                    gid = gid.strip()
                    if gid != "":
                        mapping[str(gid)] = fam

    return mapping


def _normalize_gid_token(x):
    if x is None:
        return None
    if isinstance(x, float) and pd.isna(x):
        return None

    s = str(x).strip()
    if s == "" or s.lower() == "nan":
        return None

    # normalize 109.0 -> 109
    try:
        f = float(s)
        if np.isfinite(f) and float(int(f)) == f:
            return str(int(f))
    except Exception:
        pass

    return s


def _parse_members_any(text):
    if text is None:
        return []

    if isinstance(text, float) and pd.isna(text):
        return []

    s = str(text).strip()
    if s == "" or s.lower() == "nan":
        return []

    parts = [p.strip() for p in s.split("|") if p.strip() != ""]
    out = []
    for p in parts:
        q = _normalize_gid_token(p)
        if q is not None:
            out.append(q)

    # if it was just a scalar like "109.0" rather than pipe-separated
    if len(out) == 0:
        q = _normalize_gid_token(s)
        if q is not None:
            out = [q]

    return out


def _family_mix_from_members_text(members_text: str) -> str:
    graph_to_family = _graph_to_family_map_from_tables()
    gids = _parse_members_any(members_text)

    counts = {}
    for gid in gids:
        fam = graph_to_family.get(str(gid), "?")
        counts[fam] = counts.get(fam, 0) + 1

    if len(counts) == 0:
        return ""

    return " | ".join(f"{k}:{counts[k]}" for k in sorted(counts.keys()))


def _family_mix_from_members_text(members_text: str) -> str:
    graph_to_family = _graph_to_family_map_from_tables()
    gids = [x.strip() for x in str(members_text).split("|") if x.strip() != ""]
    counts = {}
    for gid in gids:
        fam = graph_to_family.get(str(gid), "?")
        counts[fam] = counts.get(fam, 0) + 1
    if len(counts) == 0:
        return ""
    return " | ".join(f"{k}:{counts[k]}" for k in sorted(counts.keys()))


def _gcfl_last_round_clusters_df(hist: pd.DataFrame) -> pd.DataFrame:
    if hist.empty or "round" not in hist.columns:
        return pd.DataFrame()

    max_round = int(pd.to_numeric(hist["round"], errors="coerce").max())
    part = hist[pd.to_numeric(hist["round"], errors="coerce") == max_round].copy()

    keep_cols = [
        "cluster_id",
        "cluster_size",
        "members",
        "member_indices",
    ]
    keep_cols = [c for c in keep_cols if c in part.columns]
    if not keep_cols:
        return pd.DataFrame()

    out = part[keep_cols].copy()

    if "members" in out.columns:
        out["family_mix"] = [
            _family_mix_from_members_text(rr.get("members", ""))
            for _, rr in part.iterrows()
        ]

    out = out.sort_values("cluster_id").reset_index(drop=True)
    return out


def _gcfl_eligible_events_df(hist: pd.DataFrame) -> pd.DataFrame:
    if hist.empty:
        return pd.DataFrame()

    mask = pd.Series(False, index=hist.index)
    if "split_status" in hist.columns:
        mask |= hist["split_status"].astype(str).eq("eligible")
    if "split_triggered" in hist.columns:
        mask |= pd.to_numeric(hist["split_triggered"], errors="coerce").fillna(0).eq(1)
    if "split_applied" in hist.columns:
        mask |= pd.to_numeric(hist["split_applied"], errors="coerce").fillna(0).eq(1)

    out = hist[mask].copy()
    if out.empty:
        return pd.DataFrame()

    cols = [
        "round",
        "cluster_id",
        "cluster_size",
        "mean_update_norm",
        "eps1",
        "max_update_norm",
        "eps2",
        "split_status",
        "split_triggered",
        "split_applied",
        "cut_status",
        "split_rejected_reason",
        "left_size",
        "right_size",
        "child_balance_ratio",
        "singleton_graph_id",
        "cut_value",
    ]
    cols = [c for c in cols if c in out.columns]
    return out[cols].sort_values(["round", "cluster_id"]).reset_index(drop=True)


def _gcfl_applied_events_df(hist: pd.DataFrame) -> pd.DataFrame:
    if hist.empty or "split_applied" not in hist.columns:
        return pd.DataFrame()

    out = hist[
        pd.to_numeric(hist["split_applied"], errors="coerce").fillna(0).eq(1)
    ].copy()
    if out.empty:
        return pd.DataFrame()

    cols = [
        "round",
        "cluster_id",
        "cluster_size",
        "left_child_id",
        "left_size",
        "right_child_id",
        "right_size",
        "child_balance_ratio",
        "singleton_graph_id",
        "cut_value",
        "cross_dtw_mean",
        "left_dtw_mean",
        "right_dtw_mean",
        "cross_aff_mean",
        "left_aff_mean",
        "right_aff_mean",
        "member_mean_dtw_text",
        "member_mean_aff_text",
    ]
    cols = [c for c in cols if c in out.columns]
    return out[cols].sort_values(["round", "cluster_id"]).reset_index(drop=True)


def _gcfl_lineage_df(hist: pd.DataFrame) -> pd.DataFrame:
    if hist.empty:
        return pd.DataFrame()

    applied = hist[
        pd.to_numeric(hist.get("split_applied", 0), errors="coerce").fillna(0).eq(1)
    ].copy()
    if applied.empty:
        return pd.DataFrame()

    cols = [
        "round",
        "cluster_id",
        "cluster_size",
        "left_child_id",
        "left_size",
        "right_child_id",
        "right_size",
        "left_members",
        "right_members",
    ]
    cols = [c for c in cols if c in applied.columns]
    out = applied[cols].copy()
    out = out.rename(columns={"cluster_id": "parent_cluster_id"})
    return out.sort_values(["round", "parent_cluster_id"]).reset_index(drop=True)


def _gcfl_clusters_of_interest(seed_pack: dict) -> list[int]:
    hist = seed_pack["history_df"]
    if hist.empty:
        return []

    mask = pd.Series(False, index=hist.index)

    if "split_status" in hist.columns:
        mask |= hist["split_status"].astype(str).eq("eligible")
    if "split_triggered" in hist.columns:
        mask |= pd.to_numeric(hist["split_triggered"], errors="coerce").fillna(0).eq(1)
    if "split_applied" in hist.columns:
        mask |= pd.to_numeric(hist["split_applied"], errors="coerce").fillna(0).eq(1)

    ids = hist.loc[mask, "cluster_id"].dropna().astype(int).unique().tolist()

    if len(ids) == 0:
        finals = seed_pack.get("finals_df", pd.DataFrame())
        if not finals.empty and "cluster_id" in finals.columns:
            ids = finals["cluster_id"].dropna().astype(int).tolist()

    return sorted(ids)


def build_gcfl_story_seed_frames(gcfl_items, row):
    seed_frames = []

    gcfl_items = sorted(gcfl_items, key=lambda x: int(x["seed"]))

    for item in gcfl_items:
        seed = int(item["seed"])
        hist = _gcfl_history_df(item.get("clusters_df", pd.DataFrame()))
        round_diag = _gcfl_round_diag_df(item)

        triggered_rounds = _gcfl_triggered_rounds(hist)
        applied_rounds = _gcfl_applied_rounds(hist)

        finals_df = _gcfl_last_round_clusters_df(hist)
        eligible_df = _gcfl_eligible_events_df(hist)
        applied_df = _gcfl_applied_events_df(hist)
        lineage_df = _gcfl_lineage_df(hist)

        status_counts_df = _gcfl_status_count_df(hist, "split_status")
        outcome_counts_df = _gcfl_status_count_df(hist, "cut_status")

        if not finals_df.empty and "cluster_id" in finals_df.columns:
            final_cluster_ids = finals_df["cluster_id"].dropna().astype(int).tolist()
            final_cluster_sizes = (
                finals_df["cluster_size"].dropna().astype(int).tolist()
                if "cluster_size" in finals_df.columns
                else []
            )
            final_cluster_size_text = "|".join(
                f"{cid}:{sz}" for cid, sz in zip(final_cluster_ids, final_cluster_sizes)
            )
        else:
            final_cluster_ids = []
            final_cluster_size_text = ""

        summary_df = pd.DataFrame(
            [
                {
                    "Seed": seed,
                    "eps1 q": _gcfl_item_meta(item, "eps1_quantile", np.nan),
                    "eps2 q": _gcfl_item_meta(item, "eps2_quantile", np.nan),
                    "Warmup": _gcfl_item_meta(item, "warmup_rounds", np.nan),
                    "Hist len": _gcfl_item_meta(item, "grad_seq_len", np.nan),
                    "Min cluster": _gcfl_item_meta(item, "min_cluster_size", np.nan),
                    "Min child": _gcfl_item_meta(item, "min_child_size", np.nan),
                    "Triggered rounds": ", ".join(str(x) for x in triggered_rounds),
                    "Applied rounds": ", ".join(str(x) for x in applied_rounds),
                    "Final active IDs": ", ".join(str(x) for x in final_cluster_ids),
                    "Final active sizes": final_cluster_size_text,
                }
            ]
        )

        seed_frames.append(
            {
                "seed": seed,
                "item": item,
                "history_df": hist,
                "round_diag_df": round_diag,
                "eligible_df": eligible_df,
                "applied_df": applied_df,
                "lineage_df": lineage_df,
                "finals_df": finals_df,
                "status_counts_df": status_counts_df,
                "outcome_counts_df": outcome_counts_df,
                "summary_df": summary_df,
                "triggered_rounds": triggered_rounds,
                "applied_rounds": applied_rounds,
            }
        )

    return seed_frames


def make_gcfl_seed_timeline_figure(row, seed_pack, test_gen):
    fig = plt.figure(figsize=(18, 10))
    gs = GridSpec(2, 1, figure=fig, height_ratios=[1.0, 0.9], hspace=0.28)

    ax0 = fig.add_subplot(gs[0, 0])
    ax1 = fig.add_subplot(gs[1, 0])

    rd = seed_pack["round_diag_df"].copy()

    if rd.empty:
        annotate_no_data(ax0)
    else:
        ok = False
        if "num_active_clusters" in rd.columns:
            ax0.plot(
                rd["round"],
                rd["num_active_clusters"],
                linewidth=2.0,
                label="# active clusters",
            )
            ok = True
        if "num_split_triggered" in rd.columns:
            ax0.plot(
                rd["round"],
                rd["num_split_triggered"],
                linewidth=1.8,
                label="# triggered splits",
            )
            ok = True
        if "num_split_applied" in rd.columns:
            ax0.plot(
                rd["round"],
                rd["num_split_applied"],
                linewidth=1.8,
                label="# applied splits",
            )
            ok = True

        for rr in seed_pack["triggered_rounds"]:
            ax0.axvline(rr, linestyle=":", linewidth=1.0, alpha=0.18)
        for rr in seed_pack["applied_rounds"]:
            ax0.axvline(rr, linestyle="-.", linewidth=1.0, alpha=0.18)

        ax0.set_title(f"Split timeline | seed={seed_pack['seed']}")
        ax0.set_xlabel("round")
        ax0.set_ylabel("count")
        ax0.grid(alpha=0.3)
        if ok:
            ax0.legend(fontsize=8)

    summary_df = _compact_gcfl_summary_df(seed_pack["summary_df"])

    draw_generic_table(
        ax1,
        summary_df,
        f"Seed summary | seed={seed_pack['seed']}",
        fontsize=8.6,
        max_rows=2,
        wrap_widths={
            "Triggered": 20,
            "Applied": 20,
            "Final IDs": 20,
            "Final sizes": 20,
        },
    )

    _apply_page_header(
        fig,
        title=build_gcfl_seed_title(row, seed_pack, "split timeline"),
        header_text=_compose_gcfl_seed_header_text(row, test_gen),
    )
    return fig


def make_gcfl_seed_gate_figure(row, seed_pack, test_gen):
    fig = plt.figure(figsize=(18, 13))
    gs = GridSpec(
        2, 2, figure=fig, height_ratios=[0.75, 1.25], hspace=0.28, wspace=0.20
    )

    ax0 = fig.add_subplot(gs[0, 0])
    ax1 = fig.add_subplot(gs[0, 1])
    ax2 = fig.add_subplot(gs[1, :])

    status_counts_df = seed_pack["status_counts_df"]
    outcome_counts_df = seed_pack["outcome_counts_df"]
    eligible_df = _compact_gcfl_eligible_df(seed_pack["eligible_df"])

    if status_counts_df.empty:
        annotate_no_data(ax0)
    else:
        ax0.bar(status_counts_df["label"], status_counts_df["count"])
        ax0.set_title("Split-status counts")
        ax0.set_ylabel("count")
        ax0.tick_params(axis="x", rotation=30)
        ax0.grid(alpha=0.3, axis="y")

    if outcome_counts_df.empty:
        annotate_no_data(ax1)
    else:
        ax1.bar(outcome_counts_df["label"], outcome_counts_df["count"])
        ax1.set_title("Cut-status counts")
        ax1.set_ylabel("count")
        ax1.tick_params(axis="x", rotation=30)
        ax1.grid(alpha=0.3, axis="y")

    draw_generic_table(
        ax2,
        eligible_df,
        f"Eligible / triggered / applied events | seed={seed_pack['seed']}",
        fontsize=7.0,
        max_rows=22,
        wrap_widths={
            "reject": 18,
        },
    )

    _apply_page_header(
        fig,
        title=build_gcfl_seed_title(row, seed_pack, "split gate diagnosis"),
        header_text=_compose_gcfl_seed_header_text(row, test_gen),
    )
    return fig


def make_gcfl_seed_cut_figure(row, seed_pack, test_gen):
    story_df = _gcfl_story_df(seed_pack)

    if story_df.empty:
        return None

    fig = plt.figure(figsize=(18, 12))
    gs = GridSpec(2, 1, figure=fig, height_ratios=[0.95, 1.05], hspace=0.24)

    ax0 = fig.add_subplot(gs[0, 0])
    ax1 = fig.add_subplot(gs[1, 0])

    _draw_gcfl_story_lineage(ax0, story_df, seed_pack["seed"])

    compact_df = _compact_gcfl_story_df(story_df)
    draw_generic_table(
        ax1,
        compact_df,
        f"Applied split story | seed={seed_pack['seed']}",
        fontsize=7.4,
        max_rows=20,
        wrap_widths={
            "trigger rule": 24,
            "peeled family": 20,
            "kept family mix": 28,
        },
    )

    _apply_page_header(
        fig,
        title=build_gcfl_seed_title(row, seed_pack, "split story"),
        header_text=_compose_gcfl_seed_header_text(row, test_gen),
    )
    return fig


def make_gcfl_seed_first_split_figure(row, seed_pack, test_gen):
    def _parse_member_score_text(text):
        out = []
        if text is None or (isinstance(text, float) and pd.isna(text)):
            return pd.DataFrame(columns=["graph_id", "mean_dtw"])

        s = str(text).strip()
        if s == "" or s.lower() == "nan":
            return pd.DataFrame(columns=["graph_id", "mean_dtw"])

        for part in s.split("|"):
            part = str(part).strip()
            if ":" not in part:
                continue
            gid, val = part.split(":", 1)
            gid = _normalize_gid_token(gid)
            try:
                score = float(val)
            except ValueError:
                score = np.nan
            out.append({"graph_id": gid, "mean_dtw": score})

        return pd.DataFrame(out)

    def _fmt(x):
        if pd.isna(x):
            return "nan"
        return f"{float(x):.3f}"

    hist = _gcfl_history_df(seed_pack["history_df"])
    if hist.empty or "split_applied" not in hist.columns:
        return None

    part = hist[
        pd.to_numeric(hist["split_applied"], errors="coerce").fillna(0).eq(1)
    ].copy()

    if part.empty:
        return None

    part = part.sort_values(["round", "cluster_id"]).reset_index(drop=True)
    rr = part.iloc[0]

    singleton_graph = _normalize_gid_token(rr.get("singleton_graph_id", np.nan))

    dtw_df = _parse_member_score_text(rr.get("member_mean_dtw_text", ""))
    if dtw_df.empty:
        return None

    dtw_df["graph_id"] = dtw_df["graph_id"].map(_normalize_gid_token)
    dtw_sorted = (
        dtw_df.sort_values("mean_dtw", ascending=False).reset_index(drop=True).copy()
    )
    dtw_sorted["is_singleton"] = dtw_sorted["graph_id"].eq(singleton_graph)

    max_dtw_graph = (
        _normalize_gid_token(dtw_sorted.iloc[0]["graph_id"])
        if len(dtw_sorted) > 0
        else None
    )

    summary_df = pd.DataFrame(
        [
            {
                "Seed": int(seed_pack["seed"]),
                "Round": int(pd.to_numeric(rr.get("round", np.nan))),
                "Parent cluster": int(pd.to_numeric(rr.get("cluster_id", np.nan))),
                "Split sizes": (
                    f"{int(pd.to_numeric(rr.get('left_size', np.nan)))} vs "
                    f"{int(pd.to_numeric(rr.get('right_size', np.nan)))}"
                ),
                "Singleton graph": (
                    singleton_graph if singleton_graph is not None else "-"
                ),
                "Max mean DTW": max_dtw_graph if max_dtw_graph is not None else "-",
                "Singleton = maxDTW": (
                    "yes" if singleton_graph == max_dtw_graph else "no"
                ),
            }
        ]
    )

    separation_df = pd.DataFrame(
        [
            {
                "Cross DTW": _fmt(rr.get("cross_dtw_mean", np.nan)),
                "Left DTW": _fmt(rr.get("left_dtw_mean", np.nan)),
                "Right DTW": _fmt(rr.get("right_dtw_mean", np.nan)),
            }
        ]
    )

    fig = plt.figure(figsize=(18, 11))
    gs = GridSpec(
        3,
        1,
        figure=fig,
        height_ratios=[0.55, 0.38, 1.0],
        hspace=0.30,
    )

    ax0 = fig.add_subplot(gs[0, 0])
    ax1 = fig.add_subplot(gs[1, 0])
    ax2 = fig.add_subplot(gs[2, 0])

    draw_generic_table(
        ax0,
        summary_df,
        f"First applied split summary | seed={seed_pack['seed']}",
        fontsize=10.0,
        max_rows=5,
        wrap_widths={
            "Singleton = maxDTW": 18,
        },
    )

    draw_generic_table(
        ax1,
        separation_df,
        "DTW separation diagnostics",
        fontsize=10.0,
        max_rows=5,
        wrap_widths={},
    )

    dtw_colors = [
        "tab:orange" if bool(v) else "tab:blue"
        for v in dtw_sorted["is_singleton"].tolist()
    ]
    ax2.bar(dtw_sorted["graph_id"], dtw_sorted["mean_dtw"], color=dtw_colors)
    ax2.set_title("Per-member mean DTW")
    ax2.set_xlabel("graph_id")
    ax2.set_ylabel("mean DTW to other members")
    ax2.tick_params(axis="x", rotation=45)
    ax2.grid(alpha=0.3)

    _apply_page_header(
        fig,
        title=build_gcfl_seed_title(row, seed_pack, "first applied split diagnostic"),
        header_text=_compose_gcfl_seed_header_text(row, test_gen),
    )
    return fig


def _gcfl_first_applied_split_row(seed_pack):
    hist = _gcfl_history_df(seed_pack["history_df"])
    if hist.empty:
        return None

    part = hist.copy()

    if "split_applied" in part.columns:
        part = part[
            pd.to_numeric(part["split_applied"], errors="coerce").fillna(0).eq(1)
        ].copy()

    if "pairwise_logged_for_applied_split" in part.columns:
        part = part[
            pd.to_numeric(part["pairwise_logged_for_applied_split"], errors="coerce")
            .fillna(0)
            .eq(1)
        ].copy()

    if part.empty:
        return None

    if "round" in part.columns:
        part["round"] = pd.to_numeric(part["round"], errors="coerce")
    if "cluster_id" in part.columns:
        part["cluster_id"] = pd.to_numeric(part["cluster_id"], errors="coerce")

    part = part.sort_values(["round", "cluster_id"]).reset_index(drop=True)
    return part.iloc[0]


def _gcfl_pairwise_matrix_df_from_row(split_row, value_key="dtw"):
    if split_row is None:
        return pd.DataFrame()

    order_json = split_row.get("pairwise_member_order_json", None)
    order = parse_json_list_safe(order_json)
    if len(order) == 0:
        return pd.DataFrame()

    order = sorted(order, key=lambda d: int(d.get("pos", 0)))
    labels = [str(d.get("graph_id", d.get("member_idx", "?"))) for d in order]
    n = len(labels)

    diag_value = 0.0 if value_key == "dtw" else 1.0
    mat = np.full((n, n), np.nan, dtype=float)
    np.fill_diagonal(mat, diag_value)

    pair_col = "pairwise_dtw_json" if value_key == "dtw" else "pairwise_affinity_json"
    pair_data = parse_json_list_safe(split_row.get(pair_col, None))

    for rec in pair_data:
        i = int(rec["row_pos"])
        j = int(rec["col_pos"])
        v = float(rec[value_key])
        mat[i, j] = v
        mat[j, i] = v

    return pd.DataFrame(mat, index=labels, columns=labels)


def _draw_gcfl_matrix_table(
    ax,
    matrix_df,
    title,
    decimals=1,
    fontsize=5.6,
    highlight_global_max=False,
):
    ax.axis("off")

    if matrix_df is None or matrix_df.empty:
        annotate_no_data(ax)
        return

    view = matrix_df.copy()
    view.insert(0, "graph_id", view.index.astype(str))

    col_labels = list(view.columns)
    cell_text = []

    for _, rr in view.iterrows():
        row_vals = []
        for cc in col_labels:
            val = rr[cc]
            if cc == "graph_id":
                row_vals.append(str(val))
            else:
                row_vals.append("-" if pd.isna(val) else f"{float(val):.{decimals}f}")
        cell_text.append(row_vals)

    table = ax.table(
        cellText=cell_text,
        colLabels=col_labels,
        cellLoc="center",
        loc="center",
    )
    table.auto_set_font_size(False)
    table.set_fontsize(fontsize)
    table.scale(1.0, 1.18)

    if highlight_global_max and matrix_df.shape[0] > 1:
        arr = matrix_df.to_numpy(dtype=float)
        tri = np.triu_indices(arr.shape[0], k=1)
        vals = arr[tri]

        finite_mask = np.isfinite(vals)
        if finite_mask.any():
            valid_positions = np.where(finite_mask)[0]
            k_local = int(np.argmax(vals[finite_mask]))
            k = int(valid_positions[k_local])

            i = int(tri[0][k])
            j = int(tri[1][k])
            v = float(arr[i, j])

            # table row/col coordinates:
            # +1 row because row 0 is header
            # +1 col because col 0 is the inserted "graph_id" column
            table[(i + 1, j + 1)].set_facecolor("#ffcc80")
            table[(i + 1, j + 1)].set_edgecolor("red")
            table[(i + 1, j + 1)].set_linewidth(2.5)

            ax.set_title(
                f"{title}\nGlobal max DTW = {v:.1f} at ({matrix_df.index[i]}, {matrix_df.columns[j]})",
                pad=10,
            )
            return

    ax.set_title(title, pad=10)


def make_gcfl_seed_first_split_pairwise_dtw_table_figure(row, seed_pack, test_gen):
    split_row = _gcfl_first_applied_split_row(seed_pack)
    if split_row is None:
        return None

    dtw_df = _gcfl_pairwise_matrix_df_from_row(split_row, value_key="dtw")
    if dtw_df.empty:
        return None

    fig = plt.figure(figsize=(18, 12))
    gs = GridSpec(2, 1, figure=fig, height_ratios=[0.22, 0.78], hspace=0.18)

    ax0 = fig.add_subplot(gs[0, 0])
    ax1 = fig.add_subplot(gs[1, 0])

    summary_df = pd.DataFrame(
        [
            {
                "Seed": int(seed_pack["seed"]),
                "Round": (
                    int(split_row["round"])
                    if pd.notna(split_row.get("round", np.nan))
                    else np.nan
                ),
                "Parent cluster": (
                    int(split_row["cluster_id"])
                    if pd.notna(split_row.get("cluster_id", np.nan))
                    else np.nan
                ),
                "n_members": int(dtw_df.shape[0]),
                "Singleton": split_row.get("singleton_graph_id", "-"),
                "Cut value": _fmt_float(split_row.get("cut_value", np.nan), 3),
                "Cross DTW": _fmt_float(split_row.get("cross_dtw_mean", np.nan), 3),
                "Pairwise logged": (
                    int(split_row.get("pairwise_logged_for_applied_split", 0))
                    if pd.notna(
                        split_row.get("pairwise_logged_for_applied_split", np.nan)
                    )
                    else 0
                ),
            }
        ]
    )

    draw_generic_table(
        ax0,
        summary_df,
        f"First applied split pairwise DTW metadata | seed={seed_pack['seed']}",
        fontsize=8.5,
        max_rows=2,
    )

    _draw_gcfl_matrix_table(
        ax1,
        dtw_df,
        f"Pairwise DTW table | first applied split | seed={seed_pack['seed']}",
        decimals=1,
        fontsize=5.6,
        highlight_global_max=True,
    )

    _apply_page_header(
        fig,
        title=build_gcfl_seed_title(
            row, seed_pack, "first applied split pairwise DTW table"
        ),
        header_text=_compose_gcfl_seed_header_text(row, test_gen),
    )
    return fig


def _plot_gcfl_single_cluster_metric_threshold(
    ax,
    history_df,
    *,
    cluster_id,
    metric_col,
    threshold_col,
    metric_title,
    threshold_title,
    ylabel,
):
    hist = _gcfl_history_df(history_df)
    hist = hist[
        pd.to_numeric(hist["cluster_id"], errors="coerce") == int(cluster_id)
    ].copy()

    if hist.empty:
        annotate_no_data(ax)
        return

    hist["round"] = pd.to_numeric(hist["round"], errors="coerce")
    if metric_col in hist.columns:
        hist[metric_col] = pd.to_numeric(hist[metric_col], errors="coerce")
    if threshold_col in hist.columns:
        hist[threshold_col] = pd.to_numeric(hist[threshold_col], errors="coerce")
    hist = hist.sort_values("round").copy()

    x = hist["round"].to_numpy(dtype=float)

    if metric_col in hist.columns:
        y = hist[metric_col].to_numpy(dtype=float)
        mask_y = np.isfinite(x) & np.isfinite(y)
        if np.any(mask_y):
            ax.plot(x[mask_y], y[mask_y], linewidth=2.0, label=metric_title)

    if threshold_col in hist.columns:
        t = hist[threshold_col].to_numpy(dtype=float)
        mask_t = np.isfinite(x) & np.isfinite(t)
        if np.any(mask_t):
            ax.plot(
                x[mask_t],
                t[mask_t],
                linestyle="--",
                linewidth=1.8,
                label=threshold_title,
            )

    triggered_rounds = sorted(
        hist.loc[
            pd.to_numeric(hist.get("split_triggered", 0), errors="coerce") == 1, "round"
        ]
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )
    applied_rounds = sorted(
        hist.loc[
            pd.to_numeric(hist.get("split_applied", 0), errors="coerce") == 1, "round"
        ]
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )

    for rr in triggered_rounds:
        ax.axvline(rr, color="red", linestyle=":", linewidth=1.0, alpha=0.18)
    for rr in applied_rounds:
        ax.axvline(rr, color="green", linestyle="-.", linewidth=1.0, alpha=0.18)

    ax.set_title(f"Cluster {int(cluster_id)} | {metric_title}")
    ax.set_xlabel("round")
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)


def make_gcfl_seed_norm_figures(row, seed_pack, test_gen, max_clusters_per_page=2):
    history_df = seed_pack["history_df"]
    hist = _gcfl_history_df(history_df)
    if hist.empty:
        return []

    cluster_ids = _gcfl_clusters_of_interest(seed_pack)
    if len(cluster_ids) == 0:
        return []

    figs = []

    for start in range(0, len(cluster_ids), max_clusters_per_page):
        chunk = cluster_ids[start : start + max_clusters_per_page]

        # Important:
        # The old version used figsize=(18, 6 * len(chunk)).
        # That was too short once the long GCFL+ metadata header was added.
        # This gives the header enough room without crushing the plots.
        fig_height = 7.8 * len(chunk) + 4.0
        fig = plt.figure(figsize=(18, fig_height))

        gs = GridSpec(
            len(chunk),
            2,
            figure=fig,
            hspace=0.55,
            wspace=0.24,
        )

        for r, cluster_id in enumerate(chunk):
            ax_mean = fig.add_subplot(gs[r, 0])
            ax_max = fig.add_subplot(gs[r, 1])

            _plot_gcfl_single_cluster_metric_threshold(
                ax_mean,
                hist,
                cluster_id=cluster_id,
                metric_col="mean_update_norm",
                threshold_col="eps1",
                metric_title="Mean update norm",
                threshold_title="Dynamic eps1 threshold",
                ylabel="mean update norm",
            )

            _plot_gcfl_single_cluster_metric_threshold(
                ax_max,
                hist,
                cluster_id=cluster_id,
                metric_col="max_update_norm",
                threshold_col="eps2",
                metric_title="Max update norm",
                threshold_title="Dynamic eps2 threshold",
                ylabel="max update norm",
            )

        _apply_page_header(
            fig,
            title=build_gcfl_seed_title(
                row,
                seed_pack,
                f"norm / threshold panels ({chunk[0]}-{chunk[-1]})",
            ),
            header_text=_compose_gcfl_seed_header_text(row, test_gen),
            title_fontsize="auto",
            header_fontsize="auto",
            header_width=165,
            min_subplot_top=0.58,
            max_subplot_top=0.78,
            bottom=0.055,
            hspace=0.50,
            wspace=0.30,
        )

        figs.append(fig)

    return figs


def make_gcfl_seed_final_figure(row, seed_pack, test_gen):
    finals_df = _compact_gcfl_finals_df(seed_pack["finals_df"])

    fig = plt.figure(figsize=(18, 8.5))
    gs = GridSpec(1, 1, figure=fig)
    ax0 = fig.add_subplot(gs[0, 0])

    draw_generic_table(
        ax0,
        finals_df,
        f"Final active clusters | seed={seed_pack['seed']}",
        fontsize=7.8,
        max_rows=20,
        wrap_widths={
            "members": 32,
            "member_indices": 32,
            "family_mix": 28,
        },
    )

    _apply_page_header(
        fig,
        title=build_gcfl_seed_title(row, seed_pack, "final active clusters"),
        header_text=_compose_gcfl_seed_header_text(row, test_gen),
    )
    return fig

## 6. Figure builders

In [46]:
def _load_overview_method_item_map(exp_log, subset_clients, model_tag, local_epochs=1):
    method_item_map = {}

    for method in METHOD_ORDER:
        rows = filter_method_manifest(
            exp_log,
            subset_clients=subset_clients,
            model_tag=model_tag,
            method=method,
            local_epochs=local_epochs,
        )
        items = load_method_items(rows)

        if len(items) > 0:
            method_item_map[method] = items

    return method_item_map


def _make_overview_axes():
    fig = plt.figure(figsize=(18, 26))

    gs = GridSpec(
        5,
        2,
        figure=fig,
        height_ratios=[0.86, 1.0, 1.0, 1.0, 1.05],
        hspace=0.34,
        wspace=0.18,
    )

    ax_table = fig.add_subplot(gs[0, :])
    ax_loss = fig.add_subplot(gs[1, :])

    ax_local = fig.add_subplot(gs[2, 0])
    ax_fedavg = fig.add_subplot(gs[2, 1])
    ax_fedprox = fig.add_subplot(gs[3, 0])
    ax_gcfl = fig.add_subplot(gs[3, 1])
    ax_apple = fig.add_subplot(gs[4, :])

    f1_axes = {
        "local": ax_local,
        "fedavg": ax_fedavg,
        "fedprox": ax_fedprox,
        "gcfl_plus": ax_gcfl,
        "apple": ax_apple,
    }

    return fig, ax_table, ax_loss, f1_axes


def _plot_overview_common(
    *,
    row,
    exp_log,
    test_gen,
    title_fn,
    header_fn,
):
    subset_clients = row["subset_clients"]
    model_tag = row["model_tag"]
    graph_ids = [str(g) for g in row["graph_ids"]]

    local_items = load_local_items(
        filter_local_manifest(
            exp_log,
            subset_clients,
            model_tag,
            graph_ids,
            local_epochs=1,
        )
    )

    method_item_map = _load_overview_method_item_map(
        exp_log,
        subset_clients=subset_clients,
        model_tag=model_tag,
        local_epochs=1,
    )

    fig, ax_table, ax_loss, f1_axes = _make_overview_axes()

    table_df = build_overview_algo_table(
        local_items=local_items,
        method_item_map=method_item_map,
        graph_ids=graph_ids,
        split="test",
    )

    draw_algo_table(
        ax_table,
        table_df,
        title=(
            "Best logged test performance "
            f"(all methods {BASELINE_ROUND_BUDGET} rounds; "
            f"APPLE mu={APPLE_PLOT_MU:g}, Lfrac={APPLE_PLOT_SCHEDULER_FRACTION:g})"
        ),
    )

    plot_overview_loss_panel(
        ax_loss,
        local_items=local_items,
        method_item_map=method_item_map,
        graph_ids=graph_ids,
    )

    plot_overview_local_f1_panel(
        f1_axes["local"],
        local_items=local_items,
        graph_ids=graph_ids,
        title="Pooled local centralized positive F1",
    )

    for method in METHOD_ORDER:
        ax = f1_axes[method]
        method_items = method_item_map.get(method, [])

        if len(method_items) == 0:
            annotate_no_data(ax, text=f"No {METHOD_LABELS[method]} data")
            ax.set_title(f"Pooled {METHOD_LABELS[method]} positive F1")
            ax.set_xlabel("round")
            ax.set_ylabel("positive_f1")
            ax.set_ylim(0.0, 1.0)
            ax.grid(alpha=0.3)
            continue

        plot_overview_method_global_f1_panel(
            ax,
            method_items=method_items,
            graph_ids=graph_ids,
            method_name=method,
        )

    _apply_page_header(
        fig,
        title=title_fn(row),
        header_text=header_fn(row, test_gen),
    )

    return fig


def make_overview_global_figure(row, exp_log, test_gen):
    return _plot_overview_common(
        row=row,
        exp_log=exp_log,
        test_gen=test_gen,
        title_fn=build_overview_global_title,
        header_fn=_compose_global_overview_header_text,
    )


def make_overview_family_figure(row, exp_log, test_gen):
    return _plot_overview_common(
        row=row,
        exp_log=exp_log,
        test_gen=test_gen,
        title_fn=build_overview_family_title,
        header_fn=_compose_family_overview_header_text,
    )


def make_method_global_figure(method, row, exp_log, test_gen):
    subset_clients = row["subset_clients"]
    model_tag = row["model_tag"]
    graph_ids = [str(g) for g in row["graph_ids"]]

    local_items = load_local_items(
        filter_local_manifest(
            exp_log, subset_clients, model_tag, graph_ids, local_epochs=1
        )
    )
    method_items = load_method_items(
        filter_method_manifest(
            exp_log, subset_clients, model_tag, method, local_epochs=1
        )
    )

    fig = plt.figure(figsize=(18, 24))
    gs = GridSpec(
        4,
        2,
        figure=fig,
        height_ratios=[0.78, 1.0, 1.0, 1.0],
        hspace=0.34,
        wspace=0.18,
    )

    ax0 = fig.add_subplot(gs[0, :])
    ax1 = fig.add_subplot(gs[1, :])
    ax2 = fig.add_subplot(gs[2, 0])
    ax3 = fig.add_subplot(gs[2, 1])
    ax4 = fig.add_subplot(gs[3, 0])
    ax5 = fig.add_subplot(gs[3, 1])

    table_df = build_method_algo_table(
        local_items=local_items,
        method_items=method_items,
        graph_ids=graph_ids,
        method=method,
        split="test",
    )
    draw_algo_table(
        ax0,
        table_df,
        title=f"Best logged test performance ({METHOD_LABELS[method]} vs Local centralized)",
    )

    plot_method_loss_panel(
        ax1,
        local_items=local_items,
        method_items=method_items,
        graph_ids=graph_ids,
        method_name=method,
    )
    plot_method_micro_macro_panel(
        ax2,
        local_items=local_items,
        method_items=method_items,
        graph_ids=graph_ids,
        source="local",
        method_name=method,
    )
    plot_method_micro_macro_panel(
        ax3,
        local_items=local_items,
        method_items=method_items,
        graph_ids=graph_ids,
        source="preagg",
        method_name=method,
    )
    plot_method_micro_macro_panel(
        ax4,
        local_items=local_items,
        method_items=method_items,
        graph_ids=graph_ids,
        source="postagg",
        method_name=method,
    )
    plot_method_delta_micro_macro_panel(
        ax5,
        local_items=local_items,
        method_items=method_items,
        graph_ids=graph_ids,
        method_name=method,
    )

    _apply_page_header(
        fig,
        title=build_method_global_title(method, row),
        header_text=_compose_method_global_header_text(method, row, test_gen),
    )
    return fig


def make_method_family_figure(method, row, exp_log, test_gen):
    subset_clients = row["subset_clients"]
    model_tag = row["model_tag"]
    graph_ids = [str(g) for g in row["graph_ids"]]

    local_items = load_local_items(
        filter_local_manifest(
            exp_log, subset_clients, model_tag, graph_ids, local_epochs=1
        )
    )
    method_items = load_method_items(
        filter_method_manifest(
            exp_log, subset_clients, model_tag, method, local_epochs=1
        )
    )

    fig = plt.figure(figsize=(18, 24))
    gs = GridSpec(
        4,
        2,
        figure=fig,
        height_ratios=[0.78, 1.0, 1.0, 1.0],
        hspace=0.34,
        wspace=0.18,
    )

    ax0 = fig.add_subplot(gs[0, :])
    ax1 = fig.add_subplot(gs[1, :])
    ax2 = fig.add_subplot(gs[2, 0])
    ax3 = fig.add_subplot(gs[2, 1])
    ax4 = fig.add_subplot(gs[3, 0])
    ax5 = fig.add_subplot(gs[3, 1])

    table_df = build_method_algo_table(
        local_items=local_items,
        method_items=method_items,
        graph_ids=graph_ids,
        method=method,
        split="test",
    )
    draw_algo_table(
        ax0,
        table_df,
        title=f"Best logged test performance ({METHOD_LABELS[method]} vs Local centralized)",
    )

    plot_method_loss_panel(
        ax1,
        local_items=local_items,
        method_items=method_items,
        graph_ids=graph_ids,
        method_name=method,
    )
    plot_method_micro_macro_panel(
        ax2,
        local_items=local_items,
        method_items=method_items,
        graph_ids=graph_ids,
        source="local",
        method_name=method,
    )
    plot_method_micro_macro_panel(
        ax3,
        local_items=local_items,
        method_items=method_items,
        graph_ids=graph_ids,
        source="preagg",
        method_name=method,
    )
    plot_method_micro_macro_panel(
        ax4,
        local_items=local_items,
        method_items=method_items,
        graph_ids=graph_ids,
        source="postagg",
        method_name=method,
    )
    plot_method_delta_micro_macro_panel(
        ax5,
        local_items=local_items,
        method_items=method_items,
        graph_ids=graph_ids,
        method_name=method,
    )

    _apply_page_header(
        fig,
        title=build_method_family_title(method, row),
        header_text=_compose_method_family_header_text(method, row, test_gen),
    )
    return fig


def make_gcfl_global_figure(row, exp_log, test_gen):
    return make_method_global_figure("gcfl_plus", row, exp_log, test_gen)


def make_gcfl_family_figure(row, exp_log, test_gen):
    return make_method_family_figure("gcfl_plus", row, exp_log, test_gen)


def make_gcfl_story_seed_figure(row, seed_pack, test_gen):
    seed = seed_pack["seed"]
    summary_df = seed_pack["summary_df"]
    split_events_df = seed_pack["split_events_df"]
    finals_df = seed_pack["finals_df"]
    history_df = seed_pack["history_df"]

    eps1_q = seed_pack.get("eps1_quantile", np.nan)
    eps2_q = seed_pack.get("eps2_quantile", np.nan)

    fig = plt.figure(figsize=(18, 23))
    gs = GridSpec(
        5,
        1,
        figure=fig,
        height_ratios=[0.58, 1.0, 1.0, 0.95, 0.95],
        hspace=0.30,
    )

    ax0 = fig.add_subplot(gs[0, 0])
    ax1 = fig.add_subplot(gs[1, 0])
    ax2 = fig.add_subplot(gs[2, 0])
    ax3 = fig.add_subplot(gs[3, 0])
    ax4 = fig.add_subplot(gs[4, 0])

    draw_generic_table(
        ax0,
        summary_df,
        f"GCFL+ trigger / split summary | seed={seed}",
        fontsize=8.0,
        max_rows=3,
        wrap_widths={
            "Triggered rounds": 28,
            "Applied rounds": 28,
            "Cluster IDs seen": 28,
            "Status counts": 30,
        },
    )

    _plot_gcfl_metric_threshold_panel(
        ax1,
        history_df,
        metric_col="mean_update_norm",
        threshold_col="eps1",
        metric_label="mean",
        threshold_label=f"eps1(q={eps1_q:.2f})" if pd.notna(eps1_q) else "eps1",
        ylabel="mean update norm",
        title=(
            "Per-cluster mean update norm with numeric eps1 threshold "
            "(threshold is computed from previous rounds only)"
        ),
    )

    _plot_gcfl_metric_threshold_panel(
        ax2,
        history_df,
        metric_col="max_update_norm",
        threshold_col="eps2",
        metric_label="max",
        threshold_label=f"eps2(q={eps2_q:.2f})" if pd.notna(eps2_q) else "eps2",
        ylabel="max update norm",
        title=(
            "Per-cluster max update norm with numeric eps2 threshold "
            "(threshold is computed from previous rounds only)"
        ),
    )

    draw_generic_table(
        ax3,
        split_events_df,
        (
            "GCFL+ split-events table | ONLY rows where split_triggered=1 "
            f"or split_applied=1 | seed={seed}"
        ),
        fontsize=7.4,
        max_rows=16,
        wrap_widths={
            "Status": 14,
            "Left mix": 18,
            "Right mix": 18,
        },
    )

    draw_generic_table(
        ax4,
        finals_df,
        f"GCFL+ final cluster composition | seed={seed}",
        fontsize=8.0,
        max_rows=20,
        wrap_widths={
            "Members": 28,
            "Family mix": 22,
        },
    )

    fig.suptitle(
        f"Heterogeneity level: {_level_title(row)}"
        f" | GCFL+ clustering story"
        f" | subset_size={row['subset_size']}"
        f" | seed={seed}"
        f" | model={row['model_tag']}"
        f" | local_epochs=1",
        fontsize=16,
        y=0.992,
    )
    fig.text(
        0.5,
        0.978,
        _compose_gcfl_seed_header_text(row, test_gen),
        ha="center",
        va="top",
        fontsize=9,
    )

    fig.tight_layout(rect=[0.01, 0.03, 0.99, 0.965])
    return fig

## 7. PDF / combined visual table export


In [47]:
def _sort_global_table(df):
    if df is None or df.empty:
        return df

    sort_cols = [c for c in ["heterogeneity_order_idx", "model_tag"] if c in df.columns]

    if sort_cols:
        return df.sort_values(sort_cols).reset_index(drop=True)

    return df.reset_index(drop=True)


def _sort_family_table(df):
    if df is None or df.empty:
        return df

    sort_cols = [
        c
        for c in ["heterogeneity_order_idx", "family_order_idx", "model_tag"]
        if c in df.columns
    ]

    if sort_cols:
        return df.sort_values(sort_cols).reset_index(drop=True)

    return df.reset_index(drop=True)


def _family_rows_for_global_row(family_run_table, global_row):
    fam = family_run_table.copy()

    mask = (fam["subset_clients"].astype(str) == str(global_row["subset_clients"])) & (
        fam["model_tag"].astype(str) == str(global_row["model_tag"])
    )

    if (
        "heterogeneity_level" in fam.columns
        and "heterogeneity_level" in global_row.index
    ):
        mask = mask & (
            fam["heterogeneity_level"].astype(str)
            == str(global_row["heterogeneity_level"])
        )

    out = fam[mask].copy()
    return _sort_family_table(out)


def _filter_level(df, level):
    if df is None or df.empty:
        return df

    if "heterogeneity_level" not in df.columns:
        return df.iloc[[]].copy()

    return df[df["heterogeneity_level"].astype(str) == str(level)].copy()


def build_all_methods_overview_pdf(
    global_run_table, family_run_table, exp_log, test_gen, out_path
):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    global_sorted = _sort_global_table(global_run_table)

    with PdfPages(out_path) as pdf:
        for _, grow in global_sorted.iterrows():
            fig = make_overview_global_figure(grow, exp_log, test_gen)
            pdf.savefig(fig, bbox_inches="tight")
            plt.close(fig)

            fam_rows = _family_rows_for_global_row(family_run_table, grow)
            for _, frow in fam_rows.iterrows():
                fig = make_overview_family_figure(frow, exp_log, test_gen)
                pdf.savefig(fig, bbox_inches="tight")
                plt.close(fig)

    print(f"saved -> {out_path.resolve()}")


def build_all_methods_overview_pdfs_by_heterogeneity(
    global_run_table,
    family_run_table,
    exp_log,
    test_gen,
    out_dir,
):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    for level in HETEROGENEITY_LEVEL_ORDER:
        g_part = _filter_level(global_run_table, level)
        f_part = _filter_level(family_run_table, level)

        if g_part.empty:
            print(f"skip overview for {level}: no rows")
            continue

        build_all_methods_overview_pdf(
            global_run_table=g_part,
            family_run_table=f_part,
            exp_log=exp_log,
            test_gen=test_gen,
            out_path=out_dir
            / f"{level}_heterogeneity_methods_overview_with_tables.pdf",
        )


def build_method_diagnostics_pdf(
    method,
    global_run_table,
    family_run_table,
    exp_log,
    test_gen,
    out_path,
):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    global_sorted = _sort_global_table(global_run_table)

    with PdfPages(out_path) as pdf:
        # Order:
        # low global -> low families
        # mid global -> mid families
        # high global -> high families
        for _, grow in global_sorted.iterrows():
            fig = make_method_global_figure(method, grow, exp_log, test_gen)
            pdf.savefig(fig, bbox_inches="tight")
            plt.close(fig)

            fam_rows = _family_rows_for_global_row(family_run_table, grow)
            for _, frow in fam_rows.iterrows():
                fig = make_method_family_figure(method, frow, exp_log, test_gen)
                pdf.savefig(fig, bbox_inches="tight")
                plt.close(fig)

    print(f"saved -> {out_path.resolve()}")


def build_gcfl_diagnostics_pdf(
    global_run_table,
    family_run_table,
    exp_log,
    test_gen,
    out_path,
):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    global_sorted = _sort_global_table(global_run_table)

    with PdfPages(out_path) as pdf:
        # Order:
        # low GCFL story pages -> low global -> low families
        # mid GCFL story pages -> mid global -> mid families
        # high GCFL story pages -> high global -> high families
        for _, grow in global_sorted.iterrows():
            if "gcfl_plus" not in grow["methods"]:
                continue

            subset_clients = grow["subset_clients"]
            model_tag = grow["model_tag"]

            gcfl_items = load_method_items(
                filter_method_manifest(
                    exp_log,
                    subset_clients,
                    model_tag,
                    "gcfl_plus",
                    local_epochs=1,
                )
            )

            gcfl_items = sorted(gcfl_items, key=lambda x: int(x["seed"]))
            if len(gcfl_items) == 0:
                continue

            seed_story_frames = build_gcfl_story_seed_frames(gcfl_items, grow)

            for seed_pack in seed_story_frames:
                fig = make_gcfl_seed_first_split_figure(grow, seed_pack, test_gen)
                if fig is not None:
                    pdf.savefig(fig, bbox_inches="tight")
                    plt.close(fig)

                fig = make_gcfl_seed_first_split_pairwise_dtw_table_figure(
                    grow, seed_pack, test_gen
                )
                if fig is not None:
                    pdf.savefig(fig, bbox_inches="tight")
                    plt.close(fig)

                fig = make_gcfl_seed_timeline_figure(grow, seed_pack, test_gen)
                if fig is not None:
                    pdf.savefig(fig, bbox_inches="tight")
                    plt.close(fig)

                fig = make_gcfl_seed_cut_figure(grow, seed_pack, test_gen)
                if fig is not None:
                    pdf.savefig(fig, bbox_inches="tight")
                    plt.close(fig)

                fig = make_gcfl_seed_final_figure(grow, seed_pack, test_gen)
                if fig is not None:
                    pdf.savefig(fig, bbox_inches="tight")
                    plt.close(fig)

                for fig in make_gcfl_seed_norm_figures(grow, seed_pack, test_gen):
                    if fig is not None:
                        pdf.savefig(fig, bbox_inches="tight")
                        plt.close(fig)

            fig = make_gcfl_global_figure(grow, exp_log, test_gen)
            if fig is not None:
                pdf.savefig(fig, bbox_inches="tight")
                plt.close(fig)

            fam_rows = _family_rows_for_global_row(family_run_table, grow)
            for _, frow in fam_rows.iterrows():
                if "gcfl_plus" not in frow["methods"]:
                    continue

                fig = make_gcfl_family_figure(frow, exp_log, test_gen)
                if fig is not None:
                    pdf.savefig(fig, bbox_inches="tight")
                    plt.close(fig)

    print(f"saved -> {out_path.resolve()}")


# =============================================================================
# APPLE diagnostics
# =============================================================================

# APPLE after-local model:
# updated self core + start-of-round donor cores + updated DR vector.
# This is now the selected / paper-faithful APPLE validation curve.
APPLE_LOCAL_UPDATE_SCALAR_PHASE = "val_epoch"
APPLE_LOCAL_UPDATE_TASK_PHASE = "val_epoch_task"
APPLE_LOCAL_MEAN_SCALAR_PHASE = "apple_local_val_mean"

# APPLE end-of-round recomputed model:
# updated self core + updated donor cores + same learned DR vector.
# This is diagnostic only now.
APPLE_ENDROUND_SCALAR_PHASE = "apple_val_client"
APPLE_ENDROUND_TASK_PHASE = "apple_val_client_task"
APPLE_ENDROUND_MEAN_SCALAR_PHASE = "apple_val_mean"

# Backward-compatible aliases, in case older helper code still refers to these names.
APPLE_PERSONALIZED_SCALAR_PHASE = APPLE_ENDROUND_SCALAR_PHASE
APPLE_PERSONALIZED_TASK_PHASE = APPLE_ENDROUND_TASK_PHASE


def _apple_method_items_for_row(row, exp_log):
    return load_method_items(
        filter_method_manifest(
            exp_log,
            subset_clients=row["subset_clients"],
            model_tag=row["model_tag"],
            method="apple",
            local_epochs=1,
        )
    )


def _apple_local_items_for_row(row, exp_log):
    graph_ids = [str(g) for g in row["graph_ids"]]
    return load_local_items(
        filter_local_manifest(
            exp_log,
            subset_clients=row["subset_clients"],
            model_tag=row["model_tag"],
            graph_ids=graph_ids,
            local_epochs=1,
        )
    )


def _apple_graph_label_map(row):
    graph_ids = [str(g) for g in row["graph_ids"]]

    try:
        graph_to_family = _graph_to_family_map_from_row(row)
    except Exception:
        graph_to_family = {}

    out = {}
    for gid in graph_ids:
        fam = graph_to_family.get(str(gid), str(row.get("family", gid)))
        label = str(fam).replace("specialized_", "")
        out[str(gid)] = label

    return out


def plot_apple_loss_panel(ax, *, local_items, apple_items, graph_ids):
    ok = False

    local_agg = get_weighted_group_scalar_stats(
        items=local_items,
        phase="val_epoch",
        split="val",
        metric_col="eval_loss",
        graph_ids=graph_ids,
        use_graph_filter=False,
    )

    # Per-client after-local APPLE validation.
    # This is the selected / paper-faithful APPLE curve.
    apple_local_client_agg = get_weighted_group_loss_for_method(
        method_items=apple_items,
        phase=APPLE_LOCAL_UPDATE_SCALAR_PHASE,
        graph_ids=graph_ids,
    )

    # Explicit pooled row logged by apple_run_helper.py.
    # This should nearly overlap with the weighted per-client after-local curve.
    apple_local_mean_agg = get_weighted_group_scalar_stats(
        items=apple_items,
        phase=APPLE_LOCAL_MEAN_SCALAR_PHASE,
        split="val",
        metric_col="eval_loss",
        graph_ids=None,
        use_graph_filter=False,
    )

    # End-of-round recomputed APPLE model.
    # This is the old problematic curve; keep it only as diagnostic.
    apple_end_client_agg = get_weighted_group_loss_for_method(
        method_items=apple_items,
        phase=APPLE_ENDROUND_SCALAR_PHASE,
        graph_ids=graph_ids,
    )

    # Explicit pooled row for end-of-round diagnostic.
    apple_end_mean_agg = get_weighted_group_scalar_stats(
        items=apple_items,
        phase=APPLE_ENDROUND_MEAN_SCALAR_PHASE,
        split="val",
        metric_col="eval_loss",
        graph_ids=None,
        use_graph_filter=False,
    )

    ok |= plot_mean_std(ax, local_agg, label="Local centralized")
    ok |= plot_mean_std(
        ax,
        apple_local_client_agg,
        label="APPLE after-local selected model",
    )
    ok |= plot_mean_std(
        ax,
        apple_local_mean_agg,
        label="APPLE local mean row sanity",
        linestyle="--",
        alpha_fill=0.06,
    )
    ok |= plot_mean_std(
        ax,
        apple_end_client_agg,
        label="APPLE end-of-round diagnostic",
    )
    ok |= plot_mean_std(
        ax,
        apple_end_mean_agg,
        label="APPLE end-round mean row sanity",
        linestyle="--",
        alpha_fill=0.06,
    )

    ax.set_title(
        "Validation loss: selected APPLE vs end-of-round mismatch "
        f"(equal {APPLE_ROUND_BUDGET}-round budget)"
    )

    ax.set_xlabel("round")
    ax.set_ylabel("eval_loss")
    set_round_budget_axis(
        ax,
        max_round=APPLE_ROUND_BUDGET,
        show_baseline_budget_marker=True,
    )
    ax.grid(alpha=0.3)

    if ok:
        ax.legend(fontsize=8, ncol=2)
    else:
        annotate_no_data(ax)


def plot_apple_micro_macro_panel(ax, *, local_items, apple_items, graph_ids, source):
    if source == "local":
        pack = get_group_pooled_micro_macro_local(
            local_items=local_items,
            graph_ids=graph_ids,
            split="val",
        )
        title = "Pooled local centralized micro / macro F1"

    elif source == "after_local_update":
        pack = get_group_pooled_micro_macro_method(
            method_items=apple_items,
            graph_ids=graph_ids,
            phase=APPLE_LOCAL_UPDATE_TASK_PHASE,
            split="val",
        )
        title = "Pooled APPLE micro / macro F1 (after-local selected model)"
    elif source == "end_round_personalized":
        pack = get_group_pooled_micro_macro_method(
            method_items=apple_items,
            graph_ids=graph_ids,
            phase=APPLE_ENDROUND_TASK_PHASE,
            split="val",
        )
        title = "Pooled APPLE micro / macro F1 (end-of-round diagnostic only)"
    else:
        raise ValueError(f"Unknown APPLE source={source}")

    if source == "local":
        _plot_micro_macro_pack(
            ax,
            pack,
            title,
            ylabel="F1",
            max_round=APPLE_ROUND_BUDGET,
            show_baseline_budget_marker=True,
        )
    else:
        _plot_micro_macro_pack(
            ax,
            pack,
            title,
            ylabel="F1",
            max_round=APPLE_ROUND_BUDGET,
            show_baseline_budget_marker=True,
        )


def plot_apple_delta_micro_macro_panel(ax, *, local_items, apple_items, graph_ids):
    after_local_pack = get_group_delta_local_minus_method_micro_macro(
        local_items=local_items,
        method_items=apple_items,
        graph_ids=graph_ids,
        method_phase=APPLE_LOCAL_UPDATE_TASK_PHASE,
        split="val",
    )

    end_round_pack = get_group_delta_local_minus_method_micro_macro(
        local_items=local_items,
        method_items=apple_items,
        graph_ids=graph_ids,
        method_phase=APPLE_ENDROUND_TASK_PHASE,
        split="val",
    )

    apple_mismatch_pack = get_group_delta_method_minus_method_micro_macro(
        method_items=apple_items,
        graph_ids=graph_ids,
        phase_a=APPLE_LOCAL_UPDATE_TASK_PHASE,
        phase_b=APPLE_ENDROUND_TASK_PHASE,
        split="val",
    )

    ok = False

    ok |= plot_mean_std(
        ax,
        after_local_pack["micro_f1"],
        label="Local - APPLE after-local Micro-F1",
    )
    ok |= plot_mean_std(
        ax,
        after_local_pack["macro_f1"],
        label="Local - APPLE after-local Macro Pos-F1",
    )

    ok |= plot_mean_std(
        ax,
        end_round_pack["micro_f1"],
        label="Local - APPLE end-round Micro-F1",
        linestyle="--",
    )
    ok |= plot_mean_std(
        ax,
        end_round_pack["macro_f1"],
        label="Local - APPLE end-round Macro Pos-F1",
        linestyle="--",
    )

    ok |= plot_mean_std(
        ax,
        apple_mismatch_pack["micro_f1"],
        label="APPLE after-local - end-round Micro-F1",
        linestyle=":",
        linewidth=2.4,
    )
    ok |= plot_mean_std(
        ax,
        apple_mismatch_pack["macro_f1"],
        label="APPLE after-local - end-round Macro Pos-F1",
        linestyle=":",
        linewidth=2.4,
    )

    ax.axhline(0.0, linestyle="--", linewidth=1.0, color="black")
    ax.set_title("Delta diagnostics: local gap and APPLE mismatch gap")
    ax.set_xlabel("round")
    ax.set_ylabel("F1 delta")
    set_round_budget_axis(
        ax,
        max_round=APPLE_ROUND_BUDGET,
        show_baseline_budget_marker=True,
    )
    ax.grid(alpha=0.3)

    if ok:
        ax.legend(fontsize=7, ncol=2)
    else:
        annotate_no_data(ax)


def make_apple_performance_figure(row, exp_log, test_gen, *, family_page=False):
    graph_ids = [str(g) for g in row["graph_ids"]]

    local_items = _apple_local_items_for_row(row, exp_log)
    apple_items = _apple_method_items_for_row(row, exp_log)

    fig = plt.figure(figsize=(18, 24))
    gs = GridSpec(
        4,
        2,
        figure=fig,
        height_ratios=[0.78, 1.0, 1.0, 1.0],
        hspace=0.34,
        wspace=0.18,
    )

    ax0 = fig.add_subplot(gs[0, :])
    ax1 = fig.add_subplot(gs[1, :])
    ax2 = fig.add_subplot(gs[2, 0])
    ax3 = fig.add_subplot(gs[2, 1])
    ax4 = fig.add_subplot(gs[3, 0])
    ax5 = fig.add_subplot(gs[3, 1])

    table_df = build_method_algo_table(
        local_items=local_items,
        method_items=apple_items,
        graph_ids=graph_ids,
        method="apple",
        split="test",
    )

    draw_algo_table(
        ax0,
        table_df,
        title=(
            "Best test performance "
            f"(APPLE selected by after-local validation; "
            f"all methods {BASELINE_ROUND_BUDGET} rounds)"
        ),
    )

    plot_apple_loss_panel(
        ax1,
        local_items=local_items,
        apple_items=apple_items,
        graph_ids=graph_ids,
    )

    plot_apple_micro_macro_panel(
        ax2,
        local_items=local_items,
        apple_items=apple_items,
        graph_ids=graph_ids,
        source="local",
    )

    plot_apple_micro_macro_panel(
        ax3,
        local_items=local_items,
        apple_items=apple_items,
        graph_ids=graph_ids,
        source="after_local_update",
    )

    plot_apple_micro_macro_panel(
        ax4,
        local_items=local_items,
        apple_items=apple_items,
        graph_ids=graph_ids,
        source="end_round_personalized",
    )

    plot_apple_delta_micro_macro_panel(
        ax5,
        local_items=local_items,
        apple_items=apple_items,
        graph_ids=graph_ids,
    )

    if family_page:
        title = build_method_family_title("apple", row)
        header_text = _compose_method_family_header_text("apple", row, test_gen)
    else:
        title = build_method_global_title("apple", row)
        header_text = _compose_method_global_header_text("apple", row, test_gen)

    _apply_page_header(fig, title=title, header_text=header_text)
    return fig


def _apple_train_diag_df(apple_items, graph_ids=None):
    target = None if graph_ids is None else {str(g) for g in graph_ids}
    frames = []

    for item in apple_items:
        df = item["df"]
        if df is None or df.empty:
            continue

        part = df[(df["phase"] == "apple_train_diag") & (df["split"] == "train")].copy()

        if target is not None and "graph_id" in part.columns:
            part = part[part["graph_id"].astype(str).isin(target)].copy()

        if part.empty:
            continue

        part["seed"] = int(item["seed"])
        part["step"] = pd.to_numeric(part["round"], errors="coerce")
        frames.append(part)

    if not frames:
        return pd.DataFrame()

    out = pd.concat(frames, axis=0, ignore_index=True)

    for col in [
        "train_loss",
        "base_train_loss",
        "dr_prox_loss",
        "apple_lambda",
        "apple_mu",
        "dr_self_weight",
        "dr_l2_to_p0",
        "num_nodes",
    ]:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")

    return out


def _aggregate_seed_metric_from_df(df, value_col, *, group_cols=None):
    if df is None or df.empty or value_col not in df.columns:
        return pd.DataFrame(columns=["step", "mean", "std", "count"])

    if group_cols is None:
        group_cols = []

    curves = []
    for seed, seed_df in df.groupby("seed"):
        work = seed_df.copy()
        work = work.dropna(subset=["step", value_col])
        if work.empty:
            continue

        if group_cols:
            curve = (
                work.groupby(["step"] + group_cols, as_index=False)[value_col]
                .mean()
                .sort_values("step")
            )
        else:
            curve = (
                work.groupby("step", as_index=False)[value_col]
                .mean()
                .sort_values("step")
            )

        curve["seed"] = int(seed)
        curves.append(curve[["step", value_col]])

    return aggregate_seed_curves(curves, value_col)


def _plot_apple_lambda_panel(ax, diag_df):
    if diag_df is None or diag_df.empty:
        annotate_no_data(ax)
        ax.set_title("APPLE lambda scheduler")
        return

    agg = _aggregate_seed_metric_from_df(diag_df, "apple_lambda")
    ok = plot_mean_std(ax, agg, label="apple_lambda")

    mu_vals = (
        diag_df["apple_mu"].dropna().unique().tolist()
        if "apple_mu" in diag_df.columns
        else []
    )
    mu_text = ", ".join(f"{float(x):.3g}" for x in mu_vals[:3])
    if len(mu_vals) > 3:
        mu_text += ", ..."

    title = "APPLE lambda scheduler"
    if mu_text:
        title += f" | apple_mu={mu_text}"

    ax.set_title(title)
    ax.set_xlabel("round")
    ax.set_ylabel("lambda")
    set_round_budget_axis(
        ax,
        max_round=APPLE_ROUND_BUDGET,
        show_baseline_budget_marker=True,
    )
    ax.grid(alpha=0.3)

    if ok:
        ax.legend(fontsize=8)


def _plot_apple_train_loss_for_client(ax, diag_df, *, graph_id, label):
    if diag_df is None or diag_df.empty:
        annotate_no_data(ax)
        ax.set_title(label)
        return

    part = diag_df[diag_df["graph_id"].astype(str) == str(graph_id)].copy()
    if part.empty:
        annotate_no_data(ax)
        ax.set_title(label)
        return

    ok = False

    for col, curve_label in [
        ("base_train_loss", "base_train_loss"),
        ("train_loss", "total train_loss"),
    ]:
        agg = _aggregate_seed_metric_from_df(part, col)
        ok |= plot_mean_std(ax, agg, label=curve_label)

    ax2 = ax.twinx()
    prox_agg = _aggregate_seed_metric_from_df(part, "dr_prox_loss")
    prox_ok = plot_mean_std(
        ax2,
        prox_agg,
        label="dr_prox_loss",
        linestyle="--",
        alpha_fill=0.08,
    )

    ax.set_title(label)
    ax.set_xlabel("round")
    ax.set_ylabel("base / total")
    ax2.set_ylabel("dr_prox_loss")
    set_round_budget_axis(
        ax,
        max_round=APPLE_ROUND_BUDGET,
        show_baseline_budget_marker=True,
    )
    ax.grid(alpha=0.3)

    lines, labels = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()

    if ok or prox_ok:
        ax.legend(lines + lines2, labels + labels2, fontsize=7, loc="best")
    else:
        annotate_no_data(ax)


def make_apple_train_loss_decomposition_figure(row, exp_log, test_gen):
    apple_items = _apple_method_items_for_row(row, exp_log)
    graph_ids = [str(g) for g in row["graph_ids"]]
    label_map = _apple_graph_label_map(row)

    diag_df = _apple_train_diag_df(apple_items, graph_ids=graph_ids)

    fig = plt.figure(figsize=(18, 24))
    gs = GridSpec(
        4,
        2,
        figure=fig,
        height_ratios=[0.62, 1.0, 1.0, 1.0],
        hspace=0.34,
        wspace=0.25,
    )

    ax_lambda = fig.add_subplot(gs[0, :])
    _plot_apple_lambda_panel(ax_lambda, diag_df)

    client_axes = [
        fig.add_subplot(gs[1, 0]),
        fig.add_subplot(gs[1, 1]),
        fig.add_subplot(gs[2, 0]),
        fig.add_subplot(gs[2, 1]),
        fig.add_subplot(gs[3, 0]),
    ]

    ax_blank = fig.add_subplot(gs[3, 1])
    ax_blank.axis("off")

    for ax, gid in zip(client_axes, graph_ids):
        _plot_apple_train_loss_for_client(
            ax,
            diag_df,
            graph_id=gid,
            label=f"{label_map.get(str(gid), str(gid))} | graph_id={gid}",
        )

    _apply_page_header(
        fig,
        title=(
            f"APPLE training-loss decomposition | "
            f"{str(row.get('heterogeneity_level', '')).title()} heterogeneity"
        ),
        header_text=_compose_method_global_header_text("apple", row, test_gen),
    )

    return fig


def _apple_dr_summary_df(apple_items, graph_ids=None):
    target = None if graph_ids is None else {str(g) for g in graph_ids}
    frames = []

    for item in apple_items:
        dr = item.get("dr_df", pd.DataFrame())
        if dr is None or dr.empty:
            continue

        part = dr[dr["row_type"].astype(str) == "client_summary"].copy()
        if target is not None and "graph_id" in part.columns:
            part = part[part["graph_id"].astype(str).isin(target)].copy()

        if part.empty:
            continue

        part["seed"] = int(item["seed"])
        part["step"] = pd.to_numeric(part["round"], errors="coerce")
        frames.append(part)

    if not frames:
        return pd.DataFrame()

    out = pd.concat(frames, axis=0, ignore_index=True)

    for col in [
        "dr_self_weight",
        "dr_offdiag_abs_mass",
        "dr_offdiag_neg_mass",
        "dr_l2_to_p0",
        "dr_row_sum",
        "dr_row_abs_sum",
        "incoming_abs_mass_this_client",
        "incoming_signed_mass_this_client",
    ]:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")

    return out


def _metric_seed_curve_for_client(df, *, graph_id, metric_col):
    part = df[df["graph_id"].astype(str) == str(graph_id)].copy()
    if part.empty or metric_col not in part.columns:
        return pd.DataFrame(columns=["step", "mean", "std", "count"])

    curves = []
    for seed, seed_df in part.groupby("seed"):
        work = seed_df[["step", metric_col]].dropna().copy()
        if work.empty:
            continue

        curve = (
            work.groupby("step", as_index=False)[metric_col].mean().sort_values("step")
        )
        curves.append(curve)

    return aggregate_seed_curves(curves, metric_col)


def _plot_apple_dr_metric_by_client(
    ax,
    summary_df,
    *,
    graph_ids,
    label_map,
    metric_col,
    title,
    ylabel,
):
    ok = False

    for gid in graph_ids:
        agg = _metric_seed_curve_for_client(
            summary_df,
            graph_id=gid,
            metric_col=metric_col,
        )
        ok |= plot_mean_std(
            ax,
            agg,
            label=label_map.get(str(gid), str(gid)),
            alpha_fill=0.09,
        )

    ax.set_title(title)
    ax.set_xlabel("round")
    ax.set_ylabel(ylabel)
    set_round_budget_axis(
        ax,
        max_round=APPLE_ROUND_BUDGET,
        show_baseline_budget_marker=True,
    )
    ax.grid(alpha=0.3)

    if ok:
        ax.legend(fontsize=8, ncol=2)
    else:
        annotate_no_data(ax)


def _clean_apple_id(x):
    if pd.isna(x):
        return None

    s = str(x).strip()

    try:
        f = float(s)
        if np.isfinite(f) and f.is_integer():
            return str(int(f))
    except Exception:
        pass

    return s


def _apple_best_round_by_seed(apple_items):
    """
    Get selected APPLE best round per seed.

    Preferred source:
      best_apple_val_mean round

    Fallback:
      minimum apple_local_val_mean eval_loss
    """
    out = {}

    for item in apple_items:
        seed = int(item["seed"])
        df = item["df"]

        best_round = None

        part = df[
            (df["phase"] == "best_apple_val_mean") & (df["split"] == "val")
        ].copy()

        if not part.empty and "round" in part.columns:
            vals = pd.to_numeric(part["round"], errors="coerce").dropna()
            if len(vals) > 0:
                best_round = int(vals.iloc[0])

        if best_round is None:
            part = df[
                (df["phase"] == APPLE_LOCAL_MEAN_SCALAR_PHASE) & (df["split"] == "val")
            ].copy()

            if (
                not part.empty
                and "round" in part.columns
                and "eval_loss" in part.columns
            ):
                part["round_num"] = pd.to_numeric(part["round"], errors="coerce")
                part["loss_num"] = pd.to_numeric(part["eval_loss"], errors="coerce")
                part = part.dropna(subset=["round_num", "loss_num"]).copy()
                if not part.empty:
                    best_round = int(part.sort_values("loss_num").iloc[0]["round_num"])

        if best_round is not None:
            out[seed] = best_round

    return out


def _apple_selected_dr_matrix_mean(apple_items, graph_ids):
    graph_ids = [_clean_apple_id(g) for g in graph_ids]
    graph_ids = [g for g in graph_ids if g is not None]

    best_round_by_seed = _apple_best_round_by_seed(apple_items)

    matrices = []

    for item in apple_items:
        seed = int(item["seed"])
        if seed not in best_round_by_seed:
            continue

        selected_round = int(best_round_by_seed[seed])
        dr = item.get("dr_df", pd.DataFrame())

        if dr is None or dr.empty or "row_type" not in dr.columns:
            continue

        pair = dr[dr["row_type"].astype(str).str.strip() == "pair"].copy()

        if pair.empty:
            continue

        required = ["round", "graph_id", "donor_graph_id", "p_ij"]
        missing = [c for c in required if c not in pair.columns]

        if missing:
            print(f"[APPLE selected heatmap] missing columns: {missing}")
            print("available columns:", pair.columns.tolist())
            continue

        pair["receiver_id_clean"] = pair["graph_id"].apply(_clean_apple_id)
        pair["donor_id_clean"] = pair["donor_graph_id"].apply(_clean_apple_id)
        pair["round_num"] = pd.to_numeric(pair["round"], errors="coerce")
        pair["p_ij_num"] = pd.to_numeric(pair["p_ij"], errors="coerce")

        pair = pair[
            pair["receiver_id_clean"].isin(graph_ids)
            & pair["donor_id_clean"].isin(graph_ids)
            & pair["round_num"].eq(selected_round)
            & pair["p_ij_num"].notna()
        ].copy()

        if pair.empty:
            print(
                f"[APPLE selected heatmap] no DR pair rows matched "
                f"seed={seed}, selected_round={selected_round}"
            )
            continue

        mat = pd.DataFrame(index=graph_ids, columns=graph_ids, dtype=float)

        for _, rr in pair.iterrows():
            receiver = rr["receiver_id_clean"]
            donor = rr["donor_id_clean"]
            val = rr["p_ij_num"]

            if receiver in mat.index and donor in mat.columns and pd.notna(val):
                mat.loc[receiver, donor] = float(val)

        if mat.notna().any().any():
            matrices.append(mat.to_numpy(dtype=float))

    if not matrices:
        return None

    return np.nanmean(np.stack(matrices, axis=0), axis=0)


def _plot_apple_selected_dr_heatmap(ax, apple_items, *, graph_ids, label_map):
    mat = _apple_selected_dr_matrix_mean(apple_items, graph_ids)

    if mat is None:
        annotate_no_data(ax)
        ax.set_title("Selected-round APPLE DR coefficients p_ij")
        return

    labels = [label_map.get(str(gid), str(gid)) for gid in graph_ids]

    finite = mat[np.isfinite(mat)]
    if finite.size == 0:
        annotate_no_data(ax)
        ax.set_title("Selected-round APPLE DR coefficients p_ij")
        return

    vmax = float(np.nanmax(np.abs(finite)))
    if not np.isfinite(vmax) or vmax <= 0:
        vmax = 1.0

    im = ax.imshow(mat, aspect="auto", vmin=-vmax, vmax=vmax)

    ax.set_xticks(np.arange(len(labels)))
    ax.set_xticklabels(labels, rotation=30, ha="right")
    ax.set_yticks(np.arange(len(labels)))
    ax.set_yticklabels(labels)

    ax.set_xlabel("donor client j")
    ax.set_ylabel("receiver client i")
    ax.set_title("Selected-best-round APPLE DR coefficients p_ij, mean over seeds")

    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            val = mat[i, j]
            if np.isfinite(val):
                ax.text(j, i, f"{val:.2g}", ha="center", va="center", fontsize=8)

    cbar = ax.figure.colorbar(im, ax=ax, fraction=0.035, pad=0.02)
    cbar.set_label("signed p_ij")


def make_apple_dr_behavior_figure(row, exp_log, test_gen):
    apple_items = _apple_method_items_for_row(row, exp_log)
    graph_ids = [str(g) for g in row["graph_ids"]]
    label_map = _apple_graph_label_map(row)

    summary_df = _apple_dr_summary_df(apple_items, graph_ids=graph_ids)

    fig = plt.figure(figsize=(18, 25))
    gs = GridSpec(
        3,
        2,
        figure=fig,
        height_ratios=[1.0, 1.0, 1.25],
        hspace=0.32,
        wspace=0.20,
    )

    ax_self = fig.add_subplot(gs[0, 0])
    ax_abs = fig.add_subplot(gs[0, 1])
    ax_neg = fig.add_subplot(gs[1, 0])
    ax_l2 = fig.add_subplot(gs[1, 1])
    ax_heat = fig.add_subplot(gs[2, :])

    _plot_apple_dr_metric_by_client(
        ax_self,
        summary_df,
        graph_ids=graph_ids,
        label_map=label_map,
        metric_col="dr_self_weight",
        title="APPLE DR self-weight p_ii over rounds",
        ylabel="p_ii",
    )

    _plot_apple_dr_metric_by_client(
        ax_abs,
        summary_df,
        graph_ids=graph_ids,
        label_map=label_map,
        metric_col="dr_offdiag_abs_mass",
        title="APPLE off-diagonal absolute mass over rounds",
        ylabel="sum |p_ij|, j != i",
    )

    _plot_apple_dr_metric_by_client(
        ax_neg,
        summary_df,
        graph_ids=graph_ids,
        label_map=label_map,
        metric_col="dr_offdiag_neg_mass",
        title="APPLE off-diagonal negative mass over rounds",
        ylabel="sum min(p_ij, 0), j != i",
    )

    _plot_apple_dr_metric_by_client(
        ax_l2,
        summary_df,
        graph_ids=graph_ids,
        label_map=label_map,
        metric_col="dr_l2_to_p0",
        title="APPLE DR movement away from p0",
        ylabel="||p_i - p0||₂",
    )

    _plot_apple_selected_dr_heatmap(
        ax_heat,
        apple_items,
        graph_ids=graph_ids,
        label_map=label_map,
    )

    _apply_page_header(
        fig,
        title=(
            f"APPLE DR behavior | "
            f"{str(row.get('heterogeneity_level', '')).title()} heterogeneity"
        ),
        header_text=_compose_method_global_header_text("apple", row, test_gen),
    )

    return fig


def build_apple_diagnostics_pdf(
    global_run_table,
    family_run_table,
    exp_log,
    test_gen,
    out_path,
):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    global_sorted = _sort_global_table(global_run_table)

    with PdfPages(out_path) as pdf:
        for _, grow in global_sorted.iterrows():
            if "methods" in grow.index and "apple" not in grow["methods"]:
                continue

            # Section 1: performance, same structure as other method diagnostics.
            fig = make_apple_performance_figure(
                grow,
                exp_log,
                test_gen,
                family_page=False,
            )
            pdf.savefig(fig, bbox_inches="tight")
            plt.close(fig)

            fam_rows = _family_rows_for_global_row(family_run_table, grow)
            for _, frow in fam_rows.iterrows():
                fig = make_apple_performance_figure(
                    frow,
                    exp_log,
                    test_gen,
                    family_page=True,
                )
                pdf.savefig(fig, bbox_inches="tight")
                plt.close(fig)

            # Section 2: Apple objective decomposition.
            fig = make_apple_train_loss_decomposition_figure(
                grow,
                exp_log,
                test_gen,
            )
            pdf.savefig(fig, bbox_inches="tight")
            plt.close(fig)

            # Section 3: Apple DR / client-mixing behavior.
            fig = make_apple_dr_behavior_figure(
                grow,
                exp_log,
                test_gen,
            )
            pdf.savefig(fig, bbox_inches="tight")
            plt.close(fig)

    print(f"saved -> {out_path.resolve()}")

In [48]:
def plot_method_loss_panel(ax, *, local_items, method_items, graph_ids, method_name):
    ok = False

    # Local centralized baseline
    local_agg = get_weighted_group_scalar_stats(
        items=local_items,
        phase="val_epoch",
        split="val",
        metric_col="eval_loss",
        graph_ids=graph_ids,
        use_graph_filter=False,
    )

    # Correct post-aggregation validation phase per method
    if method_name == "gcfl_plus":
        post_phase = "cluster_val_client"
        method_label = "GCFL+ cluster/post-aggregation"
    else:
        post_phase = "global_val_client"
        method_label = f"{METHOD_LABELS[method_name]} post-aggregation"

    method_agg = get_weighted_group_loss_for_method(
        method_items=method_items,
        phase=post_phase,
        graph_ids=graph_ids,
    )

    ok |= plot_mean_std(ax, local_agg, label="Local centralized")
    ok |= plot_mean_std(ax, method_agg, label=method_label)

    ax.set_title("Validation loss comparison")
    ax.set_xlabel("round")
    ax.set_ylabel("eval_loss")
    ax.grid(alpha=0.3)

    if ok:
        ax.legend(fontsize=8)
    else:
        annotate_no_data(ax)

## 8. Runner

In [49]:
global_run_table = build_global_run_table(
    selected_subset,
    exp_log,
    methods=METHOD_ORDER,
    local_epochs=1,
)

family_run_table = build_family_run_table(
    selected_subset,
    exp_log,
    methods=METHOD_ORDER,
    local_epochs=1,
)

print("Global rows by heterogeneity:")
print(
    global_run_table[
        ["heterogeneity_level", "gamma", "subset_size", "model_tag", "methods"]
    ]
)

print("\nFamily rows by heterogeneity:")
print(
    family_run_table[
        ["heterogeneity_level", "gamma", "family", "subset_size", "model_tag"]
    ].head(30)
)

OUT_DIR = resolve_project_path(".") / OUT_DIR_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

# 3 separate overview PDFs:
# low, mid, high
build_all_methods_overview_pdfs_by_heterogeneity(
    global_run_table=global_run_table,
    family_run_table=family_run_table,
    exp_log=exp_log,
    test_gen=test_gen,
    out_dir=OUT_DIR,
)

# 1 diagnostics PDF for FedAvg, ordered low -> mid -> high
build_method_diagnostics_pdf(
    method="fedavg",
    global_run_table=global_run_table,
    family_run_table=family_run_table,
    exp_log=exp_log,
    test_gen=test_gen,
    out_path=OUT_DIR / "fedavg_diagnostics_low_mid_high_with_tables.pdf",
)

# 1 diagnostics PDF for FedProx, ordered low -> mid -> high
build_method_diagnostics_pdf(
    method="fedprox",
    global_run_table=global_run_table,
    family_run_table=family_run_table,
    exp_log=exp_log,
    test_gen=test_gen,
    out_path=OUT_DIR / "fedprox_diagnostics_low_mid_high_with_tables.pdf",
)

# 1 diagnostics PDF for GCFL+, ordered low -> mid -> high
build_gcfl_diagnostics_pdf(
    global_run_table=global_run_table,
    family_run_table=family_run_table,
    exp_log=exp_log,
    test_gen=test_gen,
    out_path=OUT_DIR / "gcfl_diagnostics_low_mid_high_with_tables.pdf",
)

build_apple_diagnostics_pdf(
    global_run_table=global_run_table,
    family_run_table=family_run_table,
    exp_log=exp_log,
    test_gen=test_gen,
    out_path=OUT_DIR / "apple_diagnostics_low_mid_high_with_tables.pdf",
)

Global rows by heterogeneity:
  heterogeneity_level  gamma  subset_size  \
0                 low    0.2            5   
1                 mid    0.5            5   
2                high    0.8            5   

                                           model_tag  \
0  mcwauto_layers6_lr0.001_wd0.0001_do0.1_hd64_eg...   
1  mcwauto_layers6_lr0.001_wd0.0001_do0.1_hd64_eg...   
2  mcwauto_layers6_lr0.001_wd0.0001_do0.1_hd64_eg...   

                               methods  
0  [fedavg, fedprox, gcfl_plus, apple]  
1  [fedavg, fedprox, gcfl_plus, apple]  
2  [fedavg, fedprox, gcfl_plus, apple]  

Family rows by heterogeneity:
   heterogeneity_level  gamma              family  subset_size  \
0                  low    0.2  specialized_cycle2            1   
1                  low    0.2  specialized_cycle3            1   
2                  low    0.2  specialized_cycle4            1   
3                  low    0.2  specialized_cycle5            1   
4                  low    0.2  speciali

/var/folders/nf/5zqn215x105fjm3njyczpqdm0000gn/T/ipykernel_72966/3996615723.py:60: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(
/var/folders/nf/5zqn215x105fjm3njyczpqdm0000gn/T/ipykernel_72966/3996615723.py:60: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(
/var/folders/nf/5zqn215x105fjm3njyczpqdm0000gn/T/ipykernel_72966/3996615723.py:60: DeprecationWarning: DataFrameGroupBy.apply operated o

saved -> /Users/andreali/Documents/Subgraph_Federated_Learning/andrea/plot_outputs_specialized_0.05_exp3/low_heterogeneity_methods_overview_with_tables.pdf


/var/folders/nf/5zqn215x105fjm3njyczpqdm0000gn/T/ipykernel_72966/3996615723.py:60: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(
/var/folders/nf/5zqn215x105fjm3njyczpqdm0000gn/T/ipykernel_72966/3996615723.py:60: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(
/var/folders/nf/5zqn215x105fjm3njyczpqdm0000gn/T/ipykernel_72966/3996615723.py:60: DeprecationWarning: DataFrameGroupBy.apply operated o

saved -> /Users/andreali/Documents/Subgraph_Federated_Learning/andrea/plot_outputs_specialized_0.05_exp3/mid_heterogeneity_methods_overview_with_tables.pdf


...

saved -> /Users/andreali/Documents/Subgraph_Federated_Learning/andrea/plot_outputs_specialized_0.05_exp3/fedprox_diagnostics_low_mid_high_with_tables.pdf


/var/folders/nf/5zqn215x105fjm3njyczpqdm0000gn/T/ipykernel_72966/3996615723.py:60: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(
/var/folders/nf/5zqn215x105fjm3njyczpqdm0000gn/T/ipykernel_72966/3996615723.py:60: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(
/var/folders/nf/5zqn215x105fjm3njyczpqdm0000gn/T/ipykernel_72966/3996615723.py:60: DeprecationWarning: DataFrameGroupBy.apply operated o

saved -> /Users/andreali/Documents/Subgraph_Federated_Learning/andrea/plot_outputs_specialized_0.05_exp3/gcfl_diagnostics_low_mid_high_with_tables.pdf


/var/folders/nf/5zqn215x105fjm3njyczpqdm0000gn/T/ipykernel_72966/3996615723.py:60: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(
/var/folders/nf/5zqn215x105fjm3njyczpqdm0000gn/T/ipykernel_72966/3996615723.py:60: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(
/var/folders/nf/5zqn215x105fjm3njyczpqdm0000gn/T/ipykernel_72966/3996615723.py:60: DeprecationWarning: DataFrameGroupBy.apply operated o

saved -> /Users/andreali/Documents/Subgraph_Federated_Learning/andrea/plot_outputs_specialized_0.05_exp3/apple_diagnostics_low_mid_high_with_tables.pdf
